# Flow 2 · 20 000 pasos adicionales del ganador

Esta libreta continúa **solo la arquitectura ganadora** desde su `best.pt` completo, restaurando pesos, EMA, AdamW, GradScaler y estados aleatorios. No inicia desde cero ni vuelve a entrenar cross y single.

**Input:** añade a Kaggle el output completo `flow2_v1_TODO.zip` (o su carpeta extraída). Puedes llamarlo `Flow2Works`. `summarize.zip` contiene resultados, pero carece de pesos y caches. Con el ZIP completo no necesitas volver a añadir los siete datasets originales.

Por defecto se amplía el entrenamiento con **máscaras reales** para comprobar el efecto de entrenar más. Flow 1 y VAE permanecen congelados: una arruga ausente de la máscara no puede recuperarse cambiando solamente el color. `CONTINUAR_CURRICULUM=True` permite continuar sus reglas anteriores; con cero pseudoetiquetas aceptadas no habilitará C.

También se mide error en una banda de dos píxeles del contorno para examinar detalles finos, sin cambiar la pérdida de entrenamiento ni usarlo para seleccionar por test.

Se muestran pérdidas cada **100 pasos**, se valida cada **500** y se conservan **best y last**. Se evalúan los **mismos 100 ejemplos**, con comparación antes/después mostrada en pantalla. El modelo para exportar se elige con validación; el test, ya inspeccionado, sirve como diagnóstico.

La última celda guarda **UN ZIP con TODO**, incluido el experimento original y la continuación. Estimación por tus logs: unas **1.5–2.5 horas adicionales**, según evaluación, exportación y almacenamiento. Los pesos nuevos no se han entrenado aquí.


In [ ]:

import os,sys,json,math,time,copy,random,hashlib,zipfile,io,importlib.util,gc,shutil
from pathlib import Path
from contextlib import nullcontext
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
import cv2
from tqdm.auto import tqdm
from IPython.display import display,FileLink

# Añade a Kaggle el output COMPLETO del Flow 2 anterior, extraído o en ZIP.
# summarize.zip no contiene pesos y no sirve para continuar el entrenamiento.
EXT=dict(
 PAQUETE_ANTERIOR=None,              # Ej.: '/kaggle/input/flow2works/flow2_v1_TODO.zip'
 RAIZ='/kaggle/input',SALIDA='/kaggle/working/flow2_color',VERSION='flow2_v1_20k',
 PASOS_EXTRA=20000,                  # Son ADICIONALES al entrenamiento anterior.
 LR_MAX=8e-5,WARMUP_EXTRA=500,LR_FINAL_FRACCION=.05,
 CONTINUAR_CURRICULUM=False,         # False: aísla el efecto de entrenar más con máscaras reales.
 EVAL_CADA=500,GUARDAR_CADA=250,LOG_CADA=100,
 RESELECCIONAR_CFG=True,REANUDAR=True,
)
EXT_SMOKE=os.environ.get('FLOW2_EXT_SMOKE','0')=='1'
if EXT_SMOKE:
 EXT.update(PAQUETE_ANTERIOR=os.environ['FLOW2_EXT_PARENT'],
  SALIDA=os.environ.get('FLOW2_EXT_OUT','/tmp/flow2_ext_smoke'),
  PASOS_EXTRA=int(os.environ.get('FLOW2_EXT_STEPS','4')),WARMUP_EXTRA=1,EVAL_CADA=2,GUARDAR_CADA=1)
assert EXT['PASOS_EXTRA']>0 and EXT['LOG_CADA']==100
assert EXT['VERSION'] and '/' not in EXT['VERSION'] and EXT['VERSION'] not in ['.','..']
if not hasattr(cv2,'CascadeClassifier'):
 raise ImportError("Se necesita OpenCV 4. Ejecuta %pip install 'opencv-python-headless>=4.10,<5' --no-deps y reinicia el kernel.")

def file_sha(path):
 h=hashlib.sha256()
 with open(path,'rb') as stream:
  for chunk in iter(lambda:stream.read(2**20),b''):h.update(chunk)
 return h.hexdigest()

def inspect_package(path):
 path=Path(path)
 required=['config.json','resumen.json','cache/photos.pt','cache/color_bank.pt','cache/rgba.pt',
           'cache/predictions.pt','pares_flow2.csv','manifest_color.csv','alias_color.csv',
           'assets/color/color_codec.py','assets/color/vae_control.pt','inferencia/flow2_color.pt']
 if path.is_dir():
  roots=[path] if (path/'resumen.json').is_file() else [p.parent for p in path.rglob('resumen.json')]
  good=[]
  for root in roots:
   if not all((root/name).is_file() for name in required):continue
   info=json.loads((root/'resumen.json').read_text());arch=info.get('ganador_arquitectura')
   if arch not in ['cross','single']:continue
   ck=f'runs/{arch}_curriculum/best.pt'
   if not (root/ck).is_file():continue
   cfg=json.loads((root/'config.json').read_text())
   good.append(dict(path=str(root),prefix='',arch=arch,checkpoint=ck,config=cfg,summary=info,is_zip=False))
  return good
 if not path.is_file() or path.suffix.lower()!='.zip':return []
 with zipfile.ZipFile(path) as z:
  names=set(z.namelist());good=[]
  for member in sorted(names):
   if Path(member).name!='resumen.json':continue
   prefix=member[:-len('resumen.json')]
   if not all(prefix+name in names for name in required):continue
   info=json.loads(z.read(member));arch=info.get('ganador_arquitectura')
   ck=f'runs/{arch}_curriculum/best.pt'
   if arch not in ['cross','single'] or prefix+ck not in names:continue
   cfg=json.loads(z.read(prefix+'config.json'))
   good.append(dict(path=str(path),prefix=prefix,arch=arch,checkpoint=ck,config=cfg,summary=info,is_zip=True))
  return good

if EXT['PAQUETE_ANTERIOR']:
 options=inspect_package(EXT['PAQUETE_ANTERIOR'])
else:
 root=Path(EXT['RAIZ'])
 paths=[p.parent for p in root.rglob('resumen.json')]
 paths+=list(root.rglob('*TODO.zip'))
 options=[]
 for path in tqdm(paths,desc='Localizando el Flow 2 entrenado',unit='paquete'):
  options.extend(inspect_package(path))
 # No elegir silenciosamente pesos de distintas corridas.
 options=list({(v['path'],v['prefix']):v for v in options}.values())
if not options:
 raise FileNotFoundError('Falta el paquete COMPLETO flow2_v1_TODO.zip o su carpeta extraída, con assets, cache y runs/*_curriculum/best.pt. Adjunta el output completo en Kaggle. summarize.zip solo contiene resultados; esta libreta nunca inicia desde cero.')
if len(options)!=1:
 raise ValueError('Hay varios modelos compatibles. Fija EXT[PAQUETE_ANTERIOR] a uno: '+str([v['path'] for v in options]))
SOURCE=options[0];PARENT_CONFIG=SOURCE['config'];WINNER=SOURCE['arch']
if bool(PARENT_CONFIG.get('SMOKE'))!=EXT_SMOKE:
 raise ValueError('No mezclar una prueba SMOKE con pesos de entrenamiento real.')
RUN=Path(EXT['SALIDA'])/EXT['VERSION'];RUN.mkdir(parents=True,exist_ok=True)
marker=RUN/'origen_20k.json'
if not marker.is_file():
 if SOURCE['is_zip']:
  with zipfile.ZipFile(SOURCE['path']) as z:
   members=[i for i in z.infolist() if i.filename.startswith(SOURCE['prefix'])]
   for member in tqdm(members,desc='Restaurando TODO el experimento',unit='archivo'):
    rel=Path(member.filename[len(SOURCE['prefix']):])
    if member.is_dir() or str(rel)=='.':continue
    if rel.is_absolute() or '..' in rel.parts:raise ValueError('Ruta ZIP insegura')
    dest=RUN/rel;dest.parent.mkdir(parents=True,exist_ok=True)
    with z.open(member) as stream,open(dest,'wb') as out:shutil.copyfileobj(stream,out,2**20)
 else:
  src=Path(SOURCE['path']).resolve()
  if src==RUN.resolve():raise ValueError('Usa una VERSION nueva para la extensión.')
  files=[p for p in src.rglob('*') if p.is_file()]
  for p in tqdm(files,desc='Copiando TODO el experimento',unit='archivo'):
   dest=RUN/p.relative_to(src);dest.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(p,dest)
 if not marker.is_file():
  # Conservar los resultados y el export de referencia antes de sobrescribir el export final.
  baseline=RUN/'anterior';baseline.mkdir(exist_ok=True)
  for p in list(RUN.iterdir()):
   if p.is_file():shutil.copy2(p,baseline/p.name)
  for name in ['inferencia','test_imagenes']:
   if (RUN/name).is_dir():shutil.copytree(RUN/name,baseline/name,dirs_exist_ok=True)
  origin=dict(checkpoint=SOURCE['checkpoint'],checkpoint_sha=file_sha(RUN/SOURCE['checkpoint']),
   data_sha=SOURCE['summary']['data_sha'],arch=WINNER,config=PARENT_CONFIG,
   cfg_original=float(SOURCE['summary']['cfg_w']),source_path=SOURCE['path'])
  marker.write_text(json.dumps(origin,indent=2))
ORIGIN=json.loads(marker.read_text())
PARENT_CONFIG=ORIGIN['config']
assert ORIGIN['arch']==WINNER and ORIGIN['data_sha']==SOURCE['summary']['data_sha']
assert file_sha(RUN/ORIGIN['checkpoint'])==ORIGIN['checkpoint_sha']
PARENT_CK=torch.load(RUN/ORIGIN['checkpoint'],map_location='cpu',weights_only=False)
assert PARENT_CK.get('artifact_role')=='flow2_training' and PARENT_CK['arch']==WINNER
assert all(k in PARENT_CK for k in ['model','ema','opt','scaler','rng','sampler','controller'])
assert PARENT_CK['data_sha']==ORIGIN['data_sha']
BASELINE_EXPORT=torch.load(RUN/'anterior'/'inferencia'/'flow2_color.pt',map_location='cpu',weights_only=False)
assert BASELINE_EXPORT['arch']==WINNER and BASELINE_EXPORT['data_sha']==ORIGIN['data_sha']
print('Ganador recuperado:',WINNER,'| checkpoint:',ORIGIN['checkpoint'])
print('Se entrenarán',EXT['PASOS_EXTRA'],'pasos ADICIONALES; no se repite la comparación cross/single.')
print('Curriculum adicional:',EXT['CONTINUAR_CURRICULUM'],'| fuente completa verificada; nunca se inicia desde cero.')


## Fuentes internas

Definiciones reproducibles de los modelos y de la ejecución. La configuración está arriba.

In [ ]:
FLOW1_MODEL_SOURCE = r'''def rope_2d(dim_cabeza, lado=16):
    d = dim_cabeza // 4
    frec = 1.0 / (10000 ** (torch.arange(d) / d))
    y, x = torch.meshgrid(torch.arange(lado), torch.arange(lado), indexing="ij")
    ang = torch.cat([y.flatten()[:, None] * frec, x.flatten()[:, None] * frec], 1)   # 256 × dim/2
    return torch.cos(ang), torch.sin(ang)


def aplicar_rope(x, cos, sin):                              # x: B×H×N×D
    x1, x2 = x[..., 0::2], x[..., 1::2]
    return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], -1).flatten(-2)


class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.peso, self.eps = nn.Parameter(torch.ones(d)), eps

    def forward(self, x):
        return x * torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps).to(x.dtype) * self.peso.to(x.dtype)


class Atencion(nn.Module):
    def __init__(self, dim, cabezas, cruzada=False):
        super().__init__()
        self.h, self.cruzada = cabezas, cruzada
        self.q = nn.Linear(dim, dim)
        self.kv = nn.Linear(dim, 2 * dim)
        self.nq, self.nk = RMSNorm(dim // cabezas), RMSNorm(dim // cabezas)
        self.salida = nn.Sequential(nn.Linear(dim, dim), nn.Dropout(CONFIG["DROPOUT"]))

    def forward(self, x, contexto=None, rope=None):
        b, n, d = x.shape
        c = x if contexto is None else contexto
        q = self.q(x).view(b, n, self.h, -1).transpose(1, 2)
        k, v = self.kv(c).view(b, c.shape[1], 2, self.h, -1).permute(2, 0, 3, 1, 4)
        q, k = self.nq(q), self.nk(k)
        if rope is not None:
            q, k = aplicar_rope(q, *rope), aplicar_rope(k, *rope)
        y = F.scaled_dot_product_attention(q, k, v)
        return self.salida(y.transpose(1, 2).reshape(b, n, d))


def modular(x, shift, scale):
    return x * (1 + scale[:, None]) + shift[:, None]


class Bloque(nn.Module):
    def __init__(self, dim, cabezas, mlp):
        super().__init__()
        self.n1, self.n2, self.n3 = (nn.LayerNorm(dim, elementwise_affine=False, eps=1e-6) for _ in range(3))
        self.auto = Atencion(dim, cabezas)
        self.cruzada = Atencion(dim, cabezas, cruzada=True)
        self.mlp = nn.Sequential(nn.Linear(dim, int(dim * mlp)), nn.GELU(approximate="tanh"),
                                 nn.Dropout(CONFIG["DROPOUT"]), nn.Linear(int(dim * mlp), dim))
        self.ada = nn.Sequential(nn.SiLU(), nn.Linear(dim, 9 * dim))
        nn.init.zeros_(self.ada[1].weight)
        nn.init.zeros_(self.ada[1].bias)

    def forward(self, x, c, foto, rope):
        s1, e1, g1, s2, e2, g2, s3, e3, g3 = self.ada(c).chunk(9, dim=1)
        x = x + g1[:, None] * self.auto(modular(self.n1(x), s1, e1), rope=rope)
        x = x + g2[:, None] * self.cruzada(modular(self.n2(x), s2, e2), contexto=foto)
        return x + g3[:, None] * self.mlp(modular(self.n3(x), s3, e3))


def embedding_tiempo(t, dim):
    mitad = dim // 2
    frec = torch.exp(-math.log(10000) * torch.arange(mitad, device=t.device) / mitad)
    a = t[:, None] * 1000 * frec[None]
    return torch.cat([torch.cos(a), torch.sin(a)], 1)


class DiT1(nn.Module):
    def __init__(self, dim=CONFIG["DIM"], prof=CONFIG["PROFUNDIDAD"], cabezas=CONFIG["CABEZAS"],
                 mlp=CONFIG["MLP"], dim_dino=768, n_tok=256):
        super().__init__()
        self.dim = dim
        self.entrada = nn.Linear(8, dim)
        self.pos = nn.Parameter(torch.zeros(1, n_tok, dim))
        self.tiempo = nn.Sequential(nn.Linear(dim, dim), nn.SiLU(), nn.Linear(dim, dim))
        self.proyeccion = nn.Sequential(nn.LayerNorm(dim_dino), nn.Linear(dim_dino, dim))
        self.sin_foto = nn.Parameter(torch.zeros(1, 1, dim))         # token "nulo" para CFG
        self.bloques = nn.ModuleList(Bloque(dim, cabezas, mlp) for _ in range(prof))
        self.n_final = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-6)
        self.ada_final = nn.Sequential(nn.SiLU(), nn.Linear(dim, 2 * dim))
        self.salida = nn.Linear(dim, 8)
        for m in [self.ada_final[1], self.salida]:
            nn.init.zeros_(m.weight)
            nn.init.zeros_(m.bias)
        nn.init.normal_(self.pos, std=0.02)
        nn.init.normal_(self.sin_foto, std=0.02)
        self.register_buffer("rope_cos", rope_2d(dim // cabezas)[0], persistent=False)
        self.register_buffer("rope_sin", rope_2d(dim // cabezas)[1], persistent=False)

    def forward(self, x_t, t, tok_foto, sin_foto=None):
        """x_t: B×256×8 · t: B · tok_foto: B×N×768 · sin_foto: B bool (True = omitir la foto)."""
        foto = self.proyeccion(tok_foto)
        if sin_foto is not None:
            foto = torch.where(sin_foto[:, None, None], self.sin_foto.expand_as(foto), foto)
        c = self.tiempo(embedding_tiempo(t, self.dim))
        x = self.entrada(x_t) + self.pos
        rope = (self.rope_cos, self.rope_sin)
        for b in self.bloques:
            x = b(x, c, foto, rope)
        shift, scale = self.ada_final(c).chunk(2, dim=1)
        return self.salida(modular(self.n_final(x), shift, scale))



class Regressor(nn.Module):
 def __init__(self):
  super().__init__();self.head=nn.Sequential(nn.LayerNorm(DIM_DINO),nn.Linear(DIM_DINO,512),nn.SiLU(),nn.Dropout(.1),nn.Linear(512,2048))
 def forward(self,photo):return self.head(photo.mean(1)).reshape(len(photo),256,8)
def make_model():
 return (DiT1(dim_dino=DIM_DINO) if CONFIG['MODO']=='flow' else Regressor()).to(DEVICE)'''

MASK_CODEC_SOURCE = r'''class FrozenDiTo(nn.Module):
 def __init__(self):
  super().__init__();self.kind='dito_base';self.is_vae=False
  self.encoder=codec.Encoder(ch=32,out_ch=1,ch_mult=(1,2,4,4),num_res_blocks=1,
   attn_resolutions=(),dropout=0.,in_channels=1,resolution=128,z_channels=8,double_z=False)
  self.norm=nn.LayerNorm((8,16,16),elementwise_affine=False)
  self.decoder=renderer.ConsistencyDecoderUNet(in_channels=1,z_dec_channels=8,
   c0=32,c1=64,c2=128,pe_dim=64,t_dim=128)
 def encode(self,x):return self.norm(self.encoder(x*2-1))
 def forward(self,x,t,z_dec):return self.decoder(x,t=t,z_dec=z_dec)
 @torch.no_grad()
 def decode(self,z,seeds,steps=None):
  # Una semilla propia por fila hace el ruido independiente del tamaño de lote.
  noise=torch.stack([torch.randn((1,128,128),generator=torch.Generator().manual_seed(int(s))) for s in seeds]).to(z.device)
  y=sampler.sample(self,(len(z),1,128,128),steps or CONFIG['PASOS_DITO'],net_kwargs={'z_dec':z},noise=noise)
  return ((y+1)/2).clamp(0,1)'''

FLOW1_GENERATE_SOURCE = r'''def generate(model,photo,seeds,w=1.,steps=None,method='euler'):
 model.eval()
 if CONFIG['MODO']=='regresor':return model(photo).float()
 steps=steps or CONFIG['PASOS_FLOW']
 x=torch.stack([torch.randn((256,8),generator=torch.Generator().manual_seed(int(s))) for s in seeds]).to(DEVICE)
 def vel(x,t):
  times=torch.full((len(x),),float(t),device=DEVICE)
  with amp_ctx():
   vc=model(x,times,photo,torch.zeros(len(x),dtype=torch.bool,device=DEVICE)).float()
   if w==1:return vc
   vu=model(x,times,photo,torch.ones(len(x),dtype=torch.bool,device=DEVICE)).float()
  return vu+w*(vc-vu)
 for i in range(steps):
  t=i/steps;v=vel(x,t)
  if method=='heun':x=x+(v+vel(x+v/steps,(i+1)/steps))/(2*steps)
  else:x=x+v/steps
 return x'''

SMOKE_VAE_SOURCE = r'''def mascara_a(m,lado):return F.adaptive_avg_pool2d(m,lado)
def celdas_activas(m):return (F.adaptive_max_pool2d(m,16)>0.5).float()

class Residual(nn.Module):
    def __init__(self,cin,cout):
        super().__init__()
        self.red=nn.Sequential(nn.GroupNorm(8 if cin%8==0 else 1,cin),nn.SiLU(),nn.Conv2d(cin,cout,3,1,1),
                               nn.GroupNorm(8,cout),nn.SiLU(),nn.Conv2d(cout,cout,3,1,1))
        self.atajo=nn.Identity() if cin==cout else nn.Conv2d(cin,cout,1)
    def forward(self,x):return self.atajo(x)+self.red(x)
class Bajar(nn.Module):
    def __init__(self,c):
        super().__init__();self.red=nn.Sequential(nn.GroupNorm(8,c),nn.SiLU(),nn.Conv2d(c,c,4,2,1))
    def forward(self,x):return self.red(x)
class Subir(nn.Module):
    def __init__(self,cin,cout):
        super().__init__();self.red=nn.Sequential(nn.GroupNorm(8,cin),nn.SiLU(),nn.Conv2d(cin,4*cout,3,1,1),nn.PixelShuffle(2))
    def forward(self,x):return self.red(x)
class Encoder(nn.Module):
    def __init__(self,cz,canales,cin=4):
        super().__init__();capas=[nn.Conv2d(cin,canales[0],3,1,1)];c=canales[0]
        for cn in canales:capas.extend([Residual(c,cn),Bajar(cn)]);c=cn
        self.red=nn.Sequential(*capas,Residual(c,c),nn.GroupNorm(8,c),nn.SiLU());self.cabeza=nn.Conv2d(c,2*cz,1)
    def forward(self,x):
        mu,lv=self.cabeza(self.red(x)).chunk(2,1);return mu,lv.clamp(-10,10)
class DecoderCondicionado(nn.Module):
    def __init__(self,cz,canales,cout=3):
        super().__init__();c=canales[-1]
        self.entrada=nn.Sequential(nn.Conv2d(cz+1,c,1),nn.GroupNorm(8,c))
        self.bloques=nn.ModuleList();self.subidas=nn.ModuleList()
        for cn in reversed(canales):
            self.bloques.append(Residual(c+1,c));self.subidas.append(Subir(c,cn));c=cn
        self.final=Residual(c+1,c)
        self.salida=nn.Sequential(nn.GroupNorm(8,c),nn.SiLU(),nn.Conv2d(c,cout,3,1,1))
    def forward(self,z,m):
        h=self.entrada(torch.cat([z,mascara_a(m,16)],1))
        for b,u in zip(self.bloques,self.subidas):h=u(b(torch.cat([h,mascara_a(m,h.shape[-1]).to(h.dtype)],1)))
        return self.salida(self.final(torch.cat([h,m.to(h.dtype)],1)))
class VAEColor(nn.Module):
    kind='vae_control'
    def __init__(self,cz=8,canales=(32,64,128)):
        super().__init__();self.encoder=Encoder(cz,canales);self.decoder=DecoderCondicionado(cz,canales)
    def encode(self,rgb,m):return self.encoder(torch.cat([rgb,m],1))[0]*celdas_activas(m)
    def decodificar(self,z,m):return torch.sigmoid(self.decoder(z*celdas_activas(m),m))*m
    def forward(self,rgb,m):
        mu,lv=self.encoder(torch.cat([rgb,m],1))
        z=mu+torch.exp(0.5*lv)*torch.randn_like(mu) if self.training else mu
        return self.decodificar(z,m),mu,lv
    @torch.no_grad()
    def decode(self,z,m,seeds=None,steps=None):return self.decodificar(z,m)
    @torch.no_grad()
    def reconstruct(self,rgb,m,seeds=None,steps=None):return self.decode(self.encode(rgb,m),m,seeds,steps)

'''

CROP_SOURCE = r'''def crop_photo(row):
 b=Path(row.archivo).read_bytes()
 if CONFIG['CHECK_SHA_FOTOS'] and isinstance(row.get('sha256_foto'),str) and row.sha256_foto:
  assert hashlib.sha256(b).hexdigest()==row.sha256_foto,('SHA foto distinto',row.archivo)
 with Image.open(io.BytesIO(b)) as im:im=im.convert('RGB');im.load();a=np.asarray(im)
 h,w=a.shape[:2];found=detector.detectMultiScale(cv2.cvtColor(a,cv2.COLOR_RGB2GRAY),1.1,5,minSize=(max(16,w//6),)*2)
 if len(found):
  x,y,cw,ch=max(found,key=lambda c:c[2]*c[3]);cx,cy,s=x+cw/2,y+ch/2,max(cw,ch)*1.8
 else:cx,cy,s=w/2,h*.45,min(w,h)*.75
 x0,y0=max(0,int(cx-s/2)),max(0,int(cy-s/2));x1,y1=min(w,int(cx+s/2)),min(h,int(cy+s/2))
 return np.asarray(Image.fromarray(a[y0:y1,x0:x1]).resize((256,256),Image.Resampling.BICUBIC)),bool(len(found)),hashlib.sha256(b).hexdigest()'''

MODEL_SOURCE = r'''def rope_2d(dim_cabeza, lado=16):
    d = dim_cabeza // 4
    frec = 1.0 / (10000 ** (torch.arange(d) / d))
    y, x = torch.meshgrid(torch.arange(lado), torch.arange(lado), indexing="ij")
    ang = torch.cat([y.flatten()[:, None] * frec, x.flatten()[:, None] * frec], 1)   # 256 × dim/2
    return torch.cos(ang), torch.sin(ang)

def aplicar_rope(x, cos, sin):                              # x: B×H×N×D
    x1, x2 = x[..., 0::2], x[..., 1::2]
    return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], -1).flatten(-2)

class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.peso, self.eps = nn.Parameter(torch.ones(d)), eps

    def forward(self, x):
        return x * torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps).to(x.dtype) * self.peso.to(x.dtype)

def modular(x, shift, scale):
    return x * (1 + scale[:, None]) + shift[:, None]

def embedding_tiempo(t, dim):
    mitad = dim // 2
    frec = torch.exp(-math.log(10000) * torch.arange(mitad, device=t.device) / mitad)
    a = t[:, None] * 1000 * frec[None]
    return torch.cat([torch.cos(a), torch.sin(a)], 1)

class Attention2(nn.Module):
 def __init__(self,dim,heads,dropout):
  super().__init__();self.heads=heads
  self.q=nn.Linear(dim,dim);self.kv=nn.Linear(dim,2*dim)
  self.nq=RMSNorm(dim//heads);self.nk=RMSNorm(dim//heads)
  self.out=nn.Sequential(nn.Linear(dim,dim),nn.Dropout(dropout))
 def forward(self,x,context=None,rope=None):
  b,n,d=x.shape;c=x if context is None else context
  q=self.q(x).reshape(b,n,self.heads,-1).transpose(1,2)
  k,v=self.kv(c).reshape(b,len(c[0]),2,self.heads,-1).permute(2,0,3,1,4)
  q,k=self.nq(q),self.nk(k)
  if rope is not None:q,k=aplicar_rope(q,*rope),aplicar_rope(k,*rope)
  y=F.scaled_dot_product_attention(q,k,v)
  return self.out(y.transpose(1,2).reshape(b,n,d))
class CrossBlock(nn.Module):
 def __init__(self,d,heads,ratio,drop):
  super().__init__();self.norms=nn.ModuleList([nn.LayerNorm(d,elementwise_affine=False) for _ in range(3)])
  self.selfatt=Attention2(d,heads,drop);self.cross=Attention2(d,heads,drop)
  self.mlp=nn.Sequential(nn.Linear(d,int(d*ratio)),nn.GELU(approximate='tanh'),nn.Dropout(drop),nn.Linear(int(d*ratio),d))
  self.ada=nn.Sequential(nn.SiLU(),nn.Linear(d,9*d));nn.init.zeros_(self.ada[-1].weight);nn.init.zeros_(self.ada[-1].bias)
 def forward(self,x,time,photo,rope):
  s1,e1,g1,s2,e2,g2,s3,e3,g3=self.ada(time).chunk(9,1)
  x=x+g1[:,None]*self.selfatt(modular(self.norms[0](x),s1,e1),rope=rope)
  x=x+g2[:,None]*self.cross(modular(self.norms[1](x),s2,e2),context=photo)
  return x+g3[:,None]*self.mlp(modular(self.norms[2](x),s3,e3))
class SingleBlock(nn.Module):
 def __init__(self,d,heads,ratio,drop):
  super().__init__();self.n1=nn.LayerNorm(d,elementwise_affine=False);self.n2=nn.LayerNorm(d,elementwise_affine=False)
  self.att=Attention2(d,heads,drop)
  self.mlp=nn.Sequential(nn.Linear(d,int(d*ratio)),nn.GELU(approximate='tanh'),nn.Dropout(drop),nn.Linear(int(d*ratio),d))
  self.ada=nn.Sequential(nn.SiLU(),nn.Linear(d,6*d));nn.init.zeros_(self.ada[-1].weight);nn.init.zeros_(self.ada[-1].bias)
 def forward(self,x,time,rope):
  s1,e1,g1,s2,e2,g2=self.ada(time).chunk(6,1)
  x=x+g1[:,None]*self.att(modular(self.n1(x),s1,e1),rope=rope)
  return x+g2[:,None]*self.mlp(modular(self.n2(x),s2,e2))
class PhotoColor(nn.Module):
 def __init__(self,grid):
  super().__init__();self.grid=grid
  self.net=nn.Sequential(nn.Conv2d(3,32,5,2,2),nn.SiLU(),nn.Conv2d(32,64,3,2,1),nn.SiLU(),nn.Conv2d(64,96,3,2,1),nn.SiLU())
 def forward(self,rgb):
  x=F.interpolate(rgb,size=(128,128),mode='bilinear',align_corners=False)
  f=F.adaptive_avg_pool2d(self.net(x),self.grid)
  mean=F.adaptive_avg_pool2d(x,self.grid)
  std=(F.adaptive_avg_pool2d(x*x,self.grid)-mean.square()).clamp_min(1e-6).sqrt()
  h=torch.cat([f,mean,std],1).flatten(2).transpose(1,2)
  return torch.cat([h.mean(1,keepdim=True),h],1)
class Flow2DiT(nn.Module):
 def __init__(self,arch,cfg,dino_dim):
  super().__init__();self.arch=arch;self.cfg=dict(cfg);d=cfg['DIM'];self.dim=d
  self.mask_encoder=nn.Sequential(nn.Conv2d(1,32,3,2,1),nn.SiLU(),nn.Conv2d(32,32,3,2,1),nn.SiLU(),nn.Conv2d(32,32,3,2,1),nn.SiLU())
  self.color=PhotoColor(cfg['DINO_POOL']);self.dino_proj=nn.Sequential(nn.LayerNorm(dino_dim),nn.Linear(dino_dim,d))
  self.rgb_proj=nn.Linear(102,d);self.null=nn.Parameter(torch.zeros(1,1,d))
  self.time=nn.Sequential(nn.Linear(d,d),nn.SiLU(),nn.Linear(d,d))
  self.pos=nn.Parameter(torch.randn(1,256,d)*.02);self.types=nn.Parameter(torch.randn(3,d)*.02)
  self.input=nn.Linear(41 if arch=='cross' else 8,d)
  self.mask_proj=nn.Linear(33,d)
  if arch=='cross':self.blocks=nn.ModuleList([CrossBlock(d,cfg['CABEZAS'],cfg['MLP'],cfg['DROPOUT']) for _ in range(cfg['PROF_CROSS'])])
  elif arch=='single':self.blocks=nn.ModuleList([SingleBlock(d,cfg['CABEZAS'],cfg['MLP'],cfg['DROPOUT']) for _ in range(cfg['PROF_SINGLE'])])
  else:raise ValueError(arch)
  self.final_norm=nn.LayerNorm(d,elementwise_affine=False);self.final_ada=nn.Sequential(nn.SiLU(),nn.Linear(d,2*d));self.out=nn.Linear(d,8)
  for layer in [self.final_ada[-1],self.out]:nn.init.zeros_(layer.weight);nn.init.zeros_(layer.bias)
  cos,sin=rope_2d(d//cfg['CABEZAS']);self.register_buffer('cos',cos,persistent=False);self.register_buffer('sin',sin,persistent=False)
 def forward(self,xt,t,dino,rgb,mask,drop=None):
  occ=(F.adaptive_max_pool2d(mask,16)>.5).float().flatten(2).transpose(1,2)
  shape=self.mask_encoder(mask).flatten(2).transpose(1,2)
  shape=torch.cat([shape,occ],-1)
  photo=self.dino_proj(dino)+self.rgb_proj(self.color(rgb))
  if drop is not None:photo=torch.where(drop[:,None,None],self.null.expand_as(photo),photo)
  time=self.time(embedding_tiempo(t,self.dim))
  if self.arch=='cross':
   x=self.input(torch.cat([xt,shape],-1))+self.pos
   for block in self.blocks:x=block(x,time,photo,(self.cos,self.sin))
  else:
   noisy=self.input(xt)+self.pos+self.types[0]
   shapes=self.mask_proj(shape)+self.pos+self.types[1]
   x=torch.cat([noisy,shapes,photo+self.types[2]],1)
   n=photo.shape[1]
   cos=torch.cat([self.cos,self.cos,torch.ones(n,self.cos.shape[1],device=xt.device)],0)
   sin=torch.cat([self.sin,self.sin,torch.zeros(n,self.sin.shape[1],device=xt.device)],0)
   for block in self.blocks:x=block(x,time,(cos,sin))
   x=x[:,:256]
  shift,scale=self.final_ada(time).chunk(2,1)
  return self.out(modular(self.final_norm(x),shift,scale))*occ
'''

GENERATE2_SOURCE = r'''def generate2(model,h,rgb,m,seeds,w=1.,steps=None,method='euler'):
 model.eval();steps=steps or CONFIG['PASOS_FLOW'];a=to_tokens(active(m))
 x=torch.stack([torch.randn((256,8),generator=torch.Generator().manual_seed(int(s))) for s in seeds]).to(DEVICE)*a
 def vel(x,t):
  tt=torch.full((len(x),),float(t),device=DEVICE)
  with amp_ctx():
   v=model(x,tt,h,rgb,m,torch.zeros(len(x),dtype=torch.bool,device=DEVICE)).float()
   if w!=1:
    u=model(x,tt,h,rgb,m,torch.ones(len(x),dtype=torch.bool,device=DEVICE)).float();v=u+w*(v-u)
  return v
 for j in range(steps):
  v=vel(x,j/steps)
  x=x+(v+vel(x+v/steps,(j+1)/steps))/(2*steps) if method=='heun' else x+v/steps
  x=x*a
 if not torch.isfinite(x).all():raise FloatingPointError('Latente generado no finito')
 return x'''

INFERENCE_SOURCE = r'''
import sys,json,math,io,importlib.util
from pathlib import Path
from types import SimpleNamespace
from contextlib import nullcontext
import numpy as np
import pandas as pd
from PIL import Image
import cv2
import torch
from torch import nn
from torch.nn import functional as F

def load_module(name,path):
 spec=importlib.util.spec_from_file_location(name,path);mod=importlib.util.module_from_spec(spec);spec.loader.exec_module(mod);return mod

def cargar_pipeline(root,device=None):
 root=Path(root);device=torch.device(device or ('cuda:0' if torch.cuda.is_available() else 'cpu'))
 ck=torch.load(root/'inferencia'/'flow2_color.pt',map_location='cpu',weights_only=False)
 if ck.get('artifact_role')!='flow2_color':raise ValueError('Checkpoint de otra tarea')
 cfg=ck['config'];constants=json.loads((root/'inferencia'/'runtime_constants.json').read_text())
 mod=load_module('flow2_infer_model',root/'inferencia'/'model.py')
 net=mod.Flow2DiT(ck['arch'],cfg,ck['dino_dim']).to(device).eval().requires_grad_(False);net.load_state_dict(ck['ema'],strict=True)
 cm=load_module('flow2_infer_color',root/'assets'/'color'/'color_codec.py')
 if cfg['SMOKE']:
  cp=torch.load(root/'assets'/'color'/'vae_control.pt',map_location='cpu',weights_only=False)
  color=cm.VAEColor();color.load_state_dict(cp['ema']);color=color.to(device).eval().requires_grad_(False)
  class ToyDino(nn.Module):
   def __init__(self):super().__init__();self.conv=nn.Conv2d(3,32,14,14)
   def forward(self,pixel_values):
    h=self.conv(pixel_values).flatten(2).transpose(1,2)
    return SimpleNamespace(last_hidden_state=torch.cat([h.mean(1,keepdim=True),h],1))
  dino=ToyDino();dino.load_state_dict(torch.load(root/'assets'/'dino'/'toy_dino.pt',map_location='cpu',weights_only=False))
  mean=torch.tensor([.485,.456,.406])[None,:,None,None];std=torch.tensor([.229,.224,.225])[None,:,None,None]
 else:
  color,cp=cm.cargar_color(root/'assets'/'color'/'vae_control.pt',device)
  from transformers import AutoModel,AutoImageProcessor
  processor=AutoImageProcessor.from_pretrained(root/'assets'/'dino',local_files_only=True)
  dino=AutoModel.from_pretrained(root/'assets'/'dino',local_files_only=True)
  mean=torch.tensor(processor.image_mean)[None,:,None,None];std=torch.tensor(processor.image_std)[None,:,None,None]
 dino=dino.to(device).eval().requires_grad_(False);mean=mean.to(device);std=std.to(device)
 cmean=cp['stats']['media'].to(device)[None,:,None,None];cstd=cp['stats']['desv'].to(device)[None,:,None,None]
 ctx=lambda:torch.autocast('cuda',dtype=torch.float16) if device.type=='cuda' else nullcontext()
 ns=dict(torch=torch,nn=nn,F=F,math=math,CONFIG=cfg,DEVICE=device,amp_ctx=ctx)
 ns['active']=lambda m:(F.adaptive_max_pool2d(m,16)>.5).float()
 ns['to_tokens']=lambda z:z.flatten(2).transpose(1,2)
 exec(constants['generate2'],ns)
 detector=cv2.CascadeClassifier(cv2.data.haarcascades+'haarcascade_frontalface_default.xml')
 crop_ns=dict(np=np,pd=pd,Path=Path,Image=Image,io=io,hashlib=__import__('hashlib'),cv2=cv2,detector=detector,CONFIG=dict(CHECK_SHA_FOTOS=False))
 exec(constants['crop'],crop_ns)
 f1=None
 def load_flow1():
  nonlocal f1
  if f1 is not None:return f1
  if cfg['SMOKE']:raise ValueError('El modelo SMOKE requiere máscara explícita; no genera caras humanas reales')
  fc=torch.load(root/'assets'/'flow1'/'flow1_dito.pt',map_location='cpu',weights_only=False)
  up=root/'assets'/'flow1'/'upstream';sys.path.insert(0,str(up))
  env=dict(torch=torch,nn=nn,F=F,math=math,CONFIG=fc['config'],DEVICE=device,amp_ctx=ctx,DIM_DINO=ck['dino_dim'])
  env['codec']=load_module('infer_mask_codec',up/'codec.py');env['renderer']=load_module('infer_mask_renderer',up/'renderer.py')
  env['fm']=load_module('infer_mask_fm',up/'fm.py').FM(sigma_min=1e-5,timescale=1000.)
  env['sampler']=load_module('infer_mask_sampler',up/'samplers.py').FMEulerSampler(env['fm'])
  exec(constants['flow1_model'],env);exec(constants['mask_codec'],env);exec(constants['flow1_generate'],env)
  flow=env['DiT1'](dim_dino=ck['dino_dim']).to(device).eval().requires_grad_(False);flow.load_state_dict(fc['ema'],strict=True)
  token=env['FrozenDiTo']().to(device).eval().requires_grad_(False)
  mc=torch.load(root/'assets'/'flow1'/'mask_dito.pt',map_location='cpu',weights_only=False);token.load_state_dict(mc['ema'],strict=True)
  f1=(flow,token,fc,env);return f1
 @torch.no_grad()
 def inferir_array(foto,mascara=None,seed=42,mostrar=True,pasos=None,cfg_w=None):
  a=np.asarray(foto,dtype=np.uint8)
  if a.ndim!=3 or a.shape[2]!=3:raise ValueError('Foto RGB H×W×3')
  x=torch.from_numpy(a.copy()).permute(2,0,1)[None].to(device).float()/255
  din=F.interpolate(x,(224,224),mode='bilinear',align_corners=False,antialias=True)
  with ctx():full=dino(pixel_values=(din-mean)/std).last_hidden_state.float()
  side=int(math.sqrt(full.shape[1]-1));pooled=F.adaptive_avg_pool2d(full[:,1:].transpose(1,2).reshape(1,-1,side,side),cfg['DINO_POOL'])
  h=torch.cat([full[:,:1],pooled.flatten(2).transpose(1,2)],1)
  if mascara is None:
   flow,token,fc,env=load_flow1();z=env['generate'](flow,full,[seed],w=float(fc.get('cfg_w',1.)),steps=fc['config']['PASOS_FLOW'])
   raw=z.transpose(1,2).reshape(1,8,16,16)*fc['stats']['desv'].to(device)+fc['stats']['media'].to(device)
   with ctx():mask=(token.decode(raw,[seed],fc['config'].get('PASOS_DITO_FINAL',32))>=.5).float()
  else:
   mm=np.asarray(mascara)
   if mm.shape!=(128,128):raise ValueError('Máscara ya canónica de 128×128; no se redimensiona')
   mask=torch.from_numpy((mm>(127 if mm.max()>1 else .5)).astype(np.float32))[None,None].to(device)
  if not 0<float(mask.mean())<1:raise ValueError('Máscara vacía/completa; revisar Flow 1')
  z=ns['generate2'](net,h,x,mask,[seed],w=float(ck['cfg_w'] if cfg_w is None else cfg_w),steps=pasos or cfg['PASOS_FLOW'])
  zz=(z.transpose(1,2).reshape(1,8,16,16)*cstd+cmean)*ns['active'](mask)
  with ctx():rgb=color.decodificar(zz,mask).float().cpu().numpy()[0]
  alpha=mask.cpu().numpy()[0,0];rgba=np.concatenate([(rgb.transpose(1,2,0)*255).round().clip(0,255).astype(np.uint8),(alpha[:,:,None]*255).astype(np.uint8)],2)
  if mostrar:
   import matplotlib.pyplot as plt
   background=np.array([250,200,40])[None,None,:];view=rgba[:,:,:3]*(alpha[:,:,None])+background*(1-alpha[:,:,None])
   fig,ax=plt.subplots(1,3,figsize=(9,3));ax[0].imshow(a);ax[0].set_title('Foto recortada');ax[1].imshow(alpha,cmap='gray');ax[1].set_title('Máscara');ax[2].imshow(view.astype(np.uint8));ax[2].set_title('LEGO color')
   for q in ax:q.axis('off')
   plt.tight_layout();plt.show();plt.close(fig)
  return dict(rgba=rgba,rgb=rgb,mascara=alpha,z=z.cpu())
 def inferir_foto(path,**kwargs):
  a,_,_=crop_ns['crop_photo'](pd.Series(dict(archivo=str(path))))
  return inferir_array(a,**kwargs)
 return SimpleNamespace(inferir_foto=inferir_foto,inferir_array=inferir_array,modelo=net,codec=color)
'''

RESEARCH_TEXT = r'''# Flow 2: color LEGO condicionado por la foto y la máscara

## Decisión propuesta

Usar un curriculum gradual, con Flow 1 y la VAE de color congelados. Comparar un DiT con atención cruzada, adaptado del Flow 1, contra un single-stream inspirado en Meshy T2. La VAE de color ganadora conserva su espacio latente de 8×16×16.

Esta es una propuesta experimental para el dataset LEGO: las referencias respaldan sus componentes, pero no prueban qué combinación ganará aquí.

## El problema principal: la supervisión cambia con la geometría

Si el LEGO objetivo tiene una boca pequeña y Flow 1 genera una boca más grande, el RGB original no define el color de los píxeles adicionales. Cambiar la máscara y conservar automáticamente el mismo objetivo puede introducir etiquetas incorrectas. El problema existe con cualquier arquitectura de atención.

| Estrategia | Ventaja | Riesgo |
|---|---|---|
| Máscaras reales | Supervisión alineada; permite comprobar que Flow 2 aprende color | Diferencia entre entrenamiento e inferencia |
| Solo máscaras de Flow 1 | Condiciones parecidas a producción | Objetivos desalineados y aprendizaje de errores de Flow 1 |
| Curriculum con mezcla | Primero aprende color, después tolera desviaciones | Hay que verificar la validez de los objetivos adaptados |

La primera estrategia será el control; la tercera, el candidato principal. No se recomienda comenzar exclusivamente con predicciones.

## Qué aporta la literatura

| Trabajo | Idea útil | Adaptación al proyecto |
|---|---|---|
| [Meshy T2, 2026](https://arxiv.org/abs/2607.28675) | Single-stream DiT con tokens de latente, imagen y vóxeles | Comparar fusión conjunta de código ruidoso, foto y máscara |
| [TRELLIS.2](https://arxiv.org/abs/2512.14692) | Imagen por cross-attention y geometría concatenada por canales | Unir máscara y código en posiciones correspondientes de la cuadrícula |
| [Cascaded Diffusion Models](https://arxiv.org/abs/2106.15282) | Perturbaciones de las condiciones reducen errores acumulados entre etapas | Entrenar tolerancia a condiciones imperfectas |
| [ColorizeDiffusion XL, 2026](https://arxiv.org/abs/2603.05971) | Ramas para referencias alineadas y desalineadas | Tratar explícitamente la diferencia entre referencia y dibujo |
| [ColorFM, ECCV 2026](https://arxiv.org/abs/2607.07119) | Transporte de colores guiado por semántica | Asociar regiones: labios con labios, cejas con cejas |

La descripción oficial de [Meshy T2](https://www.meshy.ai/blog/meshy-t2-native-3d-mesh-generation) confirma la concatenación de tokens. Su [repositorio](https://github.com/meshy-dev/meshy-t2), revisado el 15 de noviembre de 2026, anuncia todavía la publicación de la implementación. Se trata de una inspiración arquitectónica, no de reutilización de su código completo.

[TRELLIS.2 sí publica entrenamiento](https://github.com/microsoft/TRELLIS.2). Su [forward](https://github.com/microsoft/TRELLIS.2/blob/main/trellis2/models/structured_latent_flow.py) concatena condiciones geométricas con las características del latente. Su [pipeline de texturizado](https://github.com/microsoft/TRELLIS.2/blob/main/trellis2/pipelines/trellis2_texturing.py) normaliza geometría, genera características de textura y las desnormaliza para el decoder.

## Arquitectura inicial

Se conservan bloques DiT, AdaLN-Zero, RoPE 2D y normalización de queries y keys. Se adaptan las entradas y se entrenan los pesos del Flow 2 para color.

Las entradas son:

- **Foto:** tokens semánticos de DINOv2 y características cromáticas de una rama RGB pequeña.
- **Máscara:** características geométricas en una cuadrícula de 16×16.
- **Latente ruidoso:** código de color de 8×16×16 estandarizado con estadísticas de train.

Para cada posición se propone:

$$
h_i=\operatorname{Proyección}\left[z_{t,i},\;g_i(m),\;a_i(m)\right].
$$

La función g describe la forma local y a indica ocupación. Los tokens consultan la foto mediante cross-attention. Máscara y código comparten una posición espacial conocida; la correspondencia entre la foto y el LEGO se aprende mediante atención.

El punto de partida es dimensión 256, seis bloques y cuatro cabezas. La alternativa single-stream debe tener un presupuesto de parámetros comparable. No se debe asumir que una arquitectura con más cómputo gana por el mecanismo de fusión.

La rama RGB conserva información que podría debilitarse en los embeddings semánticos. El [entrenamiento oficial de DINOv2](https://github.com/facebookresearch/dinov2/blob/main/dinov2/data/augmentations.py) incluye ColorJitter y conversión a grises. Esto no significa que sus embeddings no tengan color; significa que no se debe presuponer fidelidad cromática exacta.

## Curriculum

### A. Supervisión limpia

El encoder congelado produce los códigos objetivo a partir de los LEGO coloreados y sus máscaras reales. Flow 2 aprende a generarlos condicionado por foto y máscara. El decoder permanece congelado.

### B. Variaciones con objetivos alineados

Aplicar deformaciones pequeñas y conocidas conjuntamente al RGB LEGO y a su máscara. Volver a codificar el par transformado. Mantener ejemplos limpios durante esta fase.

No basta con voltear el latente: un encoder convolucional no es necesariamente equivarante al espejo. Debe codificarse también el par RGB-máscara reflejado.

### C. Salidas de Flow 1

Incorporar una proporción pequeña y creciente de predicciones congeladas de Flow 1. Necesitan objetivos compatibles con la nueva geometría. Los cambios grandes de componentes no reciben automáticamente el RGB original.

Como punto de partida, conservar al menos 70 % de ejemplos limpios y reservar hasta 30 % para adaptación. Son hiperparámetros iniciales, no valores establecidos por la literatura.

El notebook acompañante usa una aproximación conservadora: admite predicciones próximas a la máscara real y extiende colores solo a píxeles nuevos cercanos con vecindarios de color uniforme. Estas son **pseudoetiquetas locales**, no ground truth humano independiente. Se registran aceptación y rechazos; si hay poca diversidad aceptada, la fase con predicciones permanece desactivada. Las máscaras distintas siguen mostrándose en la evaluación visual.

Flow 1 puede producir máscaras demasiado buenas en sus propias fotos de train. Idealmente se usarían predicciones fuera de muestra; el notebook no afirma que sus predicciones de train reproduzcan toda la distribución de errores de producción.

## Métricas y cambio de etapa

La pérdida de flow matching no basta. Evaluar RGB decodificado sobre un panel de validación fijo, con semillas fijas:

- MSE, PSNR y ΔE de color dentro de la máscara de referencia.
- Comparación frente a condiciones sin foto o barajadas.
- Detalles pequeños y regiones de color.
- Conservación de calidad en casos limpios al aumentar la adaptación.

Se propone además:

$$
R=\frac{\operatorname{MSE}(\text{Flow 2 decodificado})}{\operatorname{MSE}(\text{reconstrucción VAE})}.
$$

Ambos errores deben calcularse sobre el mismo panel. El denominador es una referencia del codec, no una cota matemática universal de la generación.

La siguiente etapa se abre cuando el modelo supera una referencia simple y su validación se estabiliza. Menos de 1 % de mejora durante tres evaluaciones es una propuesta configurable. Si la adaptación degrada la reconstrucción limpia, se reduce o se detiene su incorporación. El test no determina estas decisiones.

Para máscaras generadas muy diferentes, MSE píxel a píxel con la estampa original mezcla errores de forma y color. Las medidas sobre intersección son solo aproximaciones espaciales y no certifican correspondencia semántica. Deben reportarse por separado de la evaluación con máscara real.

## Gradientes y entrenamiento conjunto

Usar predicciones de Flow 1 no exige actualizar Flow 1. En la primera versión solo se entrenan Flow 2 y sus adaptadores.

Entrenar conjuntamente requeriría modificar el sampler con `no_grad`, la binarización y los objetivos de geometría. Es un experimento distinto: hay que impedir que la pérdida de color altere la máscara por conveniencia.

Se puede añadir una pérdida pequeña de RGB decodificado que llegue a Flow 2 a través del decoder congelado. Congelar sus parámetros no implica cortar el gradiente respecto a su entrada. En nuestro codec se utiliza `decodificar`, porque `decode` lleva `no_grad`.

## Datos y evaluación de la cascada

Auditar los manifiestos de color y Flow 1. La VAE de color se entrenó sin el CSV auditado de Flow 1, de modo que las reservas de intermedias pueden diferir. No unir tablas y volver a sortear todas las particiones.

El notebook entrena Flow 2 en la intersección de los train autorizados, conserva la evaluación del Flow 1 y registra si el codec ya había visto cada objetivo. Distingue generalización a fotos nuevas de generalización a objetivos no vistos por el codec.

Hay que comprobar también la coherencia entre los colores de las personas sintéticas y los LEGO emparejados: varias fotos pueden compartir estampa. Un modelo puede aprender la etiqueta del catálogo y aun así no copiar el tono de la persona. La comparación con fotos barajadas y la inspección regional ayudan a detectar ese problema.

## Entregables y ejecución

El notebook incluye comparación de arquitecturas, curriculum, métricas, ejemplos en pantalla y prueba de inferencia. Usa `tqdm`, resúmenes cada 100 pasos, pesos best/last, estados para reanudar y **un único ZIP con toda su carpeta de salida** al finalizar.

La selección de arquitectura se realiza primero en la etapa limpia. El curriculum se aplica después al ganador, evitando atribuir al curriculum una diferencia que provenga de la arquitectura.
'''

RUNTIME_SOURCES = {
'config': r'''
import os,sys,json,math,time,copy,random,hashlib,zipfile,io,importlib.util,gc,shutil
from pathlib import Path
from contextlib import nullcontext
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
import cv2
from tqdm.auto import tqdm
from IPython.display import display,FileLink
if not hasattr(cv2,'CascadeClassifier'):
 raise ImportError("Se necesita OpenCV 4 para reproducir el recorte de Flow 1. Ejecuta %pip install 'opencv-python-headless>=4.10,<5' --no-deps y reinicia el kernel.")

CONFIG=dict(
 VERSION='flow2_v1',RAIZ='/kaggle/input',SALIDA='/kaggle/working/flow2_color',SEMILLA=42,
 COLOR_DIR=None,CK_FLOW1=None,CK_DITO=None,PARES_AUDITADOS=None,DINO_LOCAL=None,
 ARQUITECTURAS=['cross','single'],DIM=256,CABEZAS=4,PROF_CROSS=6,PROF_SINGLE=8,
 MLP=4.,DROPOUT=.1,P_SIN_FOTO=.2,DINO='facebook/dinov2-base',DINO_POOL=8,
 PASOS_LIMPIOS=3000,PASOS_CURRICULUM=3000,LOTE=8,ACUMULAR=4,
 LR=3e-4,LR_CURRICULUM=8e-5,WD=.01,WARMUP=200,LR_FINAL=.05,EMA=.999,CLIP=1.,
 LOG_CADA=100,EVAL_CADA=500,GUARDAR_CADA=250,REANUDAR=True,REANUDAR_DE=None,
 PASOS_FLOW=25,LOTE_EVAL=8,N_VAL=48,N_TEST=100,POR_PAGINA=8,
 CFG_GRID=[1.,1.5,2.],ESPEJO=True,AUX_RGB=.02,AUX_CADA=4,AUX_MAX=4,
 N_PRED_TRAIN=1024,LOTE_PREPARACION=16,PRED_ENABLE=True,PRED_MAX=.30,
 P_AUG=.30,MIN_PASOS_A=1000,MIN_PASOS_B=1000,RAMPA_PRED=1000,
 PLATEAU_EVALS=3,PLATEAU_MEJORA=.01,MEJORA_BASE=.02,DEGRADACION_MAX=.05,
 REQUERIR_USO_FOTO=True,MEJORA_FOTO=.005,MIN_PRED_COLORES=32,MIN_VAL_PSEUDO=8,
 PRED_DICE_MIN=.85,PRED_DIST_MAX=2.,PRED_COLOR_SPREAD=.12,
 # En prueba de infraestructura se ejecutan etapas por reloj, sin inferir calidad.
 SMOKE=os.environ.get('FLOW2_SMOKE','0')=='1',CHECK_SHA_FOTOS=True,
)
if CONFIG['SMOKE']:
 CONFIG.update(DIM=32,CABEZAS=2,PROF_CROSS=1,PROF_SINGLE=1,PASOS_LIMPIOS=2,
  PASOS_CURRICULUM=6,LOTE=2,ACUMULAR=1,WARMUP=1,EVAL_CADA=2,GUARDAR_CADA=2,
  PASOS_FLOW=2,LOTE_EVAL=2,N_VAL=4,N_TEST=4,POR_PAGINA=4,DINO_POOL=8,
  MIN_PRED_COLORES=1,MIN_VAL_PSEUDO=1,MIN_PASOS_A=1,MIN_PASOS_B=2,RAMPA_PRED=2,
  N_PRED_TRAIN=24,LOTE_PREPARACION=4,CFG_GRID=[1.],AUX_CADA=1,AUX_MAX=2)
 CONFIG['SALIDA']=os.environ.get('FLOW2_SMOKE_OUT','/tmp/flow2_smoke')
assert CONFIG['LOG_CADA']==100
DEVICE=torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
DINO_DEVICE=torch.device('cuda:1' if torch.cuda.device_count()>1 else DEVICE)
AMP=DEVICE.type=='cuda'
if DEVICE.type=='cpu':torch.set_num_threads(min(4,os.cpu_count() or 1))
RUN=Path(CONFIG['SALIDA'])/CONFIG['VERSION'];RUN.mkdir(parents=True,exist_ok=True)
if CONFIG['REANUDAR_DE']:
 previous=Path(CONFIG['REANUDAR_DE'])
 if previous.is_dir():
  if previous.resolve()!=RUN.resolve():shutil.copytree(previous,RUN,dirs_exist_ok=True)
 elif previous.is_file() and previous.suffix.lower()=='.zip':
  with zipfile.ZipFile(previous) as z:
   names=z.namelist();roots={Path(n).parts[0] for n in names if Path(n).parts}
   prefix=next(iter(roots))+'/' if len(roots)==1 else ''
   for name in tqdm(names,desc='Restaurando ZIP completo',unit='archivo'):
    rel=Path(name[len(prefix):] if prefix else name)
    if name.endswith('/') or str(rel)=='.':continue
    if rel.is_absolute() or '..' in rel.parts:raise ValueError('Ruta ZIP insegura')
    target=RUN/rel;target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(z.read(name))
 else:raise FileNotFoundError(previous)
def seed_all(seed):
 random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
 if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
def amp_ctx():return torch.autocast('cuda',dtype=torch.float16) if AMP else nullcontext()
def sha_file(path):
 h=hashlib.sha256()
 with open(path,'rb') as f:
  for b in iter(lambda:f.read(1<<20),b''):h.update(b)
 return h.hexdigest()
def atomic_save(obj,path):
 path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
 temp=path.with_suffix(path.suffix+'.tmp');torch.save(obj,temp);temp.replace(path)
def load_module(name,path):
 spec=importlib.util.spec_from_file_location(name,path);mod=importlib.util.module_from_spec(spec)
 spec.loader.exec_module(mod);return mod
def cpu_weights(model):return {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
seed_all(CONFIG['SEMILLA']);torch.backends.cudnn.benchmark=False
print('Flow 2:',DEVICE,'| DINO:',DINO_DEVICE,'| AMP:',AMP)
print('Lote efectivo:',CONFIG['LOTE']*CONFIG['ACUMULAR'],'| resúmenes cada 100 pasos')
if CONFIG['SMOKE']:print('SMOKE: datos y pesos artificiales; no evalúa calidad.')

def bootstrap_source():
 constants=dict(FLOW1_MODEL_SOURCE=FLOW1_MODEL_SOURCE,MASK_CODEC_SOURCE=MASK_CODEC_SOURCE,
  FLOW1_GENERATE_SOURCE=FLOW1_GENERATE_SOURCE,SMOKE_VAE_SOURCE=SMOKE_VAE_SOURCE,
  CROP_SOURCE=CROP_SOURCE,MODEL_SOURCE=MODEL_SOURCE,GENERATE2_SOURCE=GENERATE2_SOURCE,
  INFERENCE_SOURCE=INFERENCE_SOURCE,RESEARCH_TEXT=RESEARCH_TEXT,RUNTIME_SOURCES=RUNTIME_SOURCES)
 return '\n'.join(name+' = '+repr(value) for name,value in constants.items())
''',
'assets': r'''
def safe_extract(z,names,dest):
 dest=Path(dest);dest.mkdir(parents=True,exist_ok=True)
 for name in names:
  rel=Path(name)
  if rel.is_absolute() or '..' in rel.parts:raise ValueError('Ruta ZIP insegura: '+name)
  if name.endswith('/'):continue
  p=dest/rel;p.parent.mkdir(parents=True,exist_ok=True);p.write_bytes(z.read(name))
 return dest
def choose(paths,label):
 paths=sorted(set(Path(p) for p in paths if Path(p).is_file()),key=str)
 if not paths:raise FileNotFoundError(label)
 if len(paths)>1 and len({sha_file(p) for p in paths})>1:raise ValueError(f'{label}: varias versiones; fija CONFIG: {paths}')
 return paths[0]

def prepare_assets():
 existing_color=RUN/'assets'/'color';existing_flow=RUN/'assets'/'flow1'
 if (existing_color/'color_codec.py').is_file() and (existing_flow/'flow1_dito.pt').is_file() and (RUN/'cache'/'rgba.pt').is_file():
  return existing_color,existing_flow,[]
 root=Path(CONFIG['RAIZ']);files=[p for p in tqdm(root.rglob('*'),desc='Indexando inputs',unit='ruta') if p.is_file()]
 archives=[p for p in files if p.suffix.lower()=='.zip']
 def find_asset(name,explicit=None):
  if explicit:return choose([explicit],name)
  candidates=[p for p in files if p.name==name]
  if candidates:return choose(candidates,name)
  for archive in tqdm(archives,desc='Buscando '+name+' en ZIP',unit='zip',leave=False):
   with zipfile.ZipFile(archive) as z:
    members=[n for n in z.namelist() if Path(n).name==name]
    if not members:continue
    dest=RUN/'assets'/'imports'/sha_file(archive)[:12]
    selected=set(members)
    for member in members:
     parent=Path(member).parent
     if name=='color_codec.py':selected.update(n for n in z.namelist() if str(Path(n).parent).startswith(str(parent)))
     if name=='flow1_dito.pt':
      selected.update(n for n in z.namelist() if Path(n).parent==parent and Path(n).name in ['pares_auditados.csv','stats_flow.pt'])
    safe_extract(z,sorted(selected),dest)
    candidates.extend(dest/n for n in members)
  return choose(candidates,'Falta '+name)
 color_script=find_asset('color_codec.py',Path(CONFIG['COLOR_DIR'])/'color_codec.py' if CONFIG['COLOR_DIR'] else None)
 color_dir=RUN/'assets'/'color'
 shutil.copytree(color_script.parent,color_dir,dirs_exist_ok=True,ignore=shutil.ignore_patterns('__pycache__'))
 ck_flow=find_asset('flow1_dito.pt',CONFIG['CK_FLOW1'])
 flow_dest=RUN/'assets'/'flow1';flow_dest.mkdir(exist_ok=True)
 shutil.copyfile(ck_flow,flow_dest/'flow1_dito.pt')
 audit=find_asset('pares_auditados.csv',CONFIG['PARES_AUDITADOS'])
 shutil.copyfile(audit,flow_dest/'pares_auditados.csv')
 cp=torch.load(flow_dest/'flow1_dito.pt',map_location='cpu',weights_only=False)
 if cp.get('config',{}).get('SMOKE'):raise ValueError('El Flow 1 adjunto es una prueba SMOKE')
 expected=cp.get('tokenizer_sha')
 if not expected:raise ValueError('Flow 1 sin tokenizer_sha; se requiere el export verificado')
 candidates=[Path(CONFIG['CK_DITO'])] if CONFIG['CK_DITO'] else [p for p in files if p.name=='best.pt' and p.parent.name=='dito_base']
 matched=[p for p in candidates if p.is_file() and sha_file(p)==expected]
 if not matched:
  for archive in tqdm(archives,desc='Localizando el DiTo de máscaras exacto',unit='zip',leave=False):
   with zipfile.ZipFile(archive) as z:
    for member in z.namelist():
     p=Path(member)
     if p.name=='best.pt' and p.parent.name=='dito_base':
      b=z.read(member)
      if hashlib.sha256(b).hexdigest()==expected:
       target=flow_dest/'mask_dito.pt';target.write_bytes(b);matched.append(target)
 if not matched:raise FileNotFoundError('Falta el best.pt de dito_base que corresponde al Flow 1. Añade el output de geometría.')
 if matched[0]!=flow_dest/'mask_dito.pt':shutil.copyfile(matched[0],flow_dest/'mask_dito.pt')
 return color_dir,flow_dest,files

if CONFIG['SMOKE']:
 ASSETS=RUN/'assets';ASSETS.mkdir(exist_ok=True)
 COLOR_DIR=ASSETS/'color';COLOR_DIR.mkdir(exist_ok=True)
 (COLOR_DIR/'color_codec.py').write_text('import torch\nfrom torch import nn\nfrom torch.nn import functional as F\n'+SMOKE_VAE_SOURCE)
 color_module=load_module('flow2_smoke_color',COLOR_DIR/'color_codec.py')
 COLOR=color_module.VAEColor().eval().requires_grad_(False)
 COLOR_CK=dict(kind='vae_control',stats=dict(media=torch.zeros(8),desv=torch.ones(8)),smoke=True)
 atomic_save(dict(ema=cpu_weights(COLOR),**COLOR_CK),COLOR_DIR/'vae_control.pt')
 F1_DIR=ASSETS/'flow1';F1_DIR.mkdir(exist_ok=True);INPUT_FILES=[];FLOW1_CK=None
else:
 COLOR_DIR,F1_DIR,INPUT_FILES=prepare_assets()
 selection=json.loads((COLOR_DIR/'seleccion.json').read_text())
 if selection.get('smoke'):raise ValueError('El paquete de color adjunto es SMOKE')
 color_module=load_module('flow2_color_codec',COLOR_DIR/'color_codec.py')
 COLOR,COLOR_CK=color_module.cargar_color(COLOR_DIR/selection['checkpoint'],DEVICE)
 if COLOR_CK['kind']!='vae_control':raise ValueError('Esta versión integra la VAE de color ganadora; se esperaba vae_control')
 FLOW1_CK=torch.load(F1_DIR/'flow1_dito.pt',map_location='cpu',weights_only=False)
 COLOR_STATS=COLOR_CK['stats']
 COLOR.requires_grad_(False).eval()
COLOR=COLOR.to(DEVICE).eval().requires_grad_(False)
COLOR_STATS=COLOR_CK['stats']
C_MEAN=COLOR_STATS['media'].to(DEVICE)[None,:,None,None]
C_STD=COLOR_STATS['desv'].to(DEVICE)[None,:,None,None].clamp_min(1e-6)
def active(m):return (F.adaptive_max_pool2d(m,16)>.5).float()
def to_tokens(z):return z.flatten(2).transpose(1,2)
def from_tokens(z):return z.transpose(1,2).reshape(len(z),8,16,16)
def standardize(z,m):return ((z-C_MEAN)/C_STD)*active(m)
def unstandardize(z,m):return (z*C_STD+C_MEAN)*active(m)
def color_decode(z,m):
 # Mantiene gradiente respecto a z; los parámetros del decoder están congelados.
 return COLOR.decodificar(unstandardize(from_tokens(z),m),m)
print('Codec de color congelado:',COLOR_CK['kind'])
''',
'data': r'''
def canonical_rgba(path):
 with Image.open(path) as im:
  if 'A' not in im.getbands() and 'transparency' not in im.info:raise ValueError('Estampa sin alfa: '+str(path))
  a=np.asarray(im.convert('RGBA')).copy()
 h,w=a.shape[:2]
 if h>128 or w>128:raise ValueError(f'Tamaño inesperado {w}x{h}: {path}; no redimensionar')
 out=np.zeros((4,128,128),np.uint8);y=(128-h)//2;x=(128-w)//2;m=a[:,:,3]>=128
 out[:3,y:y+h,x:x+w]=(a[:,:,:3]*m[:,:,None]).transpose(2,0,1);out[3,y:y+h,x:x+w]=m*255
 if not 0<out[3].mean()<255:raise ValueError('Máscara vacía/completa: '+str(path))
 return out

def fixture_data():
 root=RUN/'fixture';root.mkdir(exist_ok=True);aliases=[];rows=[];arrays=[]
 for i in range(18):
  rgba=np.zeros((128,128,4),np.uint8);rgba[25:34,25:36,3]=255;rgba[25:34,85:96,3]=255
  rgba[65:70+i%3,45:80,3]=255;rgba[88+i%6,20+i,3]=255
  rgba[:,:,:3]=(rgba[:,:,3,None]>0)*np.array([25+i*5,15+i*2,30],np.uint8)
  p=root/f's{i:02d}.png';Image.fromarray(rgba).save(p);c=canonical_rgba(p);arrays.append(c)
  split='entrenamiento' if i<12 else ('validacion' if i<15 else 'prueba')
  cid=('real:' if i<9 else 'int:')+f's{i:02d}'
  aliases.append(dict(id=cid,color_i=i,particion=split,sha_color=hashlib.sha256(c.tobytes()).hexdigest(),ruta=str(p),tipo='real' if i<9 else 'intermedia'))
  for k in range(2):
   photo=root/f'photo_{i}_{k}.jpg';bg=np.array([220,200,175],np.uint8)
   a=(c[:3].transpose(1,2,0)*(c[3,:, :,None]>0)+bg*(c[3,:,:,None]==0)).astype(np.uint8)
   Image.fromarray(a).resize((256,256)).save(photo)
   rows.append(dict(id=f's{i:02d}_{k:02d}',pieza=f's{i:02d}',k=k,version='g1' if i<9 else 'gi',
      split_flow=split,archivo=str(photo),color_i=i,color_id=cid,sha_foto_efectivo=sha_file(photo)))
 return pd.DataFrame(rows),pd.DataFrame(aliases),torch.from_numpy(np.stack(arrays))

if (RUN/'cache'/'rgba.pt').is_file() and (RUN/'pares_flow2.csv').is_file() and (RUN/'manifest_color.csv').is_file():
 pairs=pd.read_csv(RUN/'pares_flow2.csv',keep_default_na=False,dtype={'id':str,'pieza':str,'version':str})
 aliases=pd.read_csv(RUN/'alias_color.csv',keep_default_na=False,dtype={'id':str})
 color_table=pd.read_csv(RUN/'manifest_color.csv',keep_default_na=False,dtype={'id':str})
 RGBA=torch.load(RUN/'cache'/'rgba.pt',map_location='cpu',weights_only=False)
elif CONFIG['SMOKE']:
 pairs,aliases,RGBA=fixture_data();color_table=aliases.copy()
else:
 aliases=pd.read_csv(COLOR_DIR/'manifest_color_aliases.csv',keep_default_na=False,dtype={'id':str})
 color_table=pd.read_csv(COLOR_DIR/'manifest_color.csv',keep_default_na=False,dtype={'id':str})
 assert aliases.id.is_unique and color_table.color_i.is_unique
 pairs=pd.read_csv(F1_DIR/'pares_auditados.csv',keep_default_na=False,dtype={'id':str,'pieza':str,'version':str})
 assert {'id','pieza','version','split_flow'}<=set(pairs.columns)
 pairs['color_id']=np.where(pairs.version.eq('gi'),'int:','real:')+pairs.pieza
 mapping=aliases.set_index('id').color_i.to_dict()
 missing=set(pairs.color_id)-set(mapping)
 if missing:raise ValueError(f'Estampas emparejadas ausentes del paquete de color: {sorted(missing)[:15]}')
 pairs['color_i']=pairs.color_id.map(mapping).astype(int)
 image_index={}
 for p in INPUT_FILES:
  if p.suffix.lower() in ['.png','.jpg','.jpeg']:image_index.setdefault(p.name,[]).append(p)
 def resolve_image(raw,expected_sha=None,is_rgba=False,expected_color=None):
  raw=Path(str(raw));cand=[raw] if raw.is_file() else []
  suffix='/'.join(raw.parts[-3:])
  if not cand:cand=[p for p in image_index.get(raw.name,[]) if str(p).endswith(suffix)]
  if not cand:cand=image_index.get(raw.name,[])
  if expected_sha:cand=[p for p in cand if sha_file(p)==expected_sha]
  if is_rgba and expected_color:
   cand=[p for p in cand if hashlib.sha256(canonical_rgba(p).tobytes()).hexdigest()==expected_color]
  return choose(cand,f'Imagen ausente/ambigua: {raw}')
 paths=[];arrays=[]
 for r in tqdm(color_table.itertuples(),total=len(color_table),desc='Verificando RGB LEGO',unit='estampa'):
  p=resolve_image(r.ruta,is_rgba=True,expected_color=r.sha_color);c=canonical_rgba(p)
  assert hashlib.sha256(c.tobytes()).hexdigest()==r.sha_color
  arrays.append(c);paths.append(str(p))
 assert np.array_equal(color_table.color_i.to_numpy(),np.arange(len(color_table)))
 RGBA=torch.from_numpy(np.stack(arrays));del arrays
 color_table['ruta_actual']=paths
 resolved=[]
 for r in tqdm(pairs.to_dict('records'),desc='Verificando pares y fotos',unit='foto'):
  expected=r.get('sha_foto_efectivo') or r.get('sha256_foto') or None
  raw=r.get('archivo') or r.get('ruta')
  if not raw:raise ValueError('Fila sin ruta de foto: '+r['id'])
  p=resolve_image(raw,expected_sha=expected)
  assert p.stem==r['id'],(p,r['id'])
  resolved.append(str(p))
 pairs['archivo']=resolved

color_split=color_table.set_index('color_i').particion.to_dict()
pairs['particion_color']=pairs.color_i.map(color_split)
pairs['split_flow2']=pairs.split_flow
# No usar para train del Flow 2 objetivos que el protocolo del codec reservó.
reserved=pairs.split_flow.eq('entrenamiento') & pairs.particion_color.ne('entrenamiento')
pairs.loc[reserved,'split_flow2']='reserva_codec'
pairs['codec_color_vio_objetivo']=pairs.particion_color.eq('entrenamiento')
TRAIN=np.flatnonzero(pairs.split_flow2.eq('entrenamiento'))
VAL=np.flatnonzero(pairs.split_flow2.eq('validacion'))
TEST=np.flatnonzero(pairs.split_flow2.eq('prueba'))
PERSON=np.flatnonzero(pairs.split_flow2.eq('val_personas'))
assert min(len(TRAIN),len(VAL),len(TEST))>0,'Train/val/test vacíos; revisar manifiestos'
assert not set(pairs.iloc[TRAIN].color_i)&set(pairs.iloc[np.r_[VAL,TEST]].color_i),'Una estampa cruza train y val/test del Flow 2'
def fixed_panel(indices,n):
 rng=np.random.default_rng(CONFIG['SEMILLA']);groups={int(k):list(g.index) for k,g in pairs.iloc[indices].groupby('color_i')}
 for g in groups.values():rng.shuffle(g)
 keys=list(groups);rng.shuffle(keys);result=[]
 while len(result)<min(n,len(indices)) and any(groups.values()):
  for k in keys:
   if groups[k] and len(result)<n:result.append(groups[k].pop())
 return np.array(result,dtype=int)
VAL_FIXED=fixed_panel(VAL,CONFIG['N_VAL']);TEST_FIXED=fixed_panel(TEST,CONFIG['N_TEST'])
TRAIN_COLORS=np.array(sorted(pairs.iloc[TRAIN].color_i.unique()))
ROWS_BY_COLOR={int(k):g.index.to_numpy() for k,g in pairs.iloc[TRAIN].groupby('color_i')}
display(pairs.groupby(['version','split_flow2']).size().rename('fotos').to_frame())
print('Objetivos train:',len(TRAIN_COLORS),'| val:',len(VAL_FIXED),'| test:',len(TEST_FIXED))
pairs.to_csv(RUN/'pares_flow2.csv',index=False);aliases.to_csv(RUN/'alias_color.csv',index=False)
color_table.to_csv(RUN/'manifest_color.csv',index=False)
def rgba_batch(ci):return RGBA[torch.as_tensor(ci,dtype=torch.long)].to(DEVICE).float()/255.
''',
'features': r'''
detector=cv2.CascadeClassifier(cv2.data.haarcascades+'haarcascade_frontalface_default.xml')
assert not detector.empty()
def crop_photo(row):
 b=Path(row.archivo).read_bytes()
 if CONFIG['CHECK_SHA_FOTOS'] and isinstance(row.get('sha256_foto'),str) and row.sha256_foto:
  assert hashlib.sha256(b).hexdigest()==row.sha256_foto,('SHA foto distinto',row.archivo)
 with Image.open(io.BytesIO(b)) as im:im=im.convert('RGB');im.load();a=np.asarray(im)
 h,w=a.shape[:2];found=detector.detectMultiScale(cv2.cvtColor(a,cv2.COLOR_RGB2GRAY),1.1,5,minSize=(max(16,w//6),)*2)
 if len(found):
  x,y,cw,ch=max(found,key=lambda c:c[2]*c[3]);cx,cy,s=x+cw/2,y+ch/2,max(cw,ch)*1.8
 else:cx,cy,s=w/2,h*.45,min(w,h)*.75
 x0,y0=max(0,int(cx-s/2)),max(0,int(cy-s/2));x1,y1=min(w,int(cx+s/2)),min(h,int(cy+s/2))
 return np.asarray(Image.fromarray(a[y0:y1,x0:x1]).resize((256,256),Image.Resampling.BICUBIC)),bool(len(found)),hashlib.sha256(b).hexdigest()

PHOTO_SIGNATURE=hashlib.sha256((pairs[['id','version','color_i','archivo']].to_csv(index=False)+str(CONFIG['DINO'])).encode()).hexdigest()
cache=RUN/'cache';cache.mkdir(exist_ok=True)
if CONFIG['SMOKE']:
 class ToyDino(nn.Module):
  def __init__(self):super().__init__();self.conv=nn.Conv2d(3,32,14,14)
  def forward(self,pixel_values):
   h=self.conv(pixel_values).flatten(2).transpose(1,2)
   return type('Output',(),{'last_hidden_state':torch.cat([h.mean(1,keepdim=True),h],1)})()
 DINO=ToyDino();DINO_DIM=32
 IMNET_M=torch.tensor([.485,.456,.406])[None,:,None,None];IMNET_S=torch.tensor([.229,.224,.225])[None,:,None,None]
 (RUN/'assets'/'dino').mkdir(exist_ok=True)
 atomic_save(DINO.state_dict(),RUN/'assets'/'dino'/'toy_dino.pt')
else:
 from transformers import AutoModel,AutoImageProcessor
 local=RUN/'assets'/'dino'
 dino_source=CONFIG['DINO_LOCAL'] or (str(local) if (local/'config.json').is_file() else FLOW1_CK.get('dino',CONFIG['DINO']))
 processor=AutoImageProcessor.from_pretrained(dino_source)
 DINO=AutoModel.from_pretrained(dino_source)
 DINO_DIM=DINO.config.hidden_size
 IMNET_M=torch.tensor(processor.image_mean)[None,:,None,None];IMNET_S=torch.tensor(processor.image_std)[None,:,None,None]
 DINO.save_pretrained(RUN/'assets'/'dino');processor.save_pretrained(RUN/'assets'/'dino')
DINO=DINO.to(DINO_DEVICE).eval().requires_grad_(False)
IMNET_M,IMNET_S=IMNET_M.to(DINO_DEVICE),IMNET_S.to(DINO_DEVICE)

# Restaurar Flow 1 exactamente desde su export y el checkpoint DiTo verificado.
if not CONFIG['SMOKE']:
 f1_up=F1_DIR/'upstream';f1_up.mkdir(exist_ok=True)
 for name,content in FLOW1_CK['upstream'].items():(f1_up/name).write_text(content)
 (f1_up/'upstream_registry.py').write_text('def register(name):\n    return lambda cls: cls\n')
 sys.path.insert(0,str(f1_up))
 f1cfg=FLOW1_CK['config'];assert f1cfg['DINO']==FLOW1_CK['dino']
 if str(CONFIG['DINO'])!=str(FLOW1_CK['dino']):raise ValueError('DINO distinto al del Flow 1')
 f1ns=dict(torch=torch,nn=nn,F=F,math=math,CONFIG=f1cfg,DEVICE=DEVICE,amp_ctx=amp_ctx,DIM_DINO=DINO_DIM)
 f1ns['codec']=load_module('f2_mask_codec',f1_up/'codec.py')
 f1ns['renderer']=load_module('f2_mask_renderer',f1_up/'renderer.py')
 f1ns['fm']=load_module('f2_mask_fm',f1_up/'fm.py').FM(sigma_min=1e-5,timescale=1000.)
 f1ns['sampler']=load_module('f2_mask_sampler',f1_up/'samplers.py').FMEulerSampler(f1ns['fm'])
 exec(FLOW1_MODEL_SOURCE,f1ns);exec(MASK_CODEC_SOURCE,f1ns);exec(FLOW1_GENERATE_SOURCE,f1ns)
 F1=f1ns['DiT1'](dim_dino=DINO_DIM).to(DEVICE).eval().requires_grad_(False)
 F1.load_state_dict(FLOW1_CK['ema'],strict=True)
 MASK_CODEC=f1ns['FrozenDiTo']().to(DEVICE).eval().requires_grad_(False)
 ck=torch.load(F1_DIR/'mask_dito.pt',map_location='cpu',weights_only=False)
 assert ck.get('kind')=='dito_base';MASK_CODEC.load_state_dict(ck['ema'],strict=True)
 F1_MEAN=FLOW1_CK['stats']['media'].to(DEVICE);F1_STD=FLOW1_CK['stats']['desv'].to(DEVICE)
 del ck

@torch.no_grad()
def dino_features(rgb):
 x=F.interpolate(rgb.to(DINO_DEVICE),size=(224,224),mode='bilinear',align_corners=False,antialias=True)
 with amp_ctx():h=DINO(pixel_values=(x-IMNET_M)/IMNET_S).last_hidden_state
 return h.float().to(DEVICE)
def pool_dino(h):
 b,n,d=h.shape;s=int(math.sqrt(n-1));assert s*s==n-1
 p=F.adaptive_avg_pool2d(h[:,1:].transpose(1,2).reshape(b,d,s,s),CONFIG['DINO_POOL'])
 return torch.cat([h[:,:1],p.flatten(2).transpose(1,2)],1)
@torch.no_grad()
def predict_mask(h,seeds,rows=None):
 if CONFIG['SMOKE']:
  if rows is None:raise ValueError('Inferencia libre no disponible con modelos SMOKE')
  out=RGBA[torch.as_tensor(pairs.iloc[rows].color_i.to_numpy()),3:4].float().to(DEVICE)/255
  # Cambio local controlado para ejercitar pseudoetiquetas; no es una predicción real.
  out=out.clone();out[:,:,65,44]=1
  return out
 z=f1ns['generate'](F1,h,seeds,w=float(FLOW1_CK.get('cfg_w',1.)),steps=FLOW1_CK['config']['PASOS_FLOW'])
 raw=from_tokens(z)*F1_STD+F1_MEAN
 with amp_ctx():m=MASK_CODEC.decode(raw,seeds,steps=FLOW1_CK['config'].get('PASOS_DITO_FINAL',32))
 return (m>=.5).float()

if (cache/'photos.pt').is_file():
 cp=torch.load(cache/'photos.pt',map_location='cpu',weights_only=False)
 assert cp['signature']==PHOTO_SIGNATURE,'Cambiaron los pares/fotos; usar otra VERSION'
 FOTOS=cp['photos'];TOK=cp['tokens'];TOK_M=cp['tokens_mirror'];pairs['sha_foto_verificado']=cp['hashes'];del cp
else:
 photos=[];hashes=[]
 for _,r in tqdm(pairs.iterrows(),total=len(pairs),desc='Recortando y verificando fotos',unit='foto'):
  a,_,h=crop_photo(r);photos.append(a);hashes.append(h)
 FOTOS=torch.from_numpy(np.stack(photos)).permute(0,3,1,2).contiguous();del photos
 pairs['sha_foto_verificado']=hashes
 TOK=[];TOK_M=[]
 for j in tqdm(range(0,len(pairs),CONFIG['LOTE_PREPARACION']),desc='Cache DINO y espejos',unit='lote'):
  x=FOTOS[j:j+CONFIG['LOTE_PREPARACION']].to(DEVICE).float()/255
  TOK.append(pool_dino(dino_features(x)).half().cpu());TOK_M.append(pool_dino(dino_features(x.flip(-1))).half().cpu())
 TOK=torch.cat(TOK);TOK_M=torch.cat(TOK_M)
 atomic_save(dict(signature=PHOTO_SIGNATURE,photos=FOTOS,tokens=TOK,tokens_mirror=TOK_M,hashes=hashes),cache/'photos.pt')
assert not pairs.groupby('sha_foto_verificado').split_flow2.nunique().gt(1).any(),'Foto idéntica cruza splits'

@torch.no_grad()
def encode_color_arrays(c):
 with amp_ctx():z=COLOR.encode(c[:,:3],c[:,3:4]).float()
 return standardize(z,c[:,3:4])
if (cache/'color_bank.pt').is_file():
 bank=torch.load(cache/'color_bank.pt',map_location='cpu',weights_only=False)
 assert bank['rgba_sha']==hashlib.sha256(RGBA.numpy().tobytes()).hexdigest()
 Z=bank['z'];ZM=bank['z_mirror']
else:
 zs=[];zms=[]
 for j in tqdm(range(0,len(RGBA),CONFIG['LOTE_PREPARACION']),desc='Códigos de color y espejos',unit='lote'):
  c=rgba_batch(np.arange(j,min(j+CONFIG['LOTE_PREPARACION'],len(RGBA))))
  zs.append(encode_color_arrays(c).cpu());zms.append(encode_color_arrays(c.flip(-1)).cpu())
 Z=torch.cat(zs);ZM=torch.cat(zms)
 atomic_save(dict(z=Z,z_mirror=ZM,rgba_sha=hashlib.sha256(RGBA.numpy().tobytes()).hexdigest()),cache/'color_bank.pt')
assert Z.shape[1:]==(8,16,16) and torch.isfinite(Z).all()
atomic_save(RGBA,cache/'rgba.pt');pairs.to_csv(RUN/'pares_flow2.csv',index=False)
print('Fotos:',tuple(FOTOS.shape),'| tokens:',tuple(TOK.shape),'| códigos:',tuple(Z.shape))
''',
'predictions': r'''
def pseudo_color(c,mhat):
 # Solo extensiones locales de color uniforme. No se afirma correspondencia semántica global.
 rgb=c[:3].transpose(1,2,0).astype(np.float32)/255;gt=c[3]>127;mh=mhat.astype(bool)
 dice=2*(gt&mh).sum()/max(1,gt.sum()+mh.sum())
 if dice<CONFIG['PRED_DICE_MIN']:return None,dict(aceptado=False,motivo='dice',dice=float(dice))
 if not .85<=mh.sum()/max(1,gt.sum())<=1.15:return None,dict(aceptado=False,motivo='ocupacion',dice=float(dice))
 ng,labels_gt=cv2.connectedComponents(gt.astype(np.uint8),connectivity=8)
 npred,_=cv2.connectedComponents(mh.astype(np.uint8),connectivity=8)
 if ng!=npred:return None,dict(aceptado=False,motivo='componentes',dice=float(dice))
 for lab in range(1,ng):
  region=labels_gt==lab
  if (region&mh).sum()<.5*region.sum():return None,dict(aceptado=False,motivo='componente_perdido',dice=float(dice))
 dist,near=cv2.distanceTransformWithLabels((~gt).astype(np.uint8),cv2.DIST_L2,5,labelType=cv2.DIST_LABEL_PIXEL)
 added=mh&~gt
 if added.any() and dist[added].max()>CONFIG['PRED_DIST_MAX']:
  return None,dict(aceptado=False,motivo='extension_lejana',dice=float(dice))
 lookup=np.zeros((int(near.max())+1,3),np.float32);lookup[near[gt]]=rgb[gt]
 out=rgb.copy()
 for y,x in np.argwhere(added):
  ys=slice(max(0,y-2),min(128,y+3));xs=slice(max(0,x-2),min(128,x+3))
  colors=rgb[ys,xs][gt[ys,xs]]
  if not len(colors) or np.max(colors.max(0)-colors.min(0))>CONFIG['PRED_COLOR_SPREAD']:
   return None,dict(aceptado=False,motivo='color_ambiguo',dice=float(dice))
  out[y,x]=lookup[near[y,x]]
 result=np.zeros((4,128,128),np.uint8)
 result[:3]=(np.clip(out,0,1)*255).round().astype(np.uint8).transpose(2,0,1)*mh
 result[3]=mh.astype(np.uint8)*255
 return result,dict(aceptado=True,motivo='pseudo_extension_local',dice=float(dice),nuevos=int(added.sum()))

train_pred_rows=fixed_panel(TRAIN,CONFIG['N_PRED_TRAIN']) if CONFIG['PRED_ENABLE'] else np.array([],dtype=int)
PRED_ROWS=np.unique(np.r_[train_pred_rows,VAL_FIXED,TEST_FIXED]).astype(int)
pred_signature=hashlib.sha256((PHOTO_SIGNATURE+'|'+str(sha_file(F1_DIR/'flow1_dito.pt') if not CONFIG['SMOKE'] else 'smoke')+
   '|'+','.join(map(str,PRED_ROWS))+'|'+json.dumps({k:v for k,v in CONFIG.items() if k.startswith('PRED_')},sort_keys=True)).encode()).hexdigest()
if (cache/'predictions.pt').is_file():
 pred_cache=torch.load(cache/'predictions.pt',map_location='cpu',weights_only=False)
 assert pred_cache['signature']==pred_signature,'Cache de predicciones incompatible; usar otra VERSION'
 PRED_MASKS=pred_cache['masks'];PSEUDO=pred_cache['pseudo'];PRED_AUDIT=pd.DataFrame(pred_cache['audit'])
else:
 PRED_MASKS={};PSEUDO={};audit=[]
 for j in tqdm(range(0,len(PRED_ROWS),CONFIG['LOTE_PREPARACION']),desc='Flow 1 congelado · cache de máscaras',unit='lote'):
  rows=PRED_ROWS[j:j+CONFIG['LOTE_PREPARACION']]
  x=FOTOS[torch.as_tensor(rows)].to(DEVICE).float()/255
  h=dino_features(x);seeds=[CONFIG['SEMILLA']+int(i) for i in rows]
  m=predict_mask(h,seeds,rows).byte().cpu().numpy()
  for k,i in enumerate(rows):
   mh=m[k,0];PRED_MASKS[int(i)]=torch.from_numpy(mh[None].copy())
   c=RGBA[int(pairs.iloc[i].color_i)].numpy();pseudo,info=pseudo_color(c,mh)
   info.update(row=int(i),color_i=int(pairs.iloc[i].color_i),split=str(pairs.iloc[i].split_flow2))
   audit.append(info)
   if pseudo is not None:PSEUDO[int(i)]=torch.from_numpy(pseudo)
 PRED_AUDIT=pd.DataFrame(audit)
 atomic_save(dict(signature=pred_signature,masks=PRED_MASKS,pseudo=PSEUDO,audit=audit),cache/'predictions.pt')
PRED_AUDIT.to_csv(RUN/'auditoria_pseudoetiquetas.csv',index=False)
PRED_TRAIN=np.array([i for i in train_pred_rows if int(i) in PSEUDO],dtype=int)
PRED_VAL=np.array([i for i in VAL_FIXED if int(i) in PSEUDO],dtype=int)
accepted_colors=pairs.iloc[PRED_TRAIN].color_i.nunique() if len(PRED_TRAIN) else 0
CAN_PRED=accepted_colors>=CONFIG['MIN_PRED_COLORES']
display(PRED_AUDIT.groupby(['split','aceptado']).size().rename('mascaras').to_frame())
print('Colores train con pseudoetiqueta aceptada:',accepted_colors,'| adaptación con predicciones:',CAN_PRED)
if not CAN_PRED:print('Se conservarán las deformaciones alineadas; no se fuerzan etiquetas para máscaras diferentes.')
PZ={};PZM={}
for j in tqdm(range(0,len(PSEUDO),CONFIG['LOTE_PREPARACION']),desc='Codificando pseudoetiquetas aceptadas',unit='lote'):
 keys=list(PSEUDO)[j:j+CONFIG['LOTE_PREPARACION']]
 c=torch.stack([PSEUDO[i] for i in keys]).to(DEVICE).float()/255
 z=encode_color_arrays(c).cpu();zm=encode_color_arrays(c.flip(-1)).cpu()
 for k,i in enumerate(keys):PZ[i]=z[k];PZM[i]=zm[k]
atomic_save(dict(z=PZ,z_mirror=PZM),cache/'pseudo_bank.pt')
# La preparación no deja modelos generativos extra en la memoria de entrenamiento.
if not CONFIG['SMOKE']:
 F1.cpu();MASK_CODEC.cpu()
DINO.cpu()
if AMP:torch.cuda.empty_cache()
''',
'models': r'''def rope_2d(dim_cabeza, lado=16):
    d = dim_cabeza // 4
    frec = 1.0 / (10000 ** (torch.arange(d) / d))
    y, x = torch.meshgrid(torch.arange(lado), torch.arange(lado), indexing="ij")
    ang = torch.cat([y.flatten()[:, None] * frec, x.flatten()[:, None] * frec], 1)   # 256 × dim/2
    return torch.cos(ang), torch.sin(ang)

def aplicar_rope(x, cos, sin):                              # x: B×H×N×D
    x1, x2 = x[..., 0::2], x[..., 1::2]
    return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], -1).flatten(-2)

class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.peso, self.eps = nn.Parameter(torch.ones(d)), eps

    def forward(self, x):
        return x * torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps).to(x.dtype) * self.peso.to(x.dtype)

def modular(x, shift, scale):
    return x * (1 + scale[:, None]) + shift[:, None]

def embedding_tiempo(t, dim):
    mitad = dim // 2
    frec = torch.exp(-math.log(10000) * torch.arange(mitad, device=t.device) / mitad)
    a = t[:, None] * 1000 * frec[None]
    return torch.cat([torch.cos(a), torch.sin(a)], 1)

class Attention2(nn.Module):
 def __init__(self,dim,heads,dropout):
  super().__init__();self.heads=heads
  self.q=nn.Linear(dim,dim);self.kv=nn.Linear(dim,2*dim)
  self.nq=RMSNorm(dim//heads);self.nk=RMSNorm(dim//heads)
  self.out=nn.Sequential(nn.Linear(dim,dim),nn.Dropout(dropout))
 def forward(self,x,context=None,rope=None):
  b,n,d=x.shape;c=x if context is None else context
  q=self.q(x).reshape(b,n,self.heads,-1).transpose(1,2)
  k,v=self.kv(c).reshape(b,len(c[0]),2,self.heads,-1).permute(2,0,3,1,4)
  q,k=self.nq(q),self.nk(k)
  if rope is not None:q,k=aplicar_rope(q,*rope),aplicar_rope(k,*rope)
  y=F.scaled_dot_product_attention(q,k,v)
  return self.out(y.transpose(1,2).reshape(b,n,d))
class CrossBlock(nn.Module):
 def __init__(self,d,heads,ratio,drop):
  super().__init__();self.norms=nn.ModuleList([nn.LayerNorm(d,elementwise_affine=False) for _ in range(3)])
  self.selfatt=Attention2(d,heads,drop);self.cross=Attention2(d,heads,drop)
  self.mlp=nn.Sequential(nn.Linear(d,int(d*ratio)),nn.GELU(approximate='tanh'),nn.Dropout(drop),nn.Linear(int(d*ratio),d))
  self.ada=nn.Sequential(nn.SiLU(),nn.Linear(d,9*d));nn.init.zeros_(self.ada[-1].weight);nn.init.zeros_(self.ada[-1].bias)
 def forward(self,x,time,photo,rope):
  s1,e1,g1,s2,e2,g2,s3,e3,g3=self.ada(time).chunk(9,1)
  x=x+g1[:,None]*self.selfatt(modular(self.norms[0](x),s1,e1),rope=rope)
  x=x+g2[:,None]*self.cross(modular(self.norms[1](x),s2,e2),context=photo)
  return x+g3[:,None]*self.mlp(modular(self.norms[2](x),s3,e3))
class SingleBlock(nn.Module):
 def __init__(self,d,heads,ratio,drop):
  super().__init__();self.n1=nn.LayerNorm(d,elementwise_affine=False);self.n2=nn.LayerNorm(d,elementwise_affine=False)
  self.att=Attention2(d,heads,drop)
  self.mlp=nn.Sequential(nn.Linear(d,int(d*ratio)),nn.GELU(approximate='tanh'),nn.Dropout(drop),nn.Linear(int(d*ratio),d))
  self.ada=nn.Sequential(nn.SiLU(),nn.Linear(d,6*d));nn.init.zeros_(self.ada[-1].weight);nn.init.zeros_(self.ada[-1].bias)
 def forward(self,x,time,rope):
  s1,e1,g1,s2,e2,g2=self.ada(time).chunk(6,1)
  x=x+g1[:,None]*self.att(modular(self.n1(x),s1,e1),rope=rope)
  return x+g2[:,None]*self.mlp(modular(self.n2(x),s2,e2))
class PhotoColor(nn.Module):
 def __init__(self,grid):
  super().__init__();self.grid=grid
  self.net=nn.Sequential(nn.Conv2d(3,32,5,2,2),nn.SiLU(),nn.Conv2d(32,64,3,2,1),nn.SiLU(),nn.Conv2d(64,96,3,2,1),nn.SiLU())
 def forward(self,rgb):
  x=F.interpolate(rgb,size=(128,128),mode='bilinear',align_corners=False)
  f=F.adaptive_avg_pool2d(self.net(x),self.grid)
  mean=F.adaptive_avg_pool2d(x,self.grid)
  std=(F.adaptive_avg_pool2d(x*x,self.grid)-mean.square()).clamp_min(1e-6).sqrt()
  h=torch.cat([f,mean,std],1).flatten(2).transpose(1,2)
  return torch.cat([h.mean(1,keepdim=True),h],1)
class Flow2DiT(nn.Module):
 def __init__(self,arch,cfg,dino_dim):
  super().__init__();self.arch=arch;self.cfg=dict(cfg);d=cfg['DIM'];self.dim=d
  self.mask_encoder=nn.Sequential(nn.Conv2d(1,32,3,2,1),nn.SiLU(),nn.Conv2d(32,32,3,2,1),nn.SiLU(),nn.Conv2d(32,32,3,2,1),nn.SiLU())
  self.color=PhotoColor(cfg['DINO_POOL']);self.dino_proj=nn.Sequential(nn.LayerNorm(dino_dim),nn.Linear(dino_dim,d))
  self.rgb_proj=nn.Linear(102,d);self.null=nn.Parameter(torch.zeros(1,1,d))
  self.time=nn.Sequential(nn.Linear(d,d),nn.SiLU(),nn.Linear(d,d))
  self.pos=nn.Parameter(torch.randn(1,256,d)*.02);self.types=nn.Parameter(torch.randn(3,d)*.02)
  self.input=nn.Linear(41 if arch=='cross' else 8,d)
  self.mask_proj=nn.Linear(33,d)
  if arch=='cross':self.blocks=nn.ModuleList([CrossBlock(d,cfg['CABEZAS'],cfg['MLP'],cfg['DROPOUT']) for _ in range(cfg['PROF_CROSS'])])
  elif arch=='single':self.blocks=nn.ModuleList([SingleBlock(d,cfg['CABEZAS'],cfg['MLP'],cfg['DROPOUT']) for _ in range(cfg['PROF_SINGLE'])])
  else:raise ValueError(arch)
  self.final_norm=nn.LayerNorm(d,elementwise_affine=False);self.final_ada=nn.Sequential(nn.SiLU(),nn.Linear(d,2*d));self.out=nn.Linear(d,8)
  for layer in [self.final_ada[-1],self.out]:nn.init.zeros_(layer.weight);nn.init.zeros_(layer.bias)
  cos,sin=rope_2d(d//cfg['CABEZAS']);self.register_buffer('cos',cos,persistent=False);self.register_buffer('sin',sin,persistent=False)
 def forward(self,xt,t,dino,rgb,mask,drop=None):
  occ=(F.adaptive_max_pool2d(mask,16)>.5).float().flatten(2).transpose(1,2)
  shape=self.mask_encoder(mask).flatten(2).transpose(1,2)
  shape=torch.cat([shape,occ],-1)
  photo=self.dino_proj(dino)+self.rgb_proj(self.color(rgb))
  if drop is not None:photo=torch.where(drop[:,None,None],self.null.expand_as(photo),photo)
  time=self.time(embedding_tiempo(t,self.dim))
  if self.arch=='cross':
   x=self.input(torch.cat([xt,shape],-1))+self.pos
   for block in self.blocks:x=block(x,time,photo,(self.cos,self.sin))
  else:
   noisy=self.input(xt)+self.pos+self.types[0]
   shapes=self.mask_proj(shape)+self.pos+self.types[1]
   x=torch.cat([noisy,shapes,photo+self.types[2]],1)
   n=photo.shape[1]
   cos=torch.cat([self.cos,self.cos,torch.ones(n,self.cos.shape[1],device=xt.device)],0)
   sin=torch.cat([self.sin,self.sin,torch.zeros(n,self.sin.shape[1],device=xt.device)],0)
   for block in self.blocks:x=block(x,time,(cos,sin))
   x=x[:,:256]
  shift,scale=self.final_ada(time).chunk(2,1)
  return self.out(modular(self.final_norm(x),shift,scale))*occ
''',
'sampling': r'''def masked_mean(x,m):return (x*m).sum((1,2,3))/(x.shape[1]*m.sum((1,2,3)).clamp_min(1))
def mse_mask(x,y,m):return masked_mean((x-y).square(),m)
def rgb_to_lab(rgb):
    rgb=rgb.clamp(0,1)
    lin=torch.where(rgb<=0.04045,rgb/12.92,((rgb+0.055)/1.055).pow(2.4))
    mat=rgb.new_tensor([[.4124564,.3575761,.1804375],[.2126729,.7151522,.0721750],[.0193339,.1191920,.9503041]])
    xyz=torch.einsum('ij,bjhw->bihw',mat,lin)/rgb.new_tensor([.95047,1.,1.08883])[None,:,None,None]
    delta=6/29
    f=torch.where(xyz>delta**3,xyz.clamp_min(1e-12).pow(1/3),xyz/(3*delta**2)+4/29)
    return torch.stack([116*f[:,1]-16,500*(f[:,0]-f[:,1]),200*(f[:,1]-f[:,2])],1)
def ssim_common_background(x,y,m):
    # La ventana de SSIM ve EL MISMO fondo en objetivo y reconstrucción.
    x=x*m+0.5*(1-m);y=y*m+0.5*(1-m)
    g=torch.exp(-(torch.arange(11,device=x.device,dtype=x.dtype)-5).square()/(2*1.5**2));g=g/g.sum()
    w=(g[:,None]*g[None,:]).expand(3,1,11,11).contiguous()
    conv=lambda a:F.conv2d(a,w,padding=5,groups=3)
    mx,my=conv(x),conv(y);sx=(conv(x*x)-mx*mx).clamp_min(0);sy=(conv(y*y)-my*my).clamp_min(0)
    sxy=conv(x*y)-mx*my
    ss=((2*mx*my+.01**2)*(2*sxy+.03**2))/((mx*mx+my*my+.01**2)*(sx+sy+.03**2))
    return masked_mean(ss,m)
def metricas(x,y,m):
    x=x.float();y=y.float();m=m.float();mse=mse_mask(x,y,m)
    de=(rgb_to_lab(x)-rgb_to_lab(y)).square().sum(1,keepdim=True).sqrt()
    return dict(mse=mse,mae=masked_mean((x-y).abs(),m),
                psnr=-10*torch.log10(mse.clamp_min(1e-10)),
                ssim=ssim_common_background(x,y,m),deltaE76=masked_mean(de,m))



def conditions(rows,mirror=None,photo_override=None):
 source=rows if photo_override is None else photo_override
 source=np.asarray(source,dtype=int)
 if mirror is None:mirror=torch.zeros(len(rows),dtype=torch.bool,device=DEVICE)
 a=TOK[torch.as_tensor(source)].to(DEVICE).float();b=TOK_M[torch.as_tensor(source)].to(DEVICE).float()
 h=torch.where(mirror[:,None,None],b,a)
 rgb=FOTOS[torch.as_tensor(source)].to(DEVICE).float()/255
 rgb=torch.where(mirror[:,None,None,None],rgb.flip(-1),rgb)
 return h,rgb
@torch.no_grad()
def generate2(model,h,rgb,m,seeds,w=1.,steps=None,method='euler'):
 model.eval();steps=steps or CONFIG['PASOS_FLOW'];a=to_tokens(active(m))
 x=torch.stack([torch.randn((256,8),generator=torch.Generator().manual_seed(int(s))) for s in seeds]).to(DEVICE)*a
 def vel(x,t):
  tt=torch.full((len(x),),float(t),device=DEVICE)
  with amp_ctx():
   v=model(x,tt,h,rgb,m,torch.zeros(len(x),dtype=torch.bool,device=DEVICE)).float()
   if w!=1:
    u=model(x,tt,h,rgb,m,torch.ones(len(x),dtype=torch.bool,device=DEVICE)).float();v=u+w*(v-u)
  return v
 for j in range(steps):
  v=vel(x,j/steps)
  x=x+(v+vel(x+v/steps,(j+1)/steps))/(2*steps) if method=='heun' else x+v/steps
  x=x*a
 if not torch.isfinite(x).all():raise FloatingPointError('Latente generado no finito')
 return x

def macro(df,metric='mse'):
 return float(df.groupby('color_i')[metric].mean().mean())
@torch.no_grad()
def evaluate2(model,indices,mode='real',w=1.,shuffle=False,keep=False,progress=False):
 model.eval();rows=[];images=[]
 indices=np.asarray(indices,dtype=int)
 if mode=='pseudo':indices=np.array([i for i in indices if int(i) in PSEUDO],dtype=int)
 shuffled=np.roll(indices,1) if shuffle else indices
 for j in tqdm(range(0,len(indices),CONFIG['LOTE_EVAL']),desc=f'Evaluando {mode}',unit='lote',leave=False,disable=not progress):
  ii=indices[j:j+CONFIG['LOTE_EVAL']];ci=pairs.iloc[ii].color_i.to_numpy()
  c=rgba_batch(ci)
  if mode=='pseudo':c=torch.stack([PSEUDO[int(i)] for i in ii]).to(DEVICE).float()/255
  if mode=='generada':m=torch.stack([PRED_MASKS[int(i)] for i in ii]).to(DEVICE).float()
  else:m=c[:,3:4]
  h,photo=conditions(ii,photo_override=shuffled[j:j+len(ii)])
  seeds=[CONFIG['SEMILLA']+int(i) for i in ii]
  z=generate2(model,h,photo,m,seeds,w)
  with amp_ctx():y=color_decode(z,m).float()
  if mode!='generada':scores=metricas(c[:,:3],y,m)
  else:
   # Intersección espacial: NO representa una correspondencia semántica certificada.
   intersection=m*c[:,3:4]
   proxy=mse_mask(c[:,:3],y,intersection)
   proxy=torch.where(intersection.sum((1,2,3))>0,proxy,torch.full_like(proxy,float('nan')))
   scores={'mse_interseccion_proxy':proxy,
           'pixeles_interseccion':intersection.sum((1,2,3)),
           'mascara_vacia':(m.sum((1,2,3))==0).float(),
           'dice_mascara':2*(m*c[:,3:4]).sum((1,2,3))/(m.sum((1,2,3))+c[:,3:4].sum((1,2,3))).clamp_min(1)}
  for k,i in enumerate(ii):
   rows.append(dict(row=int(i),id=pairs.iloc[i].id,color_i=int(ci[k]),version=pairs.iloc[i].version,
    codec_color_vio_objetivo=bool(pairs.iloc[i].codec_color_vio_objetivo),
    **{name:float(v[k].cpu()) for name,v in scores.items()}))
  if keep:images.append(y.cpu())
 return pd.DataFrame(rows),torch.cat(images) if images else None

@torch.no_grad()
def reference_scores(indices):
 out=[]
 for j in range(0,len(indices),CONFIG['LOTE_EVAL']):
  ii=indices[j:j+CONFIG['LOTE_EVAL']];ci=pairs.iloc[ii].color_i.to_numpy();c=rgba_batch(ci);m=c[:,3:4]
  z=to_tokens(Z[torch.as_tensor(ci)].to(DEVICE));base=torch.zeros_like(z)
  with amp_ctx():oracle=color_decode(z,m).float();mean_color=color_decode(base,m).float()
  a=mse_mask(c[:,:3],oracle,m);b=mse_mask(c[:,:3],mean_color,m)
  for k,i in enumerate(ii):out.append(dict(row=int(i),color_i=int(ci[k]),codec_mse=float(a[k].cpu()),base_mse=float(b[k].cpu())))
 return pd.DataFrame(out)
REF_VAL=reference_scores(VAL_FIXED)
CODEC_MSE=macro(REF_VAL,'codec_mse');BASE_MSE=macro(REF_VAL,'base_mse')
print('Referencias en el mismo panel · codec:',CODEC_MSE,'| código medio:',BASE_MSE)
''',
'training': r'''
def warp_rgba(c):
 n=len(c);ang=(torch.rand(n,device=DEVICE)*2-1)*math.radians(3)
 scale=1+(torch.rand(n,device=DEVICE)*2-1)*.03
 xy=(torch.rand(n,2,device=DEVICE)*2-1)*(2/64)
 co,si=torch.cos(ang)/scale,torch.sin(ang)/scale
 theta=torch.stack([torch.stack([co,-si,xy[:,0]],1),torch.stack([si,co,xy[:,1]],1)],1)
 q=F.grid_sample(c,F.affine_grid(theta,c.shape,align_corners=False),align_corners=False,padding_mode='zeros')
 alpha=q[:,3:4];m=(alpha>.5).float()
 return torch.cat([(q[:,:3]/alpha.clamp_min(1e-3)).clamp(0,1)*m,m],1)

class RowSampler:
 def __init__(self):self.rng=np.random.default_rng(CONFIG['SEMILLA']+123)
 def get(self,n,p_pred=0.,p_aug=0.):
  rows=[];modes=[]
  for _ in range(n):
   if CAN_PRED and self.rng.random()<p_pred:
    rows.append(int(self.rng.choice(PRED_TRAIN)));modes.append('pred')
   else:
    ci=int(self.rng.choice(TRAIN_COLORS));rows.append(int(self.rng.choice(ROWS_BY_COLOR[ci])))
    modes.append('aug' if self.rng.random()<p_aug else 'clean')
  return np.array(rows,dtype=int),np.array(modes)
 def state_dict(self):return self.rng.bit_generator.state
 def load_state_dict(self,s):self.rng.bit_generator.state=s

def training_batch(sampler,p_pred,p_aug):
 rows,modes=sampler.get(CONFIG['LOTE'],p_pred,p_aug);ci=pairs.iloc[rows].color_i.to_numpy()
 c=rgba_batch(ci);flip=(torch.rand(len(rows),device=DEVICE)<.5) if CONFIG['ESPEJO'] else torch.zeros(len(rows),dtype=torch.bool,device=DEVICE)
 z=Z[torch.as_tensor(ci)].to(DEVICE);zm=ZM[torch.as_tensor(ci)].to(DEVICE)
 for j,i in enumerate(rows):
  if modes[j]=='pred':
   c[j]=PSEUDO[int(i)].to(DEVICE).float()/255
   z[j]=PZ[int(i)].to(DEVICE);zm[j]=PZM[int(i)].to(DEVICE)
 c=torch.where(flip[:,None,None,None],c.flip(-1),c)
 target=torch.where(flip[:,None,None,None],zm,z)
 aug=np.flatnonzero(modes=='aug')
 if len(aug):
  jj=torch.as_tensor(aug,device=DEVICE);cc=warp_rgba(c[jj]);c[jj]=cc
  target[jj]=encode_color_arrays(cc)
 h,photo=conditions(rows,flip)
 return rows,modes,c,h,photo,to_tokens(target)

def capture_rng():
 return dict(py=random.getstate(),np=np.random.get_state(),torch=torch.get_rng_state(),
             cuda=torch.cuda.get_rng_state_all() if AMP else None)
def restore_rng(s):
 random.setstate(s['py']);np.random.set_state(s['np']);torch.set_rng_state(s['torch'])
 if AMP and s['cuda'] is not None:
  for i,rng in enumerate(s['cuda'][:torch.cuda.device_count()]):torch.cuda.set_rng_state(rng,i)
@torch.no_grad()
def ema_update(ema,model,step):
 decay=min(CONFIG['EMA'],(1+step)/(10+step))
 for a,b in zip(ema.parameters(),model.parameters()):a.lerp_(b.detach(),1-decay)
 for a,b in zip(ema.buffers(),model.buffers()):a.copy_(b)
def lr_factor(step,total):
 if step<CONFIG['WARMUP']:return (step+1)/max(1,CONFIG['WARMUP'])
 t=(step-CONFIG['WARMUP'])/max(1,total-CONFIG['WARMUP'])
 return CONFIG['LR_FINAL']+(1-CONFIG['LR_FINAL'])*.5*(1+math.cos(math.pi*min(1.,t)))

@torch.no_grad()
def validation(model,curriculum=False):
 clean,_=evaluate2(model,VAL_FIXED)
 shuffle,_=evaluate2(model,VAL_FIXED,shuffle=True)
 mse=macro(clean);shuf=macro(shuffle)
 info=dict(mse=mse,psnr=macro(clean,'psnr'),deltaE76=macro(clean,'deltaE76'),
           mse_foto_barajada=shuf,ratio_codec=mse/max(1e-10,CODEC_MSE))
 if curriculum and len(PRED_VAL)>=CONFIG['MIN_VAL_PSEUDO']:
  pseudo,_=evaluate2(model,PRED_VAL,mode='pseudo');info['mse_pseudo']=macro(pseudo)
  info['score']=.7*mse+.3*info['mse_pseudo']
 else:info['mse_pseudo']=None;info['score']=mse
 return info

class Curriculum:
 def __init__(self,clean_hist=None):
  self.stage='A';self.entered=0;self.history=list(clean_hist or []);self.pred_disabled=False
 def ratios(self,step):
  if self.stage=='A':return 0.,0.
  if self.stage=='B':return 0.,CONFIG['P_AUG']
  p=CONFIG['PRED_MAX']*min(1.,(step-self.entered)/max(1,CONFIG['RAMPA_PRED']))
  # Se mantiene >=70% limpio; la corrupción alineada ocupa el resto del presupuesto.
  return p,max(0.,(CONFIG['P_AUG']-p)/max(1e-6,1-p))
 def update(self,step,info):
  self.history.append(info['mse'])
  if CONFIG['SMOKE']:
   if self.stage=='A' and step>=2:self.stage='B';self.entered=step
   elif self.stage=='B' and step>=4 and CAN_PRED:self.stage='C';self.entered=step
   return
  count=CONFIG['PLATEAU_EVALS']
  plateau=len(self.history)>=count and (max(self.history[-count:])-min(self.history[-count:]))/max(1e-9,min(self.history[-count:]))<CONFIG['PLATEAU_MEJORA']
  useful=info['mse']<(1-CONFIG['MEJORA_BASE'])*BASE_MSE
  photo=info['mse']<(1-CONFIG['MEJORA_FOTO'])*info['mse_foto_barajada']
  if self.stage=='A' and CONFIG['PASOS_LIMPIOS']+step>=CONFIG['MIN_PASOS_A'] and plateau and useful and (photo or not CONFIG['REQUERIR_USO_FOTO']):
   self.stage='B';self.entered=step
  elif self.stage=='B' and step-self.entered>=CONFIG['MIN_PASOS_B'] and CAN_PRED and not self.pred_disabled:
   self.stage='C';self.entered=step
  elif self.stage=='C' and info['mse']>min(self.history)*(1+CONFIG['DEGRADACION_MAX']):
   self.stage='B';self.entered=step;self.pred_disabled=True
 def state_dict(self):return dict(stage=self.stage,entered=self.entered,history=self.history,pred_disabled=self.pred_disabled)
 def load_state_dict(self,s):self.__dict__.update(s)

DATA_SHA=hashlib.sha256((pairs[['id','version','color_i','split_flow2','sha_foto_verificado']].to_csv(index=False)+
      hashlib.sha256(RGBA.numpy().tobytes()).hexdigest()).encode()).hexdigest()
CODE_SHA=hashlib.sha256(json.dumps(dict(sources=RUNTIME_SOURCES,flow1=FLOW1_MODEL_SOURCE,mask=MASK_CODEC_SOURCE),sort_keys=True).encode()).hexdigest()
def run_signature(arch,phase):
 # Ubicaciones físicas no forman parte de la firma: el ZIP completo puede moverse.
 skip={'RAIZ','SALIDA','REANUDAR_DE','COLOR_DIR','CK_FLOW1','CK_DITO','PARES_AUDITADOS','DINO_LOCAL','REANUDAR'}
 cfg={k:v for k,v in CONFIG.items() if k not in skip}
 return hashlib.sha256(json.dumps(dict(config=cfg,arch=arch,phase=phase,data=DATA_SHA,code=CODE_SHA),sort_keys=True).encode()).hexdigest()

def train_run(arch,phase,initial=None,clean_hist=None):
 total=CONFIG['PASOS_LIMPIOS'] if phase=='clean' else CONFIG['PASOS_CURRICULUM']
 folder=RUN/'runs'/f'{arch}_{phase}';folder.mkdir(parents=True,exist_ok=True)
 signature=run_signature(arch,phase);seed_all(CONFIG['SEMILLA'])
 model=Flow2DiT(arch,CONFIG,DINO_DIM).to(DEVICE)
 if initial is not None:model.load_state_dict(initial,strict=True)
 ema=copy.deepcopy(model).eval().requires_grad_(False)
 opt=torch.optim.AdamW(model.parameters(),lr=CONFIG['LR'] if phase=='clean' else CONFIG['LR_CURRICULUM'],weight_decay=CONFIG['WD'])
 scaler=torch.amp.GradScaler('cuda',enabled=AMP);sampler=RowSampler();controller=Curriculum(clean_hist)
 step=0;best=float('inf');history=[];vals=[];previous=0.;skipped=0
 def state():
  return dict(arch=arch,phase=phase,step=step,best=best,ema=cpu_weights(ema),model=cpu_weights(model),
   opt=opt.state_dict(),scaler=scaler.state_dict(),rng=capture_rng(),sampler=sampler.state_dict(),
   controller=controller.state_dict(),history=history,validations=vals,seconds=previous+time.monotonic()-start,
   signature=signature,config=CONFIG,data_sha=DATA_SHA,code_sha=CODE_SHA,skipped=skipped,dino_dim=DINO_DIM,
   artifact_role='flow2_training',smoke=CONFIG['SMOKE'])
 last=folder/'last.pt'
 if CONFIG['REANUDAR'] and last.is_file():
  ck=torch.load(last,map_location='cpu',weights_only=False)
  assert ck['signature']==signature,'Código, datos o plan de entrenamiento distintos. Usar otra VERSION.'
  model.load_state_dict(ck['model']);ema.load_state_dict(ck['ema']);opt.load_state_dict(ck['opt']);scaler.load_state_dict(ck['scaler'])
  sampler.load_state_dict(ck['sampler']);controller.load_state_dict(ck['controller']);restore_rng(ck['rng'])
  step=ck['step'];best=ck['best'];history=ck['history'];vals=ck['validations'];previous=ck['seconds'];skipped=ck['skipped'];del ck
  assert (folder/'best.pt').is_file(),'Falta best asociado al last'
  print(arch,phase,': reanudando',step,'de',total)
 start=time.monotonic()
 # La fase de adaptación también compite contra el modelo de entrada sin modificar.
 if initial is not None and not (folder/'best.pt').is_file():
  info=validation(ema,curriculum=True);best=info['score'];vals.append(dict(step=0,stage='A',**info))
  atomic_save(state(),folder/'best.pt');atomic_save(state(),last)
 params=sum(p.numel() for p in model.parameters());print(arch,phase,':',round(params/1e6,2),'M parámetros')
 bar=tqdm(total=total,initial=step,desc=f'Flow 2 · {arch} · {phase}',unit='step',mininterval=1.)
 sums={};window=0;overflows=0
 try:
  while step<total:
   model.train();opt.zero_grad(set_to_none=True)
   lr=(CONFIG['LR'] if phase=='clean' else CONFIG['LR_CURRICULUM'])*lr_factor(step,total)
   for g in opt.param_groups:g['lr']=lr
   pp,pa=(0.,0.) if phase=='clean' else controller.ratios(step)
   stats={}
   for micro in range(CONFIG['ACUMULAR']):
    rows,modes,c,h,photo,target=training_batch(sampler,pp,pa);m=c[:,3:4];a=to_tokens(active(m))
    noise=torch.randn_like(target)*a;t=torch.randn(len(rows),device=DEVICE).sigmoid()
    xt=(1-t[:,None,None])*noise+t[:,None,None]*target
    drop=torch.rand(len(rows),device=DEVICE)<CONFIG['P_SIN_FOTO']
    with amp_ctx():pred=model(xt,t,h,photo,m,drop)
    diff=(pred.float()-(target-noise)).square()*a
    fm=(diff.sum((1,2))/(8*a.sum((1,2))).clamp_min(1)).mean()
    aux=torch.zeros((),device=DEVICE)
    selected=torch.nonzero((t>=.5)&~drop).flatten()[:CONFIG['AUX_MAX']]
    if CONFIG['AUX_RGB'] and step%CONFIG['AUX_CADA']==0 and len(selected):
     estimate=xt[selected]+(1-t[selected,None,None])*pred[selected].float()
     with amp_ctx():out=color_decode(estimate,m[selected]).float()
     aux=mse_mask(c[selected,:3],out,m[selected]).mean()
    loss=fm+CONFIG['AUX_RGB']*aux
    if not torch.isfinite(loss):raise FloatingPointError('Pérdida no finita; last conserva el estado anterior')
    scaler.scale(loss/CONFIG['ACUMULAR']).backward()
    for k,v in dict(loss=loss.detach(),fm=fm.detach(),rgb=aux.detach(),pred_fraction=float(np.mean(modes=='pred'))).items():
     stats[k]=stats.get(k,0)+float(v)/CONFIG['ACUMULAR']
   scaler.unscale_(opt);gnorm=torch.nn.utils.clip_grad_norm_(model.parameters(),CONFIG['CLIP'])
   if not torch.isfinite(gnorm):
    if not AMP:raise FloatingPointError('Gradiente no finito')
    scaler.step(opt);scaler.update();skipped+=1;overflows+=1
    if overflows>=50:raise FloatingPointError('50 overflows AMP consecutivos')
    continue
   overflows=0;scaler.step(opt);scaler.update();ema_update(ema,model,step);step+=1;bar.update(1);window+=1
   for k,v in stats.items():sums[k]=sums.get(k,0)+v
   if step%CONFIG['LOG_CADA']==0 or step==total:
    row=dict(step=step,stage='A' if phase=='clean' else controller.stage,**{k:v/window for k,v in sums.items()},
      lr=lr,grad_norm=float(gnorm.cpu()),amp_skips=skipped)
    history.append(row);bar.set_postfix(loss=f'{row["loss"]:.4f}',etapa=row['stage'],pred=f'{row["pred_fraction"]:.0%}')
    if step%CONFIG['LOG_CADA']==0:
     tqdm.write(f'{arch} {phase} · step {step}: loss={row["loss"]:.5f} · FM={row["fm"]:.5f} · RGB={row["rgb"]:.5f} · etapa={row["stage"]} · pred={row["pred_fraction"]:.1%}')
    sums={};window=0
   if step%CONFIG['EVAL_CADA']==0 or step==total:
    info=validation(ema,curriculum=phase!='clean');oldstage=controller.stage
    if phase!='clean':controller.update(step,info)
    vals.append(dict(step=step,stage=controller.stage,**info));improved=info['score']<best
    if improved:best=info['score']
    ck=state()
    if improved:atomic_save(ck,folder/'best.pt')
    atomic_save(ck,last)
    pd.DataFrame(history).to_csv(folder/'history.csv',index=False);pd.DataFrame(vals).to_csv(folder/'validation.csv',index=False)
    tqdm.write(f'{arch} {phase} · {step}: MSE={info["mse"]:.5f} · R codec={info["ratio_codec"]:.2f} · etapa={controller.stage}')
    if oldstage!=controller.stage:tqdm.write(f'Curriculum: {oldstage} → {controller.stage}')
   elif step%CONFIG['GUARDAR_CADA']==0:atomic_save(state(),last)
 finally:bar.close()
 ck=torch.load(folder/'best.pt',map_location='cpu',weights_only=False)
 ema.load_state_dict(ck['ema']);ema.cpu();del model,opt,scaler,ck;gc.collect()
 if AMP:torch.cuda.empty_cache()
 return ema,pd.DataFrame(history),pd.DataFrame(vals),dict(arch=arch,phase=phase,best_score=best,params=params)
''',
'preflight': r'''
seed_all(CONFIG['SEMILLA'])
checks={}
for arch in CONFIG['ARQUITECTURAS']:
 model=Flow2DiT(arch,CONFIG,DINO_DIM).to(DEVICE).train()
 sampler=RowSampler();rows,modes,c,h,photo,target=training_batch(sampler,0.,.5)
 a=to_tokens(active(c[:,3:4]));t=torch.full((len(rows),),.6,device=DEVICE);noise=torch.randn_like(target)*a
 xt=(1-t[:,None,None])*noise+t[:,None,None]*target
 with amp_ctx():v=model(xt,t,h,photo,c[:,3:4])
 assert v.shape==target.shape and torch.isfinite(v).all()
 loss=((v.float()-(target-noise)).square()*a).mean();loss.backward()
 assert any(p.grad is not None and p.grad.abs().sum()>0 for p in model.parameters())
 assert all(p.grad is None for p in COLOR.parameters()),'El codec debe permanecer congelado'
 # El decoder congelado conserva gradiente respecto al latente.
 zz=target[:1].detach().requires_grad_(True)
 with amp_ctx():yy=color_decode(zz,c[:1,3:4])
 yy.float().mean().backward();assert zz.grad is not None and torch.isfinite(zz.grad).all()
 # Signo de Flow 2: ruido -> dato, t=0 -> t=1.
 oracle=xt+(1-t[:,None,None])*(target-noise)
 assert torch.allclose(oracle,target,atol=2e-5)
 checks[arch]=sum(p.numel() for p in model.parameters())
 del model,zz,yy,loss,v;gc.collect()
print('Preflight correcto:',checks)
if AMP:torch.cuda.empty_cache()
''',
'runs': r'''
MODELS={};HIST={};VAL_HISTORY={};SUMMARY=[]
for arch in CONFIG['ARQUITECTURAS']:
 model,h,v,info=train_run(arch,'clean');MODELS[arch]=model;HIST[arch]=h;VAL_HISTORY[arch]=v;SUMMARY.append(info)
clean_comparison=pd.DataFrame(SUMMARY).sort_values('best_score')
display(clean_comparison);clean_comparison.to_csv(RUN/'comparacion_arquitecturas_validacion.csv',index=False)
WINNER=clean_comparison.iloc[0].arch
print('Arquitectura elegida con validación limpia:',WINNER)
best_clean=cpu_weights(MODELS[WINNER]);clean_hist=VAL_HISTORY[WINNER].mse.to_list()
ADAPTED,hc,vc,adapt_info=train_run(WINNER,'curriculum',initial=best_clean,clean_hist=clean_hist)
HIST['curriculum']=hc;VAL_HISTORY['curriculum']=vc
# El best de curriculum incluye también el punto de partida sin modificar, evaluado con el mismo criterio.
FINAL=ADAPTED
print('Referencia limpia y adaptación conservadas; el test no elige pesos.')

cfg_scores=[];FINAL.to(DEVICE)
for w in tqdm(CONFIG['CFG_GRID'],desc='Seleccionando CFG · validación',unit='valor'):
 d,_=evaluate2(FINAL,VAL_FIXED,w=w);cfg_scores.append(dict(w=w,mse=macro(d)))
CFG_BEST=float(min(cfg_scores,key=lambda q:q['mse'])['w']);FINAL.cpu()
pd.DataFrame(cfg_scores).to_csv(RUN/'cfg_validacion.csv',index=False)
print('CFG elegido antes del test:',CFG_BEST)
fig,axes=plt.subplots(2,2,figsize=(12,7))
for key,h in HIST.items():
 if len(h):axes[0,0].plot(h.step,h.loss,label=key);axes[0,1].plot(h.step,h.pred_fraction,label=key)
for key,v in VAL_HISTORY.items():
 axes[1,0].plot(v.step,v.mse,label=key);axes[1,1].plot(v.step,v.ratio_codec,label=key)
for ax,title in zip(axes.flat,['Objetivo de entrenamiento','Fracción real de pseudoetiquetas','MSE RGB · validación','Error relativo al codec']):
 ax.set_title(title);ax.set_xlabel('Paso de la fase');ax.grid(alpha=.2);ax.legend()
fig.tight_layout();fig.savefig(RUN/'curvas.png',dpi=140);plt.show();plt.close(fig)
''',
'test': r'''
TEST_SCORES={};TEST_IMAGES={}
for name,model in list(MODELS.items())+[('curriculum',FINAL)]:
 model.to(DEVICE)
 w=CFG_BEST if name=='curriculum' else 1.
 scores,images=evaluate2(model,TEST_FIXED,w=w,keep=True,progress=True)
 TEST_SCORES[name]=scores;TEST_IMAGES[name]=images
 scores.to_csv(RUN/f'test_mascara_real_{name}.csv',index=False);model.cpu()
FINAL.to(DEVICE)
generated,GENERATED_IMAGES=evaluate2(FINAL,TEST_FIXED,mode='generada',w=CFG_BEST,keep=True,progress=True)
generated.to_csv(RUN/'test_mascara_generada_proxies.csv',index=False);FINAL.cpu()
comparison=pd.DataFrame([dict(modelo=name,mse=macro(df),psnr=macro(df,'psnr'),deltaE76=macro(df,'deltaE76')) for name,df in TEST_SCORES.items()])
display(comparison);comparison.to_csv(RUN/'comparacion_test.csv',index=False)
for name,df in TEST_SCORES.items():
 by_codec=df.groupby(['codec_color_vio_objetivo','color_i'])[['mse','deltaE76']].mean().groupby('codec_color_vio_objetivo').mean()
 by_codec.to_csv(RUN/f'test_codec_visto_{name}.csv')
print('Panel:',len(TEST_FIXED),'fotos. Proxies sobre máscara generada no equivalen a color con GT alineado.')

def composite(rgb,m):
 rgb=np.asarray(rgb);m=np.asarray(m)
 return np.clip((rgb*m+np.array([.98,.79,.16])[:,None,None]*(1-m)).transpose(1,2,0),0,1)
figdir=RUN/'test_imagenes';figdir.mkdir(exist_ok=True)
for start in tqdm(range(0,len(TEST_FIXED),CONFIG['POR_PAGINA']),desc='Mostrando ejemplos en pantalla',unit='página'):
 end=min(len(TEST_FIXED),start+CONFIG['POR_PAGINA']);ii=TEST_FIXED[start:end];ci=pairs.iloc[ii].color_i.to_numpy()
 c=rgba_batch(ci);m=c[:,3:4]
 with torch.no_grad(),amp_ctx():oracle=color_decode(to_tokens(Z[torch.as_tensor(ci)].to(DEVICE)),m).cpu()
 c=c.cpu();columns=6+len(MODELS)
 fig,axes=plt.subplots(len(ii),columns,figsize=(2.3*columns,2.5*len(ii)),squeeze=False,facecolor='#f4f6fb')
 titles=['Foto','LEGO objetivo','VAE referencia']+list(MODELS)+['Curriculum · GT','Flow 1 · máscara','Curriculum · generada']
 for row,idx in enumerate(ii):
  j=start+row;mask=c[row,3:4].numpy()
  panels=[FOTOS[idx].permute(1,2,0).numpy(),composite(c[row,:3],mask),composite(oracle[row],mask)]
  panels += [composite(TEST_IMAGES[name][j],mask) for name in MODELS]
  panels += [composite(TEST_IMAGES['curriculum'][j],mask),PRED_MASKS[int(idx)][0].numpy(),
             composite(GENERATED_IMAGES[j],PRED_MASKS[int(idx)].numpy())]
  for col,panel in enumerate(panels):
   axes[row,col].imshow(panel,cmap='gray' if panel.ndim==2 else None,vmin=0,vmax=1 if panel.ndim==2 else None)
   axes[row,col].axis('off')
   if row==0:axes[row,col].set_title(titles[col],fontsize=9)
  axes[row,0].set_ylabel(pairs.iloc[idx].id,fontsize=7)
 fig.suptitle(f'Flow 2 · {start+1}–{end} de {len(TEST_FIXED)} · CFG {CFG_BEST:g}',fontsize=15,fontweight='bold')
 fig.tight_layout(rect=[0,0,1,.97]);fig.savefig(figdir/f'test_{start+1:03d}_{end:03d}.png',dpi=120)
 plt.show();plt.close(fig)
''',
'export': r'''
EXPORT=RUN/'inferencia';EXPORT.mkdir(exist_ok=True)
atomic_save(dict(ema=cpu_weights(FINAL),arch=WINNER,config=CONFIG,dino_dim=DINO_DIM,cfg_w=CFG_BEST,
 data_sha=DATA_SHA,code_sha=CODE_SHA,artifact_role='flow2_color',smoke=CONFIG['SMOKE']),EXPORT/'flow2_color.pt')
(EXPORT/'model.py').write_text('import math\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n'+MODEL_SOURCE)
(EXPORT/'runtime_constants.json').write_text(json.dumps(dict(flow1_model=FLOW1_MODEL_SOURCE,mask_codec=MASK_CODEC_SOURCE,
 flow1_generate=FLOW1_GENERATE_SOURCE,smoke_vae=SMOKE_VAE_SOURCE,crop=CROP_SOURCE,generate2=GENERATE2_SOURCE)))
(EXPORT/'pipeline.py').write_text(INFERENCE_SOURCE)
manifest=dict(ganador_arquitectura=WINNER,cfg_w=CFG_BEST,smoke=CONFIG['SMOKE'],data_sha=DATA_SHA,
 n_train=len(TRAIN),n_val=len(VAL_FIXED),n_test=len(TEST_FIXED),
 pred_train_aceptadas=len(PRED_TRAIN),colores_pred_aceptados=int(accepted_colors),
 curriculum_llego_a_C=bool((VAL_HISTORY['curriculum'].stage=='C').any()),
 pseudobjetivo='extension_local_uniforme_no_GT_independiente',
 metricas_generadas='proxies_interseccion_sin_correspondencia_semantica',
 eleccion='validacion; test no usado para elegir pesos')
(RUN/'resumen.json').write_text(json.dumps(manifest,indent=2))
(RUN/'config.json').write_text(json.dumps(CONFIG,indent=2))
(RUN/'Flow2_Color_LEGO_Arquitectura_y_Curriculum.md').write_text(RESEARCH_TEXT)

# Script y notebook reproducibles, con las mismas fuentes ejecutadas.
bootstrap=bootstrap_source()
(RUN/'entrenar_flow2.py').write_text(bootstrap+'\n\n'+'\n\n'.join(RUNTIME_SOURCES.values()))
notebook=dict(nbformat=4,nbformat_minor=5,metadata=dict(kernelspec=dict(name='python3',display_name='Python 3',language='python')),
 cells=[dict(cell_type='code',id='bootstrap',metadata={},execution_count=None,outputs=[],source=bootstrap)]+
 [dict(cell_type='code',id=name,metadata={},execution_count=None,outputs=[],source=source) for name,source in RUNTIME_SOURCES.items()])
(RUN/'flow2_completo.ipynb').write_text(json.dumps(notebook,ensure_ascii=False,indent=1))
(RUN/'LEEME.txt').write_text('Flow 2 LEGO color. Entrenamiento solo del flow y adaptadores; VAE y Flow 1 congelados.\n'
 'Reanudar: añade este ZIP y fija CONFIG[REANUDAR_DE] a su ruta, conservando VERSION y el plan.\n'
 'Inferencia: importar inferencia/pipeline.py y usar cargar_pipeline(RAIZ_DE_LA_CARPETA).\n'
 'Incluye modelos best/last, optimizador, RNG, cachés, fotos recortadas, RGBA, métricas y fuentes.\n'
 'Las pseudoetiquetas de máscaras generadas son aproximaciones locales conservadoras.\n')

# Recarga independiente: no depende de los objetos del notebook ni de TOK/FOTOS globales.
pipeline_mod=load_module('flow2_exported_pipeline',EXPORT/'pipeline.py')
pipeline=pipeline_mod.cargar_pipeline(RUN,device=str(DEVICE))
idx=int(TEST_FIXED[0]);mh=PRED_MASKS[idx].numpy()[0]
answer=pipeline.inferir_array(FOTOS[idx].permute(1,2,0).numpy(),mascara=mh,seed=CONFIG['SEMILLA']+idx,mostrar=False)
assert answer['rgba'].shape==(128,128,4) and np.array_equal(answer['rgba'][:,:,3],mh.astype(np.uint8)*255)
assert np.isfinite(answer['rgb']).all()
del pipeline
print('Export independiente comprobado.')

# UN SOLO ZIP: absolutamente todos los archivos de RUN. Se guarda fuera de RUN para evitar incluirse.
full=RUN.parent/(RUN.name+'_TODO.zip')
files=sorted(p for p in RUN.rglob('*') if p.is_file())
with zipfile.ZipFile(full,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=4,allowZip64=True) as z:
 for p in tqdm(files,desc='UN ZIP · toda la salida',unit='archivo'):z.write(p,Path(RUN.name)/p.relative_to(RUN))
with zipfile.ZipFile(full) as z:
 for member in tqdm(z.infolist(),desc='Verificando CRC del ZIP completo',unit='archivo'):
  if member.is_dir():continue
  with z.open(member) as stream:
   for _ in iter(lambda:stream.read(2**20),b''):pass
print('Descargar este único ZIP:',full,'| MiB:',round(full.stat().st_size/2**20,1))
display(FileLink(str(full)))
''',
}

EXT_RUNTIME_SOURCES = {
'01_origen.py': r'''
import os,sys,json,math,time,copy,random,hashlib,zipfile,io,importlib.util,gc,shutil
from pathlib import Path
from contextlib import nullcontext
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
import cv2
from tqdm.auto import tqdm
from IPython.display import display,FileLink

# Añade a Kaggle el output COMPLETO del Flow 2 anterior, extraído o en ZIP.
# summarize.zip no contiene pesos y no sirve para continuar el entrenamiento.
EXT=dict(
 PAQUETE_ANTERIOR=None,              # Ej.: '/kaggle/input/flow2works/flow2_v1_TODO.zip'
 RAIZ='/kaggle/input',SALIDA='/kaggle/working/flow2_color',VERSION='flow2_v1_20k',
 PASOS_EXTRA=20000,                  # Son ADICIONALES al entrenamiento anterior.
 LR_MAX=8e-5,WARMUP_EXTRA=500,LR_FINAL_FRACCION=.05,
 CONTINUAR_CURRICULUM=False,         # False: aísla el efecto de entrenar más con máscaras reales.
 EVAL_CADA=500,GUARDAR_CADA=250,LOG_CADA=100,
 RESELECCIONAR_CFG=True,REANUDAR=True,
)
EXT_SMOKE=os.environ.get('FLOW2_EXT_SMOKE','0')=='1'
if EXT_SMOKE:
 EXT.update(PAQUETE_ANTERIOR=os.environ['FLOW2_EXT_PARENT'],
  SALIDA=os.environ.get('FLOW2_EXT_OUT','/tmp/flow2_ext_smoke'),
  PASOS_EXTRA=int(os.environ.get('FLOW2_EXT_STEPS','4')),WARMUP_EXTRA=1,EVAL_CADA=2,GUARDAR_CADA=1)
assert EXT['PASOS_EXTRA']>0 and EXT['LOG_CADA']==100
assert EXT['VERSION'] and '/' not in EXT['VERSION'] and EXT['VERSION'] not in ['.','..']
if not hasattr(cv2,'CascadeClassifier'):
 raise ImportError("Se necesita OpenCV 4. Ejecuta %pip install 'opencv-python-headless>=4.10,<5' --no-deps y reinicia el kernel.")

def file_sha(path):
 h=hashlib.sha256()
 with open(path,'rb') as stream:
  for chunk in iter(lambda:stream.read(2**20),b''):h.update(chunk)
 return h.hexdigest()

def inspect_package(path):
 path=Path(path)
 required=['config.json','resumen.json','cache/photos.pt','cache/color_bank.pt','cache/rgba.pt',
           'cache/predictions.pt','pares_flow2.csv','manifest_color.csv','alias_color.csv',
           'assets/color/color_codec.py','assets/color/vae_control.pt','inferencia/flow2_color.pt']
 if path.is_dir():
  roots=[path] if (path/'resumen.json').is_file() else [p.parent for p in path.rglob('resumen.json')]
  good=[]
  for root in roots:
   if not all((root/name).is_file() for name in required):continue
   info=json.loads((root/'resumen.json').read_text());arch=info.get('ganador_arquitectura')
   if arch not in ['cross','single']:continue
   ck=f'runs/{arch}_curriculum/best.pt'
   if not (root/ck).is_file():continue
   cfg=json.loads((root/'config.json').read_text())
   good.append(dict(path=str(root),prefix='',arch=arch,checkpoint=ck,config=cfg,summary=info,is_zip=False))
  return good
 if not path.is_file() or path.suffix.lower()!='.zip':return []
 with zipfile.ZipFile(path) as z:
  names=set(z.namelist());good=[]
  for member in sorted(names):
   if Path(member).name!='resumen.json':continue
   prefix=member[:-len('resumen.json')]
   if not all(prefix+name in names for name in required):continue
   info=json.loads(z.read(member));arch=info.get('ganador_arquitectura')
   ck=f'runs/{arch}_curriculum/best.pt'
   if arch not in ['cross','single'] or prefix+ck not in names:continue
   cfg=json.loads(z.read(prefix+'config.json'))
   good.append(dict(path=str(path),prefix=prefix,arch=arch,checkpoint=ck,config=cfg,summary=info,is_zip=True))
  return good

if EXT['PAQUETE_ANTERIOR']:
 options=inspect_package(EXT['PAQUETE_ANTERIOR'])
else:
 root=Path(EXT['RAIZ'])
 paths=[p.parent for p in root.rglob('resumen.json')]
 paths+=list(root.rglob('*TODO.zip'))
 options=[]
 for path in tqdm(paths,desc='Localizando el Flow 2 entrenado',unit='paquete'):
  options.extend(inspect_package(path))
 # No elegir silenciosamente pesos de distintas corridas.
 options=list({(v['path'],v['prefix']):v for v in options}.values())
if not options:
 raise FileNotFoundError('Falta el paquete COMPLETO flow2_v1_TODO.zip o su carpeta extraída, con assets, cache y runs/*_curriculum/best.pt. Adjunta el output completo en Kaggle. summarize.zip solo contiene resultados; esta libreta nunca inicia desde cero.')
if len(options)!=1:
 raise ValueError('Hay varios modelos compatibles. Fija EXT[PAQUETE_ANTERIOR] a uno: '+str([v['path'] for v in options]))
SOURCE=options[0];PARENT_CONFIG=SOURCE['config'];WINNER=SOURCE['arch']
if bool(PARENT_CONFIG.get('SMOKE'))!=EXT_SMOKE:
 raise ValueError('No mezclar una prueba SMOKE con pesos de entrenamiento real.')
RUN=Path(EXT['SALIDA'])/EXT['VERSION'];RUN.mkdir(parents=True,exist_ok=True)
marker=RUN/'origen_20k.json'
if not marker.is_file():
 if SOURCE['is_zip']:
  with zipfile.ZipFile(SOURCE['path']) as z:
   members=[i for i in z.infolist() if i.filename.startswith(SOURCE['prefix'])]
   for member in tqdm(members,desc='Restaurando TODO el experimento',unit='archivo'):
    rel=Path(member.filename[len(SOURCE['prefix']):])
    if member.is_dir() or str(rel)=='.':continue
    if rel.is_absolute() or '..' in rel.parts:raise ValueError('Ruta ZIP insegura')
    dest=RUN/rel;dest.parent.mkdir(parents=True,exist_ok=True)
    with z.open(member) as stream,open(dest,'wb') as out:shutil.copyfileobj(stream,out,2**20)
 else:
  src=Path(SOURCE['path']).resolve()
  if src==RUN.resolve():raise ValueError('Usa una VERSION nueva para la extensión.')
  files=[p for p in src.rglob('*') if p.is_file()]
  for p in tqdm(files,desc='Copiando TODO el experimento',unit='archivo'):
   dest=RUN/p.relative_to(src);dest.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(p,dest)
 if not marker.is_file():
  # Conservar los resultados y el export de referencia antes de sobrescribir el export final.
  baseline=RUN/'anterior';baseline.mkdir(exist_ok=True)
  for p in list(RUN.iterdir()):
   if p.is_file():shutil.copy2(p,baseline/p.name)
  for name in ['inferencia','test_imagenes']:
   if (RUN/name).is_dir():shutil.copytree(RUN/name,baseline/name,dirs_exist_ok=True)
  origin=dict(checkpoint=SOURCE['checkpoint'],checkpoint_sha=file_sha(RUN/SOURCE['checkpoint']),
   data_sha=SOURCE['summary']['data_sha'],arch=WINNER,config=PARENT_CONFIG,
   cfg_original=float(SOURCE['summary']['cfg_w']),source_path=SOURCE['path'])
  marker.write_text(json.dumps(origin,indent=2))
ORIGIN=json.loads(marker.read_text())
PARENT_CONFIG=ORIGIN['config']
assert ORIGIN['arch']==WINNER and ORIGIN['data_sha']==SOURCE['summary']['data_sha']
assert file_sha(RUN/ORIGIN['checkpoint'])==ORIGIN['checkpoint_sha']
PARENT_CK=torch.load(RUN/ORIGIN['checkpoint'],map_location='cpu',weights_only=False)
assert PARENT_CK.get('artifact_role')=='flow2_training' and PARENT_CK['arch']==WINNER
assert all(k in PARENT_CK for k in ['model','ema','opt','scaler','rng','sampler','controller'])
assert PARENT_CK['data_sha']==ORIGIN['data_sha']
BASELINE_EXPORT=torch.load(RUN/'anterior'/'inferencia'/'flow2_color.pt',map_location='cpu',weights_only=False)
assert BASELINE_EXPORT['arch']==WINNER and BASELINE_EXPORT['data_sha']==ORIGIN['data_sha']
print('Ganador recuperado:',WINNER,'| checkpoint:',ORIGIN['checkpoint'])
print('Se entrenarán',EXT['PASOS_EXTRA'],'pasos ADICIONALES; no se repite la comparación cross/single.')
print('Curriculum adicional:',EXT['CONTINUAR_CURRICULUM'],'| fuente completa verificada; nunca se inicia desde cero.')
''',
'02_setup.py': r'''
CONFIG=dict(PARENT_CONFIG)
CONFIG.update(VERSION=EXT['VERSION'],SALIDA=EXT['SALIDA'],RAIZ=EXT['RAIZ'],
 REANUDAR_DE=None,REANUDAR=True,LOG_CADA=100,EVAL_CADA=EXT['EVAL_CADA'],
 GUARDAR_CADA=EXT['GUARDAR_CADA'])
DEVICE=torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
DINO_DEVICE=torch.device('cuda:1' if torch.cuda.device_count()>1 else DEVICE)
AMP=DEVICE.type=='cuda'
if DEVICE.type=='cpu':torch.set_num_threads(min(4,os.cpu_count() or 1))
def seed_all(seed):
 random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
 if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
def amp_ctx():return torch.autocast('cuda',dtype=torch.float16) if AMP else nullcontext()
def sha_file(path):
 h=hashlib.sha256()
 with open(path,'rb') as f:
  for b in iter(lambda:f.read(1<<20),b''):h.update(b)
 return h.hexdigest()
def atomic_save(obj,path):
 path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
 temp=path.with_suffix(path.suffix+'.tmp');torch.save(obj,temp);temp.replace(path)
def load_module(name,path):
 spec=importlib.util.spec_from_file_location(name,path);mod=importlib.util.module_from_spec(spec)
 spec.loader.exec_module(mod);return mod
def cpu_weights(model):return {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
seed_all(CONFIG['SEMILLA']);torch.backends.cudnn.benchmark=False
print('Flow 2:',DEVICE,'| DINO:',DINO_DEVICE,'| AMP:',AMP)
print('Lote efectivo:',CONFIG['LOTE']*CONFIG['ACUMULAR'],'| resúmenes cada 100 pasos')
if CONFIG['SMOKE']:print('SMOKE: datos y pesos artificiales; no evalúa calidad.')

''',
'03_assets.py': r'''
def safe_extract(z,names,dest):
 dest=Path(dest);dest.mkdir(parents=True,exist_ok=True)
 for name in names:
  rel=Path(name)
  if rel.is_absolute() or '..' in rel.parts:raise ValueError('Ruta ZIP insegura: '+name)
  if name.endswith('/'):continue
  p=dest/rel;p.parent.mkdir(parents=True,exist_ok=True);p.write_bytes(z.read(name))
 return dest
def choose(paths,label):
 paths=sorted(set(Path(p) for p in paths if Path(p).is_file()),key=str)
 if not paths:raise FileNotFoundError(label)
 if len(paths)>1 and len({sha_file(p) for p in paths})>1:raise ValueError(f'{label}: varias versiones; fija CONFIG: {paths}')
 return paths[0]

def prepare_assets():
 existing_color=RUN/'assets'/'color';existing_flow=RUN/'assets'/'flow1'
 if (existing_color/'color_codec.py').is_file() and (existing_flow/'flow1_dito.pt').is_file() and (RUN/'cache'/'rgba.pt').is_file():
  return existing_color,existing_flow,[]
 root=Path(CONFIG['RAIZ']);files=[p for p in tqdm(root.rglob('*'),desc='Indexando inputs',unit='ruta') if p.is_file()]
 archives=[p for p in files if p.suffix.lower()=='.zip']
 def find_asset(name,explicit=None):
  if explicit:return choose([explicit],name)
  candidates=[p for p in files if p.name==name]
  if candidates:return choose(candidates,name)
  for archive in tqdm(archives,desc='Buscando '+name+' en ZIP',unit='zip',leave=False):
   with zipfile.ZipFile(archive) as z:
    members=[n for n in z.namelist() if Path(n).name==name]
    if not members:continue
    dest=RUN/'assets'/'imports'/sha_file(archive)[:12]
    selected=set(members)
    for member in members:
     parent=Path(member).parent
     if name=='color_codec.py':selected.update(n for n in z.namelist() if str(Path(n).parent).startswith(str(parent)))
     if name=='flow1_dito.pt':
      selected.update(n for n in z.namelist() if Path(n).parent==parent and Path(n).name in ['pares_auditados.csv','stats_flow.pt'])
    safe_extract(z,sorted(selected),dest)
    candidates.extend(dest/n for n in members)
  return choose(candidates,'Falta '+name)
 color_script=find_asset('color_codec.py',Path(CONFIG['COLOR_DIR'])/'color_codec.py' if CONFIG['COLOR_DIR'] else None)
 color_dir=RUN/'assets'/'color'
 shutil.copytree(color_script.parent,color_dir,dirs_exist_ok=True,ignore=shutil.ignore_patterns('__pycache__'))
 ck_flow=find_asset('flow1_dito.pt',CONFIG['CK_FLOW1'])
 flow_dest=RUN/'assets'/'flow1';flow_dest.mkdir(exist_ok=True)
 shutil.copyfile(ck_flow,flow_dest/'flow1_dito.pt')
 audit=find_asset('pares_auditados.csv',CONFIG['PARES_AUDITADOS'])
 shutil.copyfile(audit,flow_dest/'pares_auditados.csv')
 cp=torch.load(flow_dest/'flow1_dito.pt',map_location='cpu',weights_only=False)
 if cp.get('config',{}).get('SMOKE'):raise ValueError('El Flow 1 adjunto es una prueba SMOKE')
 expected=cp.get('tokenizer_sha')
 if not expected:raise ValueError('Flow 1 sin tokenizer_sha; se requiere el export verificado')
 candidates=[Path(CONFIG['CK_DITO'])] if CONFIG['CK_DITO'] else [p for p in files if p.name=='best.pt' and p.parent.name=='dito_base']
 matched=[p for p in candidates if p.is_file() and sha_file(p)==expected]
 if not matched:
  for archive in tqdm(archives,desc='Localizando el DiTo de máscaras exacto',unit='zip',leave=False):
   with zipfile.ZipFile(archive) as z:
    for member in z.namelist():
     p=Path(member)
     if p.name=='best.pt' and p.parent.name=='dito_base':
      b=z.read(member)
      if hashlib.sha256(b).hexdigest()==expected:
       target=flow_dest/'mask_dito.pt';target.write_bytes(b);matched.append(target)
 if not matched:raise FileNotFoundError('Falta el best.pt de dito_base que corresponde al Flow 1. Añade el output de geometría.')
 if matched[0]!=flow_dest/'mask_dito.pt':shutil.copyfile(matched[0],flow_dest/'mask_dito.pt')
 return color_dir,flow_dest,files

if CONFIG['SMOKE']:
 COLOR_DIR=RUN/'assets'/'color';F1_DIR=RUN/'assets'/'flow1';INPUT_FILES=[];FLOW1_CK=None
 color_module=load_module('flow2_smoke_color',COLOR_DIR/'color_codec.py')
 COLOR_CK=torch.load(COLOR_DIR/'vae_control.pt',map_location='cpu',weights_only=False)
 COLOR=color_module.VAEColor();COLOR.load_state_dict(COLOR_CK['ema']);COLOR.eval().requires_grad_(False)
else:
 COLOR_DIR,F1_DIR,INPUT_FILES=prepare_assets()
 selection=json.loads((COLOR_DIR/'seleccion.json').read_text())
 if selection.get('smoke'):raise ValueError('El paquete de color adjunto es SMOKE')
 color_module=load_module('flow2_color_codec',COLOR_DIR/'color_codec.py')
 COLOR,COLOR_CK=color_module.cargar_color(COLOR_DIR/selection['checkpoint'],DEVICE)
 if COLOR_CK['kind']!='vae_control':raise ValueError('Esta versión integra la VAE de color ganadora; se esperaba vae_control')
 FLOW1_CK=torch.load(F1_DIR/'flow1_dito.pt',map_location='cpu',weights_only=False)
 COLOR_STATS=COLOR_CK['stats']
 COLOR.requires_grad_(False).eval()
COLOR=COLOR.to(DEVICE).eval().requires_grad_(False)
COLOR_STATS=COLOR_CK['stats']
C_MEAN=COLOR_STATS['media'].to(DEVICE)[None,:,None,None]
C_STD=COLOR_STATS['desv'].to(DEVICE)[None,:,None,None].clamp_min(1e-6)
def active(m):return (F.adaptive_max_pool2d(m,16)>.5).float()
def to_tokens(z):return z.flatten(2).transpose(1,2)
def from_tokens(z):return z.transpose(1,2).reshape(len(z),8,16,16)
def standardize(z,m):return ((z-C_MEAN)/C_STD)*active(m)
def unstandardize(z,m):return (z*C_STD+C_MEAN)*active(m)
def color_decode(z,m):
 # Mantiene gradiente respecto a z; los parámetros del decoder están congelados.
 return COLOR.decodificar(unstandardize(from_tokens(z),m),m)
print('Codec de color congelado:',COLOR_CK['kind'])
''',
'04_datos.py': r'''
def canonical_rgba(path):
 with Image.open(path) as im:
  if 'A' not in im.getbands() and 'transparency' not in im.info:raise ValueError('Estampa sin alfa: '+str(path))
  a=np.asarray(im.convert('RGBA')).copy()
 h,w=a.shape[:2]
 if h>128 or w>128:raise ValueError(f'Tamaño inesperado {w}x{h}: {path}; no redimensionar')
 out=np.zeros((4,128,128),np.uint8);y=(128-h)//2;x=(128-w)//2;m=a[:,:,3]>=128
 out[:3,y:y+h,x:x+w]=(a[:,:,:3]*m[:,:,None]).transpose(2,0,1);out[3,y:y+h,x:x+w]=m*255
 if not 0<out[3].mean()<255:raise ValueError('Máscara vacía/completa: '+str(path))
 return out

def fixture_data():
 root=RUN/'fixture';root.mkdir(exist_ok=True);aliases=[];rows=[];arrays=[]
 for i in range(18):
  rgba=np.zeros((128,128,4),np.uint8);rgba[25:34,25:36,3]=255;rgba[25:34,85:96,3]=255
  rgba[65:70+i%3,45:80,3]=255;rgba[88+i%6,20+i,3]=255
  rgba[:,:,:3]=(rgba[:,:,3,None]>0)*np.array([25+i*5,15+i*2,30],np.uint8)
  p=root/f's{i:02d}.png';Image.fromarray(rgba).save(p);c=canonical_rgba(p);arrays.append(c)
  split='entrenamiento' if i<12 else ('validacion' if i<15 else 'prueba')
  cid=('real:' if i<9 else 'int:')+f's{i:02d}'
  aliases.append(dict(id=cid,color_i=i,particion=split,sha_color=hashlib.sha256(c.tobytes()).hexdigest(),ruta=str(p),tipo='real' if i<9 else 'intermedia'))
  for k in range(2):
   photo=root/f'photo_{i}_{k}.jpg';bg=np.array([220,200,175],np.uint8)
   a=(c[:3].transpose(1,2,0)*(c[3,:, :,None]>0)+bg*(c[3,:,:,None]==0)).astype(np.uint8)
   Image.fromarray(a).resize((256,256)).save(photo)
   rows.append(dict(id=f's{i:02d}_{k:02d}',pieza=f's{i:02d}',k=k,version='g1' if i<9 else 'gi',
      split_flow=split,archivo=str(photo),color_i=i,color_id=cid,sha_foto_efectivo=sha_file(photo)))
 return pd.DataFrame(rows),pd.DataFrame(aliases),torch.from_numpy(np.stack(arrays))

if (RUN/'cache'/'rgba.pt').is_file() and (RUN/'pares_flow2.csv').is_file() and (RUN/'manifest_color.csv').is_file():
 pairs=pd.read_csv(RUN/'pares_flow2.csv',keep_default_na=False,dtype={'id':str,'pieza':str,'version':str})
 aliases=pd.read_csv(RUN/'alias_color.csv',keep_default_na=False,dtype={'id':str})
 color_table=pd.read_csv(RUN/'manifest_color.csv',keep_default_na=False,dtype={'id':str})
 RGBA=torch.load(RUN/'cache'/'rgba.pt',map_location='cpu',weights_only=False)
elif CONFIG['SMOKE']:
 pairs,aliases,RGBA=fixture_data();color_table=aliases.copy()
else:
 aliases=pd.read_csv(COLOR_DIR/'manifest_color_aliases.csv',keep_default_na=False,dtype={'id':str})
 color_table=pd.read_csv(COLOR_DIR/'manifest_color.csv',keep_default_na=False,dtype={'id':str})
 assert aliases.id.is_unique and color_table.color_i.is_unique
 pairs=pd.read_csv(F1_DIR/'pares_auditados.csv',keep_default_na=False,dtype={'id':str,'pieza':str,'version':str})
 assert {'id','pieza','version','split_flow'}<=set(pairs.columns)
 pairs['color_id']=np.where(pairs.version.eq('gi'),'int:','real:')+pairs.pieza
 mapping=aliases.set_index('id').color_i.to_dict()
 missing=set(pairs.color_id)-set(mapping)
 if missing:raise ValueError(f'Estampas emparejadas ausentes del paquete de color: {sorted(missing)[:15]}')
 pairs['color_i']=pairs.color_id.map(mapping).astype(int)
 image_index={}
 for p in INPUT_FILES:
  if p.suffix.lower() in ['.png','.jpg','.jpeg']:image_index.setdefault(p.name,[]).append(p)
 def resolve_image(raw,expected_sha=None,is_rgba=False,expected_color=None):
  raw=Path(str(raw));cand=[raw] if raw.is_file() else []
  suffix='/'.join(raw.parts[-3:])
  if not cand:cand=[p for p in image_index.get(raw.name,[]) if str(p).endswith(suffix)]
  if not cand:cand=image_index.get(raw.name,[])
  if expected_sha:cand=[p for p in cand if sha_file(p)==expected_sha]
  if is_rgba and expected_color:
   cand=[p for p in cand if hashlib.sha256(canonical_rgba(p).tobytes()).hexdigest()==expected_color]
  return choose(cand,f'Imagen ausente/ambigua: {raw}')
 paths=[];arrays=[]
 for r in tqdm(color_table.itertuples(),total=len(color_table),desc='Verificando RGB LEGO',unit='estampa'):
  p=resolve_image(r.ruta,is_rgba=True,expected_color=r.sha_color);c=canonical_rgba(p)
  assert hashlib.sha256(c.tobytes()).hexdigest()==r.sha_color
  arrays.append(c);paths.append(str(p))
 assert np.array_equal(color_table.color_i.to_numpy(),np.arange(len(color_table)))
 RGBA=torch.from_numpy(np.stack(arrays));del arrays
 color_table['ruta_actual']=paths
 resolved=[]
 for r in tqdm(pairs.to_dict('records'),desc='Verificando pares y fotos',unit='foto'):
  expected=r.get('sha_foto_efectivo') or r.get('sha256_foto') or None
  raw=r.get('archivo') or r.get('ruta')
  if not raw:raise ValueError('Fila sin ruta de foto: '+r['id'])
  p=resolve_image(raw,expected_sha=expected)
  assert p.stem==r['id'],(p,r['id'])
  resolved.append(str(p))
 pairs['archivo']=resolved

color_split=color_table.set_index('color_i').particion.to_dict()
pairs['particion_color']=pairs.color_i.map(color_split)
pairs['split_flow2']=pairs.split_flow
# No usar para train del Flow 2 objetivos que el protocolo del codec reservó.
reserved=pairs.split_flow.eq('entrenamiento') & pairs.particion_color.ne('entrenamiento')
pairs.loc[reserved,'split_flow2']='reserva_codec'
pairs['codec_color_vio_objetivo']=pairs.particion_color.eq('entrenamiento')
TRAIN=np.flatnonzero(pairs.split_flow2.eq('entrenamiento'))
VAL=np.flatnonzero(pairs.split_flow2.eq('validacion'))
TEST=np.flatnonzero(pairs.split_flow2.eq('prueba'))
PERSON=np.flatnonzero(pairs.split_flow2.eq('val_personas'))
assert min(len(TRAIN),len(VAL),len(TEST))>0,'Train/val/test vacíos; revisar manifiestos'
assert not set(pairs.iloc[TRAIN].color_i)&set(pairs.iloc[np.r_[VAL,TEST]].color_i),'Una estampa cruza train y val/test del Flow 2'
def fixed_panel(indices,n):
 rng=np.random.default_rng(CONFIG['SEMILLA']);groups={int(k):list(g.index) for k,g in pairs.iloc[indices].groupby('color_i')}
 for g in groups.values():rng.shuffle(g)
 keys=list(groups);rng.shuffle(keys);result=[]
 while len(result)<min(n,len(indices)) and any(groups.values()):
  for k in keys:
   if groups[k] and len(result)<n:result.append(groups[k].pop())
 return np.array(result,dtype=int)
VAL_FIXED=fixed_panel(VAL,CONFIG['N_VAL']);TEST_FIXED=fixed_panel(TEST,CONFIG['N_TEST'])
TRAIN_COLORS=np.array(sorted(pairs.iloc[TRAIN].color_i.unique()))
ROWS_BY_COLOR={int(k):g.index.to_numpy() for k,g in pairs.iloc[TRAIN].groupby('color_i')}
display(pairs.groupby(['version','split_flow2']).size().rename('fotos').to_frame())
print('Objetivos train:',len(TRAIN_COLORS),'| val:',len(VAL_FIXED),'| test:',len(TEST_FIXED))
pairs.to_csv(RUN/'pares_flow2.csv',index=False);aliases.to_csv(RUN/'alias_color.csv',index=False)
color_table.to_csv(RUN/'manifest_color.csv',index=False)
def rgba_batch(ci):return RGBA[torch.as_tensor(ci,dtype=torch.long)].to(DEVICE).float()/255.
''',
'05_cache.py': r'''
detector=cv2.CascadeClassifier(cv2.data.haarcascades+'haarcascade_frontalface_default.xml')
assert not detector.empty()
def crop_photo(row):
 b=Path(row.archivo).read_bytes()
 if CONFIG['CHECK_SHA_FOTOS'] and isinstance(row.get('sha256_foto'),str) and row.sha256_foto:
  assert hashlib.sha256(b).hexdigest()==row.sha256_foto,('SHA foto distinto',row.archivo)
 with Image.open(io.BytesIO(b)) as im:im=im.convert('RGB');im.load();a=np.asarray(im)
 h,w=a.shape[:2];found=detector.detectMultiScale(cv2.cvtColor(a,cv2.COLOR_RGB2GRAY),1.1,5,minSize=(max(16,w//6),)*2)
 if len(found):
  x,y,cw,ch=max(found,key=lambda c:c[2]*c[3]);cx,cy,s=x+cw/2,y+ch/2,max(cw,ch)*1.8
 else:cx,cy,s=w/2,h*.45,min(w,h)*.75
 x0,y0=max(0,int(cx-s/2)),max(0,int(cy-s/2));x1,y1=min(w,int(cx+s/2)),min(h,int(cy+s/2))
 return np.asarray(Image.fromarray(a[y0:y1,x0:x1]).resize((256,256),Image.Resampling.BICUBIC)),bool(len(found)),hashlib.sha256(b).hexdigest()

PHOTO_SIGNATURE=hashlib.sha256((pairs[['id','version','color_i','archivo']].to_csv(index=False)+str(CONFIG['DINO'])).encode()).hexdigest()
cache=RUN/'cache';cache.mkdir(exist_ok=True)
if CONFIG['SMOKE']:
 class ToyDino(nn.Module):
  def __init__(self):super().__init__();self.conv=nn.Conv2d(3,32,14,14)
  def forward(self,pixel_values):
   h=self.conv(pixel_values).flatten(2).transpose(1,2)
   return type('Output',(),{'last_hidden_state':torch.cat([h.mean(1,keepdim=True),h],1)})()
 DINO=ToyDino();DINO_DIM=32
 if (RUN/'assets'/'dino'/'toy_dino.pt').is_file():DINO.load_state_dict(torch.load(RUN/'assets'/'dino'/'toy_dino.pt',map_location='cpu',weights_only=False))
 IMNET_M=torch.tensor([.485,.456,.406])[None,:,None,None];IMNET_S=torch.tensor([.229,.224,.225])[None,:,None,None]
 (RUN/'assets'/'dino').mkdir(exist_ok=True)
 atomic_save(DINO.state_dict(),RUN/'assets'/'dino'/'toy_dino.pt')
else:
 from transformers import AutoModel,AutoImageProcessor
 local=RUN/'assets'/'dino'
 dino_source=CONFIG['DINO_LOCAL'] or (str(local) if (local/'config.json').is_file() else FLOW1_CK.get('dino',CONFIG['DINO']))
 processor=AutoImageProcessor.from_pretrained(dino_source)
 DINO=AutoModel.from_pretrained(dino_source)
 DINO_DIM=DINO.config.hidden_size
 IMNET_M=torch.tensor(processor.image_mean)[None,:,None,None];IMNET_S=torch.tensor(processor.image_std)[None,:,None,None]
 DINO.save_pretrained(RUN/'assets'/'dino');processor.save_pretrained(RUN/'assets'/'dino')
DINO=DINO.to(DINO_DEVICE).eval().requires_grad_(False)
IMNET_M,IMNET_S=IMNET_M.to(DINO_DEVICE),IMNET_S.to(DINO_DEVICE)

# Restaurar Flow 1 exactamente desde su export y el checkpoint DiTo verificado.
if not CONFIG['SMOKE']:
 f1_up=F1_DIR/'upstream';f1_up.mkdir(exist_ok=True)
 for name,content in FLOW1_CK['upstream'].items():(f1_up/name).write_text(content)
 (f1_up/'upstream_registry.py').write_text('def register(name):\n    return lambda cls: cls\n')
 sys.path.insert(0,str(f1_up))
 f1cfg=FLOW1_CK['config'];assert f1cfg['DINO']==FLOW1_CK['dino']
 if str(CONFIG['DINO'])!=str(FLOW1_CK['dino']):raise ValueError('DINO distinto al del Flow 1')
 f1ns=dict(torch=torch,nn=nn,F=F,math=math,CONFIG=f1cfg,DEVICE=DEVICE,amp_ctx=amp_ctx,DIM_DINO=DINO_DIM)
 f1ns['codec']=load_module('f2_mask_codec',f1_up/'codec.py')
 f1ns['renderer']=load_module('f2_mask_renderer',f1_up/'renderer.py')
 f1ns['fm']=load_module('f2_mask_fm',f1_up/'fm.py').FM(sigma_min=1e-5,timescale=1000.)
 f1ns['sampler']=load_module('f2_mask_sampler',f1_up/'samplers.py').FMEulerSampler(f1ns['fm'])
 exec(FLOW1_MODEL_SOURCE,f1ns);exec(MASK_CODEC_SOURCE,f1ns);exec(FLOW1_GENERATE_SOURCE,f1ns)
 F1=f1ns['DiT1'](dim_dino=DINO_DIM).to(DEVICE).eval().requires_grad_(False)
 F1.load_state_dict(FLOW1_CK['ema'],strict=True)
 MASK_CODEC=f1ns['FrozenDiTo']().to(DEVICE).eval().requires_grad_(False)
 ck=torch.load(F1_DIR/'mask_dito.pt',map_location='cpu',weights_only=False)
 assert ck.get('kind')=='dito_base';MASK_CODEC.load_state_dict(ck['ema'],strict=True)
 F1_MEAN=FLOW1_CK['stats']['media'].to(DEVICE);F1_STD=FLOW1_CK['stats']['desv'].to(DEVICE)
 del ck

@torch.no_grad()
def dino_features(rgb):
 x=F.interpolate(rgb.to(DINO_DEVICE),size=(224,224),mode='bilinear',align_corners=False,antialias=True)
 with amp_ctx():h=DINO(pixel_values=(x-IMNET_M)/IMNET_S).last_hidden_state
 return h.float().to(DEVICE)
def pool_dino(h):
 b,n,d=h.shape;s=int(math.sqrt(n-1));assert s*s==n-1
 p=F.adaptive_avg_pool2d(h[:,1:].transpose(1,2).reshape(b,d,s,s),CONFIG['DINO_POOL'])
 return torch.cat([h[:,:1],p.flatten(2).transpose(1,2)],1)
@torch.no_grad()
def predict_mask(h,seeds,rows=None):
 if CONFIG['SMOKE']:
  if rows is None:raise ValueError('Inferencia libre no disponible con modelos SMOKE')
  out=RGBA[torch.as_tensor(pairs.iloc[rows].color_i.to_numpy()),3:4].float().to(DEVICE)/255
  # Cambio local controlado para ejercitar pseudoetiquetas; no es una predicción real.
  out=out.clone();out[:,:,65,44]=1
  return out
 z=f1ns['generate'](F1,h,seeds,w=float(FLOW1_CK.get('cfg_w',1.)),steps=FLOW1_CK['config']['PASOS_FLOW'])
 raw=from_tokens(z)*F1_STD+F1_MEAN
 with amp_ctx():m=MASK_CODEC.decode(raw,seeds,steps=FLOW1_CK['config'].get('PASOS_DITO_FINAL',32))
 return (m>=.5).float()

if (cache/'photos.pt').is_file():
 cp=torch.load(cache/'photos.pt',map_location='cpu',weights_only=False)
 assert cp['signature']==PHOTO_SIGNATURE,'Cambiaron los pares/fotos; usar otra VERSION'
 FOTOS=cp['photos'];TOK=cp['tokens'];TOK_M=cp['tokens_mirror'];pairs['sha_foto_verificado']=cp['hashes'];del cp
else:
 photos=[];hashes=[]
 for _,r in tqdm(pairs.iterrows(),total=len(pairs),desc='Recortando y verificando fotos',unit='foto'):
  a,_,h=crop_photo(r);photos.append(a);hashes.append(h)
 FOTOS=torch.from_numpy(np.stack(photos)).permute(0,3,1,2).contiguous();del photos
 pairs['sha_foto_verificado']=hashes
 TOK=[];TOK_M=[]
 for j in tqdm(range(0,len(pairs),CONFIG['LOTE_PREPARACION']),desc='Cache DINO y espejos',unit='lote'):
  x=FOTOS[j:j+CONFIG['LOTE_PREPARACION']].to(DEVICE).float()/255
  TOK.append(pool_dino(dino_features(x)).half().cpu());TOK_M.append(pool_dino(dino_features(x.flip(-1))).half().cpu())
 TOK=torch.cat(TOK);TOK_M=torch.cat(TOK_M)
 atomic_save(dict(signature=PHOTO_SIGNATURE,photos=FOTOS,tokens=TOK,tokens_mirror=TOK_M,hashes=hashes),cache/'photos.pt')
assert not pairs.groupby('sha_foto_verificado').split_flow2.nunique().gt(1).any(),'Foto idéntica cruza splits'

@torch.no_grad()
def encode_color_arrays(c):
 with amp_ctx():z=COLOR.encode(c[:,:3],c[:,3:4]).float()
 return standardize(z,c[:,3:4])
if (cache/'color_bank.pt').is_file():
 bank=torch.load(cache/'color_bank.pt',map_location='cpu',weights_only=False)
 assert bank['rgba_sha']==hashlib.sha256(RGBA.numpy().tobytes()).hexdigest()
 Z=bank['z'];ZM=bank['z_mirror']
else:
 zs=[];zms=[]
 for j in tqdm(range(0,len(RGBA),CONFIG['LOTE_PREPARACION']),desc='Códigos de color y espejos',unit='lote'):
  c=rgba_batch(np.arange(j,min(j+CONFIG['LOTE_PREPARACION'],len(RGBA))))
  zs.append(encode_color_arrays(c).cpu());zms.append(encode_color_arrays(c.flip(-1)).cpu())
 Z=torch.cat(zs);ZM=torch.cat(zms)
 atomic_save(dict(z=Z,z_mirror=ZM,rgba_sha=hashlib.sha256(RGBA.numpy().tobytes()).hexdigest()),cache/'color_bank.pt')
assert Z.shape[1:]==(8,16,16) and torch.isfinite(Z).all()
atomic_save(RGBA,cache/'rgba.pt');pairs.to_csv(RUN/'pares_flow2.csv',index=False)
print('Fotos:',tuple(FOTOS.shape),'| tokens:',tuple(TOK.shape),'| códigos:',tuple(Z.shape))
''',
'06_predicciones.py': r'''
def pseudo_color(c,mhat):
 # Solo extensiones locales de color uniforme. No se afirma correspondencia semántica global.
 rgb=c[:3].transpose(1,2,0).astype(np.float32)/255;gt=c[3]>127;mh=mhat.astype(bool)
 dice=2*(gt&mh).sum()/max(1,gt.sum()+mh.sum())
 if dice<CONFIG['PRED_DICE_MIN']:return None,dict(aceptado=False,motivo='dice',dice=float(dice))
 if not .85<=mh.sum()/max(1,gt.sum())<=1.15:return None,dict(aceptado=False,motivo='ocupacion',dice=float(dice))
 ng,labels_gt=cv2.connectedComponents(gt.astype(np.uint8),connectivity=8)
 npred,_=cv2.connectedComponents(mh.astype(np.uint8),connectivity=8)
 if ng!=npred:return None,dict(aceptado=False,motivo='componentes',dice=float(dice))
 for lab in range(1,ng):
  region=labels_gt==lab
  if (region&mh).sum()<.5*region.sum():return None,dict(aceptado=False,motivo='componente_perdido',dice=float(dice))
 dist,near=cv2.distanceTransformWithLabels((~gt).astype(np.uint8),cv2.DIST_L2,5,labelType=cv2.DIST_LABEL_PIXEL)
 added=mh&~gt
 if added.any() and dist[added].max()>CONFIG['PRED_DIST_MAX']:
  return None,dict(aceptado=False,motivo='extension_lejana',dice=float(dice))
 lookup=np.zeros((int(near.max())+1,3),np.float32);lookup[near[gt]]=rgb[gt]
 out=rgb.copy()
 for y,x in np.argwhere(added):
  ys=slice(max(0,y-2),min(128,y+3));xs=slice(max(0,x-2),min(128,x+3))
  colors=rgb[ys,xs][gt[ys,xs]]
  if not len(colors) or np.max(colors.max(0)-colors.min(0))>CONFIG['PRED_COLOR_SPREAD']:
   return None,dict(aceptado=False,motivo='color_ambiguo',dice=float(dice))
  out[y,x]=lookup[near[y,x]]
 result=np.zeros((4,128,128),np.uint8)
 result[:3]=(np.clip(out,0,1)*255).round().astype(np.uint8).transpose(2,0,1)*mh
 result[3]=mh.astype(np.uint8)*255
 return result,dict(aceptado=True,motivo='pseudo_extension_local',dice=float(dice),nuevos=int(added.sum()))

train_pred_rows=fixed_panel(TRAIN,CONFIG['N_PRED_TRAIN']) if CONFIG['PRED_ENABLE'] else np.array([],dtype=int)
PRED_ROWS=np.unique(np.r_[train_pred_rows,VAL_FIXED,TEST_FIXED]).astype(int)
pred_signature=hashlib.sha256((PHOTO_SIGNATURE+'|'+str(sha_file(F1_DIR/'flow1_dito.pt') if not CONFIG['SMOKE'] else 'smoke')+
   '|'+','.join(map(str,PRED_ROWS))+'|'+json.dumps({k:v for k,v in CONFIG.items() if k.startswith('PRED_')},sort_keys=True)).encode()).hexdigest()
if (cache/'predictions.pt').is_file():
 pred_cache=torch.load(cache/'predictions.pt',map_location='cpu',weights_only=False)
 assert pred_cache['signature']==pred_signature,'Cache de predicciones incompatible; usar otra VERSION'
 PRED_MASKS=pred_cache['masks'];PSEUDO=pred_cache['pseudo'];PRED_AUDIT=pd.DataFrame(pred_cache['audit'])
else:
 PRED_MASKS={};PSEUDO={};audit=[]
 for j in tqdm(range(0,len(PRED_ROWS),CONFIG['LOTE_PREPARACION']),desc='Flow 1 congelado · cache de máscaras',unit='lote'):
  rows=PRED_ROWS[j:j+CONFIG['LOTE_PREPARACION']]
  x=FOTOS[torch.as_tensor(rows)].to(DEVICE).float()/255
  h=dino_features(x);seeds=[CONFIG['SEMILLA']+int(i) for i in rows]
  m=predict_mask(h,seeds,rows).byte().cpu().numpy()
  for k,i in enumerate(rows):
   mh=m[k,0];PRED_MASKS[int(i)]=torch.from_numpy(mh[None].copy())
   c=RGBA[int(pairs.iloc[i].color_i)].numpy();pseudo,info=pseudo_color(c,mh)
   info.update(row=int(i),color_i=int(pairs.iloc[i].color_i),split=str(pairs.iloc[i].split_flow2))
   audit.append(info)
   if pseudo is not None:PSEUDO[int(i)]=torch.from_numpy(pseudo)
 PRED_AUDIT=pd.DataFrame(audit)
 atomic_save(dict(signature=pred_signature,masks=PRED_MASKS,pseudo=PSEUDO,audit=audit),cache/'predictions.pt')
PRED_AUDIT.to_csv(RUN/'auditoria_pseudoetiquetas.csv',index=False)
PRED_TRAIN=np.array([i for i in train_pred_rows if int(i) in PSEUDO],dtype=int)
PRED_VAL=np.array([i for i in VAL_FIXED if int(i) in PSEUDO],dtype=int)
accepted_colors=pairs.iloc[PRED_TRAIN].color_i.nunique() if len(PRED_TRAIN) else 0
CAN_PRED=accepted_colors>=CONFIG['MIN_PRED_COLORES']
display(PRED_AUDIT.groupby(['split','aceptado']).size().rename('mascaras').to_frame())
print('Colores train con pseudoetiqueta aceptada:',accepted_colors,'| adaptación con predicciones:',CAN_PRED)
if not CAN_PRED:print('Se conservarán las deformaciones alineadas; no se fuerzan etiquetas para máscaras diferentes.')
PZ={};PZM={}
for j in tqdm(range(0,len(PSEUDO),CONFIG['LOTE_PREPARACION']),desc='Codificando pseudoetiquetas aceptadas',unit='lote'):
 keys=list(PSEUDO)[j:j+CONFIG['LOTE_PREPARACION']]
 c=torch.stack([PSEUDO[i] for i in keys]).to(DEVICE).float()/255
 z=encode_color_arrays(c).cpu();zm=encode_color_arrays(c.flip(-1)).cpu()
 for k,i in enumerate(keys):PZ[i]=z[k];PZM[i]=zm[k]
atomic_save(dict(z=PZ,z_mirror=PZM),cache/'pseudo_bank.pt')
# La preparación no deja modelos generativos extra en la memoria de entrenamiento.
if not CONFIG['SMOKE']:
 F1.cpu();MASK_CODEC.cpu()
DINO.cpu()
if AMP:torch.cuda.empty_cache()
''',
'07_modelo.py': r'''def rope_2d(dim_cabeza, lado=16):
    d = dim_cabeza // 4
    frec = 1.0 / (10000 ** (torch.arange(d) / d))
    y, x = torch.meshgrid(torch.arange(lado), torch.arange(lado), indexing="ij")
    ang = torch.cat([y.flatten()[:, None] * frec, x.flatten()[:, None] * frec], 1)   # 256 × dim/2
    return torch.cos(ang), torch.sin(ang)

def aplicar_rope(x, cos, sin):                              # x: B×H×N×D
    x1, x2 = x[..., 0::2], x[..., 1::2]
    return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], -1).flatten(-2)

class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.peso, self.eps = nn.Parameter(torch.ones(d)), eps

    def forward(self, x):
        return x * torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps).to(x.dtype) * self.peso.to(x.dtype)

def modular(x, shift, scale):
    return x * (1 + scale[:, None]) + shift[:, None]

def embedding_tiempo(t, dim):
    mitad = dim // 2
    frec = torch.exp(-math.log(10000) * torch.arange(mitad, device=t.device) / mitad)
    a = t[:, None] * 1000 * frec[None]
    return torch.cat([torch.cos(a), torch.sin(a)], 1)

class Attention2(nn.Module):
 def __init__(self,dim,heads,dropout):
  super().__init__();self.heads=heads
  self.q=nn.Linear(dim,dim);self.kv=nn.Linear(dim,2*dim)
  self.nq=RMSNorm(dim//heads);self.nk=RMSNorm(dim//heads)
  self.out=nn.Sequential(nn.Linear(dim,dim),nn.Dropout(dropout))
 def forward(self,x,context=None,rope=None):
  b,n,d=x.shape;c=x if context is None else context
  q=self.q(x).reshape(b,n,self.heads,-1).transpose(1,2)
  k,v=self.kv(c).reshape(b,len(c[0]),2,self.heads,-1).permute(2,0,3,1,4)
  q,k=self.nq(q),self.nk(k)
  if rope is not None:q,k=aplicar_rope(q,*rope),aplicar_rope(k,*rope)
  y=F.scaled_dot_product_attention(q,k,v)
  return self.out(y.transpose(1,2).reshape(b,n,d))
class CrossBlock(nn.Module):
 def __init__(self,d,heads,ratio,drop):
  super().__init__();self.norms=nn.ModuleList([nn.LayerNorm(d,elementwise_affine=False) for _ in range(3)])
  self.selfatt=Attention2(d,heads,drop);self.cross=Attention2(d,heads,drop)
  self.mlp=nn.Sequential(nn.Linear(d,int(d*ratio)),nn.GELU(approximate='tanh'),nn.Dropout(drop),nn.Linear(int(d*ratio),d))
  self.ada=nn.Sequential(nn.SiLU(),nn.Linear(d,9*d));nn.init.zeros_(self.ada[-1].weight);nn.init.zeros_(self.ada[-1].bias)
 def forward(self,x,time,photo,rope):
  s1,e1,g1,s2,e2,g2,s3,e3,g3=self.ada(time).chunk(9,1)
  x=x+g1[:,None]*self.selfatt(modular(self.norms[0](x),s1,e1),rope=rope)
  x=x+g2[:,None]*self.cross(modular(self.norms[1](x),s2,e2),context=photo)
  return x+g3[:,None]*self.mlp(modular(self.norms[2](x),s3,e3))
class SingleBlock(nn.Module):
 def __init__(self,d,heads,ratio,drop):
  super().__init__();self.n1=nn.LayerNorm(d,elementwise_affine=False);self.n2=nn.LayerNorm(d,elementwise_affine=False)
  self.att=Attention2(d,heads,drop)
  self.mlp=nn.Sequential(nn.Linear(d,int(d*ratio)),nn.GELU(approximate='tanh'),nn.Dropout(drop),nn.Linear(int(d*ratio),d))
  self.ada=nn.Sequential(nn.SiLU(),nn.Linear(d,6*d));nn.init.zeros_(self.ada[-1].weight);nn.init.zeros_(self.ada[-1].bias)
 def forward(self,x,time,rope):
  s1,e1,g1,s2,e2,g2=self.ada(time).chunk(6,1)
  x=x+g1[:,None]*self.att(modular(self.n1(x),s1,e1),rope=rope)
  return x+g2[:,None]*self.mlp(modular(self.n2(x),s2,e2))
class PhotoColor(nn.Module):
 def __init__(self,grid):
  super().__init__();self.grid=grid
  self.net=nn.Sequential(nn.Conv2d(3,32,5,2,2),nn.SiLU(),nn.Conv2d(32,64,3,2,1),nn.SiLU(),nn.Conv2d(64,96,3,2,1),nn.SiLU())
 def forward(self,rgb):
  x=F.interpolate(rgb,size=(128,128),mode='bilinear',align_corners=False)
  f=F.adaptive_avg_pool2d(self.net(x),self.grid)
  mean=F.adaptive_avg_pool2d(x,self.grid)
  std=(F.adaptive_avg_pool2d(x*x,self.grid)-mean.square()).clamp_min(1e-6).sqrt()
  h=torch.cat([f,mean,std],1).flatten(2).transpose(1,2)
  return torch.cat([h.mean(1,keepdim=True),h],1)
class Flow2DiT(nn.Module):
 def __init__(self,arch,cfg,dino_dim):
  super().__init__();self.arch=arch;self.cfg=dict(cfg);d=cfg['DIM'];self.dim=d
  self.mask_encoder=nn.Sequential(nn.Conv2d(1,32,3,2,1),nn.SiLU(),nn.Conv2d(32,32,3,2,1),nn.SiLU(),nn.Conv2d(32,32,3,2,1),nn.SiLU())
  self.color=PhotoColor(cfg['DINO_POOL']);self.dino_proj=nn.Sequential(nn.LayerNorm(dino_dim),nn.Linear(dino_dim,d))
  self.rgb_proj=nn.Linear(102,d);self.null=nn.Parameter(torch.zeros(1,1,d))
  self.time=nn.Sequential(nn.Linear(d,d),nn.SiLU(),nn.Linear(d,d))
  self.pos=nn.Parameter(torch.randn(1,256,d)*.02);self.types=nn.Parameter(torch.randn(3,d)*.02)
  self.input=nn.Linear(41 if arch=='cross' else 8,d)
  self.mask_proj=nn.Linear(33,d)
  if arch=='cross':self.blocks=nn.ModuleList([CrossBlock(d,cfg['CABEZAS'],cfg['MLP'],cfg['DROPOUT']) for _ in range(cfg['PROF_CROSS'])])
  elif arch=='single':self.blocks=nn.ModuleList([SingleBlock(d,cfg['CABEZAS'],cfg['MLP'],cfg['DROPOUT']) for _ in range(cfg['PROF_SINGLE'])])
  else:raise ValueError(arch)
  self.final_norm=nn.LayerNorm(d,elementwise_affine=False);self.final_ada=nn.Sequential(nn.SiLU(),nn.Linear(d,2*d));self.out=nn.Linear(d,8)
  for layer in [self.final_ada[-1],self.out]:nn.init.zeros_(layer.weight);nn.init.zeros_(layer.bias)
  cos,sin=rope_2d(d//cfg['CABEZAS']);self.register_buffer('cos',cos,persistent=False);self.register_buffer('sin',sin,persistent=False)
 def forward(self,xt,t,dino,rgb,mask,drop=None):
  occ=(F.adaptive_max_pool2d(mask,16)>.5).float().flatten(2).transpose(1,2)
  shape=self.mask_encoder(mask).flatten(2).transpose(1,2)
  shape=torch.cat([shape,occ],-1)
  photo=self.dino_proj(dino)+self.rgb_proj(self.color(rgb))
  if drop is not None:photo=torch.where(drop[:,None,None],self.null.expand_as(photo),photo)
  time=self.time(embedding_tiempo(t,self.dim))
  if self.arch=='cross':
   x=self.input(torch.cat([xt,shape],-1))+self.pos
   for block in self.blocks:x=block(x,time,photo,(self.cos,self.sin))
  else:
   noisy=self.input(xt)+self.pos+self.types[0]
   shapes=self.mask_proj(shape)+self.pos+self.types[1]
   x=torch.cat([noisy,shapes,photo+self.types[2]],1)
   n=photo.shape[1]
   cos=torch.cat([self.cos,self.cos,torch.ones(n,self.cos.shape[1],device=xt.device)],0)
   sin=torch.cat([self.sin,self.sin,torch.zeros(n,self.sin.shape[1],device=xt.device)],0)
   for block in self.blocks:x=block(x,time,(cos,sin))
   x=x[:,:256]
  shift,scale=self.final_ada(time).chunk(2,1)
  return self.out(modular(self.final_norm(x),shift,scale))*occ
''',
'08_evaluacion.py': r'''def masked_mean(x,m):return (x*m).sum((1,2,3))/(x.shape[1]*m.sum((1,2,3)).clamp_min(1))
def mse_mask(x,y,m):return masked_mean((x-y).square(),m)
def rgb_to_lab(rgb):
    rgb=rgb.clamp(0,1)
    lin=torch.where(rgb<=0.04045,rgb/12.92,((rgb+0.055)/1.055).pow(2.4))
    mat=rgb.new_tensor([[.4124564,.3575761,.1804375],[.2126729,.7151522,.0721750],[.0193339,.1191920,.9503041]])
    xyz=torch.einsum('ij,bjhw->bihw',mat,lin)/rgb.new_tensor([.95047,1.,1.08883])[None,:,None,None]
    delta=6/29
    f=torch.where(xyz>delta**3,xyz.clamp_min(1e-12).pow(1/3),xyz/(3*delta**2)+4/29)
    return torch.stack([116*f[:,1]-16,500*(f[:,0]-f[:,1]),200*(f[:,1]-f[:,2])],1)
def ssim_common_background(x,y,m):
    # La ventana de SSIM ve EL MISMO fondo en objetivo y reconstrucción.
    x=x*m+0.5*(1-m);y=y*m+0.5*(1-m)
    g=torch.exp(-(torch.arange(11,device=x.device,dtype=x.dtype)-5).square()/(2*1.5**2));g=g/g.sum()
    w=(g[:,None]*g[None,:]).expand(3,1,11,11).contiguous()
    conv=lambda a:F.conv2d(a,w,padding=5,groups=3)
    mx,my=conv(x),conv(y);sx=(conv(x*x)-mx*mx).clamp_min(0);sy=(conv(y*y)-my*my).clamp_min(0)
    sxy=conv(x*y)-mx*my
    ss=((2*mx*my+.01**2)*(2*sxy+.03**2))/((mx*mx+my*my+.01**2)*(sx+sy+.03**2))
    return masked_mean(ss,m)
def metricas(x,y,m):
    x=x.float();y=y.float();m=m.float();mse=mse_mask(x,y,m)
    de=(rgb_to_lab(x)-rgb_to_lab(y)).square().sum(1,keepdim=True).sqrt()
    return dict(mse=mse,mae=masked_mean((x-y).abs(),m),
                psnr=-10*torch.log10(mse.clamp_min(1e-10)),
                ssim=ssim_common_background(x,y,m),deltaE76=masked_mean(de,m))



def conditions(rows,mirror=None,photo_override=None):
 source=rows if photo_override is None else photo_override
 source=np.asarray(source,dtype=int)
 if mirror is None:mirror=torch.zeros(len(rows),dtype=torch.bool,device=DEVICE)
 a=TOK[torch.as_tensor(source)].to(DEVICE).float();b=TOK_M[torch.as_tensor(source)].to(DEVICE).float()
 h=torch.where(mirror[:,None,None],b,a)
 rgb=FOTOS[torch.as_tensor(source)].to(DEVICE).float()/255
 rgb=torch.where(mirror[:,None,None,None],rgb.flip(-1),rgb)
 return h,rgb
@torch.no_grad()
def generate2(model,h,rgb,m,seeds,w=1.,steps=None,method='euler'):
 model.eval();steps=steps or CONFIG['PASOS_FLOW'];a=to_tokens(active(m))
 x=torch.stack([torch.randn((256,8),generator=torch.Generator().manual_seed(int(s))) for s in seeds]).to(DEVICE)*a
 def vel(x,t):
  tt=torch.full((len(x),),float(t),device=DEVICE)
  with amp_ctx():
   v=model(x,tt,h,rgb,m,torch.zeros(len(x),dtype=torch.bool,device=DEVICE)).float()
   if w!=1:
    u=model(x,tt,h,rgb,m,torch.ones(len(x),dtype=torch.bool,device=DEVICE)).float();v=u+w*(v-u)
  return v
 for j in range(steps):
  v=vel(x,j/steps)
  x=x+(v+vel(x+v/steps,(j+1)/steps))/(2*steps) if method=='heun' else x+v/steps
  x=x*a
 if not torch.isfinite(x).all():raise FloatingPointError('Latente generado no finito')
 return x

def macro(df,metric='mse'):
 return float(df.groupby('color_i')[metric].mean().mean())
@torch.no_grad()
def evaluate2(model,indices,mode='real',w=1.,shuffle=False,keep=False,progress=False):
 model.eval();rows=[];images=[]
 indices=np.asarray(indices,dtype=int)
 if mode=='pseudo':indices=np.array([i for i in indices if int(i) in PSEUDO],dtype=int)
 shuffled=np.roll(indices,1) if shuffle else indices
 for j in tqdm(range(0,len(indices),CONFIG['LOTE_EVAL']),desc=f'Evaluando {mode}',unit='lote',leave=False,disable=not progress):
  ii=indices[j:j+CONFIG['LOTE_EVAL']];ci=pairs.iloc[ii].color_i.to_numpy()
  c=rgba_batch(ci)
  if mode=='pseudo':c=torch.stack([PSEUDO[int(i)] for i in ii]).to(DEVICE).float()/255
  if mode=='generada':m=torch.stack([PRED_MASKS[int(i)] for i in ii]).to(DEVICE).float()
  else:m=c[:,3:4]
  h,photo=conditions(ii,photo_override=shuffled[j:j+len(ii)])
  seeds=[CONFIG['SEMILLA']+int(i) for i in ii]
  z=generate2(model,h,photo,m,seeds,w)
  with amp_ctx():y=color_decode(z,m).float()
  if mode!='generada':scores=metricas(c[:,:3],y,m)
  else:
   # Intersección espacial: NO representa una correspondencia semántica certificada.
   intersection=m*c[:,3:4]
   proxy=mse_mask(c[:,:3],y,intersection)
   proxy=torch.where(intersection.sum((1,2,3))>0,proxy,torch.full_like(proxy,float('nan')))
   scores={'mse_interseccion_proxy':proxy,
           'pixeles_interseccion':intersection.sum((1,2,3)),
           'mascara_vacia':(m.sum((1,2,3))==0).float(),
           'dice_mascara':2*(m*c[:,3:4]).sum((1,2,3))/(m.sum((1,2,3))+c[:,3:4].sum((1,2,3))).clamp_min(1)}
  for k,i in enumerate(ii):
   rows.append(dict(row=int(i),id=pairs.iloc[i].id,color_i=int(ci[k]),version=pairs.iloc[i].version,
    codec_color_vio_objetivo=bool(pairs.iloc[i].codec_color_vio_objetivo),
    **{name:float(v[k].cpu()) for name,v in scores.items()}))
  if keep:images.append(y.cpu())
 return pd.DataFrame(rows),torch.cat(images) if images else None

@torch.no_grad()
def reference_scores(indices):
 out=[]
 for j in range(0,len(indices),CONFIG['LOTE_EVAL']):
  ii=indices[j:j+CONFIG['LOTE_EVAL']];ci=pairs.iloc[ii].color_i.to_numpy();c=rgba_batch(ci);m=c[:,3:4]
  z=to_tokens(Z[torch.as_tensor(ci)].to(DEVICE));base=torch.zeros_like(z)
  with amp_ctx():oracle=color_decode(z,m).float();mean_color=color_decode(base,m).float()
  a=mse_mask(c[:,:3],oracle,m);b=mse_mask(c[:,:3],mean_color,m)
  for k,i in enumerate(ii):out.append(dict(row=int(i),color_i=int(ci[k]),codec_mse=float(a[k].cpu()),base_mse=float(b[k].cpu())))
 return pd.DataFrame(out)
REF_VAL=reference_scores(VAL_FIXED)
CODEC_MSE=macro(REF_VAL,'codec_mse');BASE_MSE=macro(REF_VAL,'base_mse')
print('Referencias en el mismo panel · codec:',CODEC_MSE,'| código medio:',BASE_MSE)
''',
'09_funciones.py': r'''
def warp_rgba(c):
 n=len(c);ang=(torch.rand(n,device=DEVICE)*2-1)*math.radians(3)
 scale=1+(torch.rand(n,device=DEVICE)*2-1)*.03
 xy=(torch.rand(n,2,device=DEVICE)*2-1)*(2/64)
 co,si=torch.cos(ang)/scale,torch.sin(ang)/scale
 theta=torch.stack([torch.stack([co,-si,xy[:,0]],1),torch.stack([si,co,xy[:,1]],1)],1)
 q=F.grid_sample(c,F.affine_grid(theta,c.shape,align_corners=False),align_corners=False,padding_mode='zeros')
 alpha=q[:,3:4];m=(alpha>.5).float()
 return torch.cat([(q[:,:3]/alpha.clamp_min(1e-3)).clamp(0,1)*m,m],1)

class RowSampler:
 def __init__(self):self.rng=np.random.default_rng(CONFIG['SEMILLA']+123)
 def get(self,n,p_pred=0.,p_aug=0.):
  rows=[];modes=[]
  for _ in range(n):
   if CAN_PRED and self.rng.random()<p_pred:
    rows.append(int(self.rng.choice(PRED_TRAIN)));modes.append('pred')
   else:
    ci=int(self.rng.choice(TRAIN_COLORS));rows.append(int(self.rng.choice(ROWS_BY_COLOR[ci])))
    modes.append('aug' if self.rng.random()<p_aug else 'clean')
  return np.array(rows,dtype=int),np.array(modes)
 def state_dict(self):return self.rng.bit_generator.state
 def load_state_dict(self,s):self.rng.bit_generator.state=s

def training_batch(sampler,p_pred,p_aug):
 rows,modes=sampler.get(CONFIG['LOTE'],p_pred,p_aug);ci=pairs.iloc[rows].color_i.to_numpy()
 c=rgba_batch(ci);flip=(torch.rand(len(rows),device=DEVICE)<.5) if CONFIG['ESPEJO'] else torch.zeros(len(rows),dtype=torch.bool,device=DEVICE)
 z=Z[torch.as_tensor(ci)].to(DEVICE);zm=ZM[torch.as_tensor(ci)].to(DEVICE)
 for j,i in enumerate(rows):
  if modes[j]=='pred':
   c[j]=PSEUDO[int(i)].to(DEVICE).float()/255
   z[j]=PZ[int(i)].to(DEVICE);zm[j]=PZM[int(i)].to(DEVICE)
 c=torch.where(flip[:,None,None,None],c.flip(-1),c)
 target=torch.where(flip[:,None,None,None],zm,z)
 aug=np.flatnonzero(modes=='aug')
 if len(aug):
  jj=torch.as_tensor(aug,device=DEVICE);cc=warp_rgba(c[jj]);c[jj]=cc
  target[jj]=encode_color_arrays(cc)
 h,photo=conditions(rows,flip)
 return rows,modes,c,h,photo,to_tokens(target)

def capture_rng():
 return dict(py=random.getstate(),np=np.random.get_state(),torch=torch.get_rng_state(),
             cuda=torch.cuda.get_rng_state_all() if AMP else None)
def restore_rng(s):
 random.setstate(s['py']);np.random.set_state(s['np']);torch.set_rng_state(s['torch'])
 if AMP and s['cuda'] is not None:
  for i,rng in enumerate(s['cuda'][:torch.cuda.device_count()]):torch.cuda.set_rng_state(rng,i)
@torch.no_grad()
def ema_update(ema,model,step):
 decay=min(CONFIG['EMA'],(1+step)/(10+step))
 for a,b in zip(ema.parameters(),model.parameters()):a.lerp_(b.detach(),1-decay)
 for a,b in zip(ema.buffers(),model.buffers()):a.copy_(b)
def lr_factor(step,total):
 if step<CONFIG['WARMUP']:return (step+1)/max(1,CONFIG['WARMUP'])
 t=(step-CONFIG['WARMUP'])/max(1,total-CONFIG['WARMUP'])
 return CONFIG['LR_FINAL']+(1-CONFIG['LR_FINAL'])*.5*(1+math.cos(math.pi*min(1.,t)))

@torch.no_grad()
def validation(model,curriculum=False):
 clean,_=evaluate2(model,VAL_FIXED)
 shuffle,_=evaluate2(model,VAL_FIXED,shuffle=True)
 mse=macro(clean);shuf=macro(shuffle)
 info=dict(mse=mse,psnr=macro(clean,'psnr'),deltaE76=macro(clean,'deltaE76'),
           mse_foto_barajada=shuf,ratio_codec=mse/max(1e-10,CODEC_MSE))
 if curriculum and len(PRED_VAL)>=CONFIG['MIN_VAL_PSEUDO']:
  pseudo,_=evaluate2(model,PRED_VAL,mode='pseudo');info['mse_pseudo']=macro(pseudo)
  info['score']=.7*mse+.3*info['mse_pseudo']
 else:info['mse_pseudo']=None;info['score']=mse
 return info

class Curriculum:
 def __init__(self,clean_hist=None):
  self.stage='A';self.entered=0;self.history=list(clean_hist or []);self.pred_disabled=False
 def ratios(self,step):
  if self.stage=='A':return 0.,0.
  if self.stage=='B':return 0.,CONFIG['P_AUG']
  p=CONFIG['PRED_MAX']*min(1.,(step-self.entered)/max(1,CONFIG['RAMPA_PRED']))
  # Se mantiene >=70% limpio; la corrupción alineada ocupa el resto del presupuesto.
  return p,max(0.,(CONFIG['P_AUG']-p)/max(1e-6,1-p))
 def update(self,step,info):
  self.history.append(info['mse'])
  if CONFIG['SMOKE']:
   if self.stage=='A' and step>=2:self.stage='B';self.entered=step
   elif self.stage=='B' and step>=4 and CAN_PRED:self.stage='C';self.entered=step
   return
  count=CONFIG['PLATEAU_EVALS']
  plateau=len(self.history)>=count and (max(self.history[-count:])-min(self.history[-count:]))/max(1e-9,min(self.history[-count:]))<CONFIG['PLATEAU_MEJORA']
  useful=info['mse']<(1-CONFIG['MEJORA_BASE'])*BASE_MSE
  photo=info['mse']<(1-CONFIG['MEJORA_FOTO'])*info['mse_foto_barajada']
  if self.stage=='A' and CONFIG['PASOS_LIMPIOS']+step>=CONFIG['MIN_PASOS_A'] and plateau and useful and (photo or not CONFIG['REQUERIR_USO_FOTO']):
   self.stage='B';self.entered=step
  elif self.stage=='B' and step-self.entered>=CONFIG['MIN_PASOS_B'] and CAN_PRED and not self.pred_disabled:
   self.stage='C';self.entered=step
  elif self.stage=='C' and info['mse']>min(self.history)*(1+CONFIG['DEGRADACION_MAX']):
   self.stage='B';self.entered=step;self.pred_disabled=True
 def state_dict(self):return dict(stage=self.stage,entered=self.entered,history=self.history,pred_disabled=self.pred_disabled)
 def load_state_dict(self,s):self.__dict__.update(s)

DATA_SHA=hashlib.sha256((pairs[['id','version','color_i','split_flow2','sha_foto_verificado']].to_csv(index=False)+
      hashlib.sha256(RGBA.numpy().tobytes()).hexdigest()).encode()).hexdigest()
CODE_SHA=hashlib.sha256(json.dumps(dict(sources=RUNTIME_SOURCES,flow1=FLOW1_MODEL_SOURCE,mask=MASK_CODEC_SOURCE),sort_keys=True).encode()).hexdigest()
def run_signature(arch,phase):
 # Ubicaciones físicas no forman parte de la firma: el ZIP completo puede moverse.
 skip={'RAIZ','SALIDA','REANUDAR_DE','COLOR_DIR','CK_FLOW1','CK_DITO','PARES_AUDITADOS','DINO_LOCAL','REANUDAR'}
 cfg={k:v for k,v in CONFIG.items() if k not in skip}
 return hashlib.sha256(json.dumps(dict(config=cfg,arch=arch,phase=phase,data=DATA_SHA,code=CODE_SHA),sort_keys=True).encode()).hexdigest()

''',
'10_preflight.py': r'''
assert DATA_SHA==ORIGIN['data_sha']==PARENT_CK['data_sha'],'Los datos no coinciden con los pesos.'
assert CODE_SHA==PARENT_CK['code_sha'],'Fuentes de modelo distintas al checkpoint original.'
assert CONFIG['DIM']==PARENT_CK['config']['DIM'] and DINO_DIM==PARENT_CK['dino_dim']
REFERENCE=Flow2DiT(WINNER,CONFIG,DINO_DIM).eval().requires_grad_(False)
REFERENCE.load_state_dict(BASELINE_EXPORT['ema'],strict=True)
# El EMA del checkpoint best y el export seleccionado deben representar los mismos pesos.
assert all(torch.equal(PARENT_CK['ema'][k],BASELINE_EXPORT['ema'][k]) for k in BASELINE_EXPORT['ema'])
print('Preflight correcto: pesos originales, código, datos y normalización compatibles.')
print('Referencia anterior · CFG:',ORIGIN['cfg_original'],'| val y test conservados:',len(VAL_FIXED),len(TEST_FIXED))
''',
'11_entrenamiento_20k.py': r'''
_ext_cfg={k:v for k,v in EXT.items() if k not in ['PAQUETE_ANTERIOR','RAIZ','SALIDA','REANUDAR']}
_cfg={k:v for k,v in CONFIG.items() if k not in ['RAIZ','SALIDA','REANUDAR_DE','COLOR_DIR','CK_FLOW1','CK_DITO','PARES_AUDITADOS','DINO_LOCAL','REANUDAR']}
EXT_CODE_SHA=hashlib.sha256(json.dumps(EXT_RUNTIME_SOURCES,sort_keys=True).encode()).hexdigest()
EXT_SIGNATURE=hashlib.sha256(json.dumps(dict(parent=ORIGIN['checkpoint_sha'],config=_cfg,
 extension=_ext_cfg,data=DATA_SHA,code=CODE_SHA,extension_code=EXT_CODE_SHA),sort_keys=True).encode()).hexdigest()
START_LR=float(PARENT_CK['opt']['param_groups'][0]['lr'])
def extension_lr(step,total):
 warm=min(EXT['WARMUP_EXTRA'],max(1,total-1));peak=float(EXT['LR_MAX'])
 if step<warm:return START_LR+(peak-START_LR)*(step+1)/warm
 u=(step-warm)/max(1,total-warm)
 return peak*(EXT['LR_FINAL_FRACCION']+(1-EXT['LR_FINAL_FRACCION'])*.5*(1+math.cos(math.pi*min(1.,u))))

def train_extension(arch):
 phase='extension_20k';initial=None;clean_hist=[]
 total=EXT['PASOS_EXTRA']
 folder=RUN/'runs'/f'{arch}_{phase}';folder.mkdir(parents=True,exist_ok=True)
 signature=EXT_SIGNATURE;seed_all(CONFIG['SEMILLA'])
 model=Flow2DiT(arch,CONFIG,DINO_DIM).to(DEVICE)
 if initial is not None:model.load_state_dict(initial,strict=True)
 ema=copy.deepcopy(model).eval().requires_grad_(False)
 opt=torch.optim.AdamW(model.parameters(),lr=EXT['LR_MAX'],weight_decay=CONFIG['WD'])
 scaler=torch.amp.GradScaler('cuda',enabled=AMP);sampler=RowSampler();controller=Curriculum(clean_hist)
 step=0;best=float('inf');history=[];vals=[];previous=0.;skipped=0
 def state():
  return dict(arch=arch,phase=phase,step=step,best=best,ema=cpu_weights(ema),model=cpu_weights(model),
   opt=opt.state_dict(),scaler=scaler.state_dict(),rng=capture_rng(),sampler=sampler.state_dict(),
   controller=controller.state_dict(),history=history,validations=vals,seconds=previous+time.monotonic()-start,
   signature=signature,config=CONFIG,data_sha=DATA_SHA,code_sha=CODE_SHA,skipped=skipped,dino_dim=DINO_DIM,
   artifact_role='flow2_training',smoke=CONFIG['SMOKE'],parent_sha=ORIGIN['checkpoint_sha'],extra_steps=step,extension=EXT)
 last=folder/'last.pt'
 if CONFIG['REANUDAR'] and last.is_file():
  ck=torch.load(last,map_location='cpu',weights_only=False)
  assert ck['signature']==signature,'Código, datos o plan de entrenamiento distintos. Usar otra VERSION.'
  model.load_state_dict(ck['model']);ema.load_state_dict(ck['ema']);opt.load_state_dict(ck['opt']);scaler.load_state_dict(ck['scaler'])
  sampler.load_state_dict(ck['sampler']);controller.load_state_dict(ck['controller']);restore_rng(ck['rng'])
  step=ck['step'];best=ck['best'];history=ck['history'];vals=ck['validations'];previous=ck['seconds'];skipped=ck['skipped'];del ck
  assert (folder/'best.pt').is_file(),'Falta best asociado al last'
  print(arch,phase,': reanudando',step,'de',total)
 else:
  model.load_state_dict(PARENT_CK['model'],strict=True);ema.load_state_dict(PARENT_CK['ema'],strict=True)
  opt.load_state_dict(PARENT_CK['opt']);scaler.load_state_dict(PARENT_CK['scaler'])
  sampler.load_state_dict(PARENT_CK['sampler']);restore_rng(PARENT_CK['rng'])
  controller.load_state_dict(copy.deepcopy(PARENT_CK['controller']))
  if not EXT['CONTINUAR_CURRICULUM']:controller.stage='A'
 start=time.monotonic()
 # La fase de adaptación también compite contra el modelo de entrada sin modificar.
 if not (folder/'best.pt').is_file():
  info=validation(ema,curriculum=False);best=info['score'];vals.append(dict(step=0,stage='A',**info))
  atomic_save(state(),folder/'best.pt');atomic_save(state(),last)
 params=sum(p.numel() for p in model.parameters());print(arch,phase,':',round(params/1e6,2),'M parámetros')
 bar=tqdm(total=total,initial=step,desc=f'Flow 2 · {arch} · {phase}',unit='step',mininterval=1.)
 sums={};window=0;overflows=0
 try:
  while step<total:
   model.train();opt.zero_grad(set_to_none=True)
   lr=extension_lr(step,total)
   for g in opt.param_groups:g['lr']=lr
   pp,pa=controller.ratios(PARENT_CK['step']+step) if EXT['CONTINUAR_CURRICULUM'] else (0.,0.)
   stats={}
   for micro in range(CONFIG['ACUMULAR']):
    rows,modes,c,h,photo,target=training_batch(sampler,pp,pa);m=c[:,3:4];a=to_tokens(active(m))
    noise=torch.randn_like(target)*a;t=torch.randn(len(rows),device=DEVICE).sigmoid()
    xt=(1-t[:,None,None])*noise+t[:,None,None]*target
    drop=torch.rand(len(rows),device=DEVICE)<CONFIG['P_SIN_FOTO']
    with amp_ctx():pred=model(xt,t,h,photo,m,drop)
    diff=(pred.float()-(target-noise)).square()*a
    fm=(diff.sum((1,2))/(8*a.sum((1,2))).clamp_min(1)).mean()
    aux=torch.zeros((),device=DEVICE)
    selected=torch.nonzero((t>=.5)&~drop).flatten()[:CONFIG['AUX_MAX']]
    if CONFIG['AUX_RGB'] and step%CONFIG['AUX_CADA']==0 and len(selected):
     estimate=xt[selected]+(1-t[selected,None,None])*pred[selected].float()
     with amp_ctx():out=color_decode(estimate,m[selected]).float()
     aux=mse_mask(c[selected,:3],out,m[selected]).mean()
    loss=fm+CONFIG['AUX_RGB']*aux
    if not torch.isfinite(loss):raise FloatingPointError('Pérdida no finita; last conserva el estado anterior')
    scaler.scale(loss/CONFIG['ACUMULAR']).backward()
    for k,v in dict(loss=loss.detach(),fm=fm.detach(),rgb=aux.detach(),pred_fraction=float(np.mean(modes=='pred'))).items():
     stats[k]=stats.get(k,0)+float(v)/CONFIG['ACUMULAR']
   scaler.unscale_(opt);gnorm=torch.nn.utils.clip_grad_norm_(model.parameters(),CONFIG['CLIP'])
   if not torch.isfinite(gnorm):
    if not AMP:raise FloatingPointError('Gradiente no finito')
    scaler.step(opt);scaler.update();skipped+=1;overflows+=1
    if overflows>=50:raise FloatingPointError('50 overflows AMP consecutivos')
    continue
   overflows=0;scaler.step(opt);scaler.update();ema_update(ema,model,PARENT_CK['step']+step);step+=1;bar.update(1);window+=1
   for k,v in stats.items():sums[k]=sums.get(k,0)+v
   if step%CONFIG['LOG_CADA']==0 or step==total:
    row=dict(step=step,stage='A' if phase=='clean' else controller.stage,**{k:v/window for k,v in sums.items()},
      lr=lr,grad_norm=float(gnorm.cpu()),amp_skips=skipped)
    history.append(row);bar.set_postfix(loss=f'{row["loss"]:.4f}',etapa=row['stage'],pred=f'{row["pred_fraction"]:.0%}')
    if step%CONFIG['LOG_CADA']==0:
     tqdm.write(f'{arch} {phase} · step {step}: loss={row["loss"]:.5f} · FM={row["fm"]:.5f} · RGB={row["rgb"]:.5f} · etapa={row["stage"]} · pred={row["pred_fraction"]:.1%}')
    sums={};window=0
   if step%CONFIG['EVAL_CADA']==0 or step==total:
    info=validation(ema,curriculum=False);oldstage=controller.stage
    if EXT['CONTINUAR_CURRICULUM']:controller.update(PARENT_CK['step']+step,info)
    vals.append(dict(step=step,stage=controller.stage,**info));improved=info['score']<best
    if improved:best=info['score']
    ck=state()
    if improved:atomic_save(ck,folder/'best.pt')
    atomic_save(ck,last)
    pd.DataFrame(history).to_csv(folder/'history.csv',index=False);pd.DataFrame(vals).to_csv(folder/'validation.csv',index=False)
    tqdm.write(f'{arch} {phase} · {step}: MSE={info["mse"]:.5f} · R codec={info["ratio_codec"]:.2f} · etapa={controller.stage}')
    if oldstage!=controller.stage:tqdm.write(f'Curriculum: {oldstage} → {controller.stage}')
   elif step%CONFIG['GUARDAR_CADA']==0:atomic_save(state(),last)
 finally:bar.close()
 ck=torch.load(folder/'best.pt',map_location='cpu',weights_only=False)
 ema.load_state_dict(ck['ema']);ema.cpu();del model,opt,scaler,ck;gc.collect()
 if AMP:torch.cuda.empty_cache()
 return ema,pd.DataFrame(history),pd.DataFrame(vals),dict(arch=arch,phase=phase,best_score=best,params=params)

FINAL,HISTORY,VALS,TRAIN_INFO=train_extension(WINNER)
''',
'12_seleccion.py': r'''
FINAL.to(DEVICE);REFERENCE.to(DEVICE)
# La referencia permanece fija, con sus pesos y CFG originales.
BEFORE_VAL,_=evaluate2(REFERENCE,VAL_FIXED,w=ORIGIN['cfg_original'])
BEFORE_VAL.to_csv(RUN/'val_antes_20k.csv',index=False)
grid=CONFIG['CFG_GRID'] if EXT['RESELECCIONAR_CFG'] else [ORIGIN['cfg_original']]
cfg_results=[]
for w in tqdm(grid,desc='CFG · SOLO validación',unit='valor'):
 scores,_=evaluate2(FINAL,VAL_FIXED,w=w);cfg_results.append(dict(w=w,mse=macro(scores)))
pd.DataFrame(cfg_results).to_csv(RUN/'cfg_validacion_20k.csv',index=False)
CFG_BEST=float(min(cfg_results,key=lambda a:a['mse'])['w'])
AFTER_VAL,_=evaluate2(FINAL,VAL_FIXED,w=CFG_BEST)
AFTER_VAL.to_csv(RUN/'val_despues_20k.csv',index=False)
BEFORE_MSE=macro(BEFORE_VAL);AFTER_MSE=macro(AFTER_VAL)
USE_EXTENSION=AFTER_MSE<BEFORE_MSE
# Puede conservarse el anterior si ni el best extendido lo supera con su CFG elegido.
DEPLOY=FINAL if USE_EXTENSION else REFERENCE
DEPLOY_CFG=CFG_BEST if USE_EXTENSION else ORIGIN['cfg_original']
print('Validación fija: antes=',round(BEFORE_MSE,6),'| best extendido=',round(AFTER_MSE,6))
print('Export seleccionado SOLO con validación:', 'extensión' if USE_EXTENSION else 'anterior', '| CFG:',DEPLOY_CFG)
print('El test que sigue no cambia la selección.')
FINAL.cpu();REFERENCE.cpu()
fig,ax=plt.subplots(1,2,figsize=(12,4))
if len(HISTORY):ax[0].plot(HISTORY.step,HISTORY.loss,label='Loss');ax[0].plot(HISTORY.step,HISTORY.fm,label='FM')
ax[0].set_title('Entrenamiento adicional');ax[0].legend()
ax[1].plot(VALS.step,VALS.mse,label='Validación · CFG 1')
ax[1].axhline(float(VALS.iloc[0].mse),ls='--',color='gray',label='Entrada · CFG 1')
ax[1].set_title('Validación fija durante 20k');ax[1].legend()
for a in ax:a.set_xlabel('Pasos adicionales');a.grid(alpha=.2)
fig.tight_layout();fig.savefig(RUN/'curvas_20k.png',dpi=140);plt.show();plt.close(fig)
''',
'13_test.py': r'''
# Son los mismos 100 ejemplos y las mismas semillas del experimento anterior.
# Este panel ya fue inspeccionado: comparar resultados es diagnóstico; no es un nuevo test independiente.
T_SCORES={};T_IMAGES={};G_SCORES={};G_IMAGES={}
for label,model,w in [('antes',REFERENCE,ORIGIN['cfg_original']),('despues_best',FINAL,CFG_BEST)]:
 model.to(DEVICE)
 d,im=evaluate2(model,TEST_FIXED,w=w,keep=True,progress=True)
 g,gi=evaluate2(model,TEST_FIXED,mode='generada',w=w,keep=True,progress=True)
 # Diagnóstico de detalles finos: error en una banda interior de dos píxeles del contorno GT.
 # Es una región geométrica reproducible, no una anotación semántica de arrugas o lentes.
 ci=pairs.iloc[TEST_FIXED].color_i.to_numpy();gt=RGBA[torch.as_tensor(ci)].float()/255
 mm=gt[:,3:4];eroded=1-F.max_pool2d(F.pad(1-mm,(2,2,2,2),value=1),5,stride=1)
 edge=mm*(1-eroded);edge_mse=mse_mask(gt[:,:3],im,edge)
 edge_mse=torch.where(edge.sum((1,2,3))>0,edge_mse,torch.full_like(edge_mse,float('nan')))
 d['mse_borde_2px']=edge_mse.numpy()
 T_SCORES[label]=d;T_IMAGES[label]=im;G_SCORES[label]=g;G_IMAGES[label]=gi
 d.to_csv(RUN/f'test_20k_mascara_real_{label}.csv',index=False)
 g.to_csv(RUN/f'test_20k_mascara_generada_{label}_PROXY.csv',index=False)
 model.cpu()
comparison=pd.DataFrame([dict(modelo=k,mse=macro(d),mse_borde_2px=macro(d,'mse_borde_2px'),psnr=macro(d,'psnr'),deltaE76=macro(d,'deltaE76')) for k,d in T_SCORES.items()])
display(comparison);comparison.to_csv(RUN/'comparacion_test_20k.csv',index=False)
paired=T_SCORES['antes'][['row','id','color_i','mse','mse_borde_2px','deltaE76']].merge(
 T_SCORES['despues_best'][['row','mse','mse_borde_2px','deltaE76']],on='row',suffixes=('_antes','_despues'),validate='one_to_one')
paired['delta_mse']=paired.mse_despues-paired.mse_antes
paired['delta_mse_borde_2px']=paired.mse_borde_2px_despues-paired.mse_borde_2px_antes
paired['delta_deltaE76']=paired.deltaE76_despues-paired.deltaE76_antes
paired.to_csv(RUN/'test_20k_por_foto_antes_despues.csv',index=False)
for label,d in T_SCORES.items():
 d.groupby(['codec_color_vio_objetivo','color_i'])[['mse','deltaE76']].mean().groupby('codec_color_vio_objetivo').mean().to_csv(RUN/f'test_20k_codec_visto_{label}.csv')
print('Fotos con menor MSE después:',round(float((paired.delta_mse<0).mean())*100,1),'%')
print('Con máscaras generadas se reportan proxies espaciales; no GT cromático alineado.')

def composite(rgb,m):
 rgb=np.asarray(rgb);m=np.asarray(m)
 return np.clip((rgb*m+np.array([.98,.79,.16])[:,None,None]*(1-m)).transpose(1,2,0),0,1)
figdir=RUN/'test_imagenes_20k';figdir.mkdir(exist_ok=True)
for start in tqdm(range(0,len(TEST_FIXED),CONFIG['POR_PAGINA']),desc='MOSTRANDO antes/después · test fijo',unit='página'):
 end=min(len(TEST_FIXED),start+CONFIG['POR_PAGINA']);ii=TEST_FIXED[start:end];ci=pairs.iloc[ii].color_i.to_numpy()
 c=rgba_batch(ci);m=c[:,3:4]
 with torch.no_grad(),amp_ctx():oracle=color_decode(to_tokens(Z[torch.as_tensor(ci)].to(DEVICE)),m).cpu()
 c=c.cpu();fig,axes=plt.subplots(len(ii),8,figsize=(19,2.5*len(ii)),squeeze=False,facecolor='#f4f6fb')
 titles=['Foto','LEGO objetivo','VAE referencia','Antes · GT','Después best · GT','Flow 1 · máscara','Antes · generada','Después best · generada']
 for row,idx in enumerate(ii):
  j=start+row;mask=c[row,3:4].numpy();mh=PRED_MASKS[int(idx)].numpy()
  panels=[FOTOS[idx].permute(1,2,0).numpy(),composite(c[row,:3],mask),composite(oracle[row],mask),
          composite(T_IMAGES['antes'][j],mask),composite(T_IMAGES['despues_best'][j],mask),mh[0],
          composite(G_IMAGES['antes'][j],mh),composite(G_IMAGES['despues_best'][j],mh)]
  for col,panel in enumerate(panels):
   axes[row,col].imshow(panel,cmap='gray' if panel.ndim==2 else None,vmin=0,vmax=1 if panel.ndim==2 else None)
   axes[row,col].axis('off')
   if row==0:axes[row,col].set_title(titles[col],fontsize=9)
  axes[row,0].text(0,-.03,str(pairs.iloc[idx].id),transform=axes[row,0].transAxes,fontsize=7)
  for col,label in [(3,'antes'),(4,'despues_best')]:
   value=float(T_SCORES[label].iloc[j].mse)
   axes[row,col].text(.5,-.035,f'MSE {value:.4f}',ha='center',transform=axes[row,col].transAxes,fontsize=7)
 fig.suptitle(f'20k adicionales · {start+1}–{end} de {len(TEST_FIXED)} · CFG antes {ORIGIN["cfg_original"]:g} / después {CFG_BEST:g}',fontsize=14,fontweight='bold')
 fig.tight_layout(rect=[0,0,1,.97]);fig.savefig(figdir/f'test_20k_{start+1:03d}_{end:03d}.png',dpi=120)
 plt.show();plt.close(fig)
''',
'14_export.py': r'''
EXPORT=RUN/'inferencia';EXPORT.mkdir(exist_ok=True)
# Original intacto en anterior/inferencia. Nuevo export: ganador entre anterior y extensión, por validación.
atomic_save(dict(ema=cpu_weights(DEPLOY),arch=WINNER,config=CONFIG,dino_dim=DINO_DIM,cfg_w=DEPLOY_CFG,
 data_sha=DATA_SHA,code_sha=CODE_SHA,artifact_role='flow2_color',smoke=CONFIG['SMOKE']),EXPORT/'flow2_color.pt')
# Guardar también el candidato extendido aunque validación prefiera el anterior.
atomic_save(dict(ema=cpu_weights(FINAL),arch=WINNER,config=CONFIG,dino_dim=DINO_DIM,cfg_w=CFG_BEST,
 data_sha=DATA_SHA,code_sha=CODE_SHA,artifact_role='flow2_color',smoke=CONFIG['SMOKE']),EXPORT/'flow2_color_candidato_20k.pt')
(EXPORT/'model.py').write_text('import math\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n'+MODEL_SOURCE)
(EXPORT/'runtime_constants.json').write_text(json.dumps(dict(flow1_model=FLOW1_MODEL_SOURCE,mask_codec=MASK_CODEC_SOURCE,
 flow1_generate=FLOW1_GENERATE_SOURCE,smoke_vae=SMOKE_VAE_SOURCE,crop=CROP_SOURCE,generate2=GENERATE2_SOURCE)))
(EXPORT/'pipeline.py').write_text(INFERENCE_SOURCE)
manifest=dict(ganador_arquitectura=WINNER,cfg_w=DEPLOY_CFG,smoke=CONFIG['SMOKE'],data_sha=DATA_SHA,
 n_train=len(TRAIN),n_val=len(VAL_FIXED),n_test=len(TEST_FIXED),pasos_extra=EXT['PASOS_EXTRA'],
 parent_checkpoint_sha=ORIGIN['checkpoint_sha'],extension_signature=EXT_SIGNATURE,
 validacion_antes=BEFORE_MSE,validacion_best_20k=AFTER_MSE,seleccion='extension' if USE_EXTENSION else 'anterior',
 pred_train_aceptadas=len(PRED_TRAIN),colores_pred_aceptados=int(accepted_colors),
 curriculum_llego_a_C=bool((VALS.stage=='C').any()),continuar_curriculum=EXT['CONTINUAR_CURRICULUM'],
 eleccion='SOLO validacion; el test ya fue inspeccionado y se usa como comparacion diagnostica')
(RUN/'resumen.json').write_text(json.dumps(manifest,indent=2));(RUN/'config.json').write_text(json.dumps(CONFIG,indent=2))
(RUN/'config_extension_20k.json').write_text(json.dumps(EXT,indent=2))
(RUN/'LEEME_20K.txt').write_text('Continuación de 20000 pasos ADICIONALES solo del ganador.\n'
 'El modelo inicial y sus resultados se conservan en anterior/. Todos los runs y assets originales permanecen.\n'
 'runs/*_extension_20k/best.pt y last.pt guardan optimizador, EMA, RNG, sampler y configuración para continuar.\n'
 'El export inferencia/flow2_color.pt se elige con validación; inferencia/flow2_color_candidato_20k.pt conserva el candidato.\n'
 'Por defecto se entrena con máscaras reales para aislar el efecto de más pasos; Flow 1 y VAE siguen congelados.\n'
 'test_imagenes_20k contiene las 100 comparaciones mostradas en pantalla.\n'
 'Para reanudar una interrupción: adjuntar ESTE ZIP completo y mantener el mismo plan de 20000 pasos.\n')

def render_literal(value):
 if isinstance(value,str):
  for q in [chr(39)*3,chr(34)*3]:
   if q not in value and not value.endswith(('\\',q[0])):return 'r'+q+value+q
  return repr(value)
 if isinstance(value,dict):return '{\n'+''.join(repr(k)+': '+render_literal(v)+',\n' for k,v in value.items())+'}'
 return repr(value)
constants=dict(FLOW1_MODEL_SOURCE=FLOW1_MODEL_SOURCE,MASK_CODEC_SOURCE=MASK_CODEC_SOURCE,
 FLOW1_GENERATE_SOURCE=FLOW1_GENERATE_SOURCE,SMOKE_VAE_SOURCE=SMOKE_VAE_SOURCE,CROP_SOURCE=CROP_SOURCE,
 MODEL_SOURCE=MODEL_SOURCE,GENERATE2_SOURCE=GENERATE2_SOURCE,INFERENCE_SOURCE=INFERENCE_SOURCE,
 RESEARCH_TEXT=RESEARCH_TEXT,RUNTIME_SOURCES=RUNTIME_SOURCES,EXT_RUNTIME_SOURCES=EXT_RUNTIME_SOURCES)
bootstrap='\n\n'.join(k+' = '+render_literal(v) for k,v in constants.items())
# El intake debe preceder al setup; el bootstrap de fuentes puede ir primero en la versión reproducible.
nb=dict(nbformat=4,nbformat_minor=5,metadata=dict(kernelspec=dict(name='python3',display_name='Python 3',language='python')),
 cells=[dict(cell_type='code',id='fuentes',metadata={'jupyter':{'source_hidden':True}},execution_count=None,outputs=[],source=bootstrap)]+
 [dict(cell_type='code',id=f'ext{i}',metadata={},execution_count=None,outputs=[],source=src) for i,src in enumerate(EXT_RUNTIME_SOURCES.values())])
(RUN/'flow2_continuacion_20k.ipynb').write_text(json.dumps(nb,ensure_ascii=False,indent=1))
(RUN/'entrenar_continuacion_20k.py').write_text(bootstrap+'\n\n'+'\n\n'.join(EXT_RUNTIME_SOURCES.values()))

# Comprobar el export fuera de los objetos del notebook.
pipeline_mod=load_module('flow2_ext_export',EXPORT/'pipeline.py')
pipeline=pipeline_mod.cargar_pipeline(RUN,device=str(DEVICE))
idx=int(TEST_FIXED[0]);mh=PRED_MASKS[idx].numpy()[0]
if not 0<float(mh.mean())<1:
 mh=RGBA[int(pairs.iloc[idx].color_i),3].numpy()/255
answer=pipeline.inferir_array(FOTOS[idx].permute(1,2,0).numpy(),mascara=mh,seed=CONFIG['SEMILLA']+idx,mostrar=False)
assert answer['rgba'].shape==(128,128,4) and np.array_equal(answer['rgba'][:,:,3],mh.astype(np.uint8)*255)
assert np.isfinite(answer['rgb']).all()
del pipeline
print('Export independiente comprobado.')

# UN ZIP contiene TODO: original, continuación, modelos, optimizador, caches, resultados e imágenes.
full=RUN.parent/(RUN.name+'_TODO.zip');files=sorted(p for p in RUN.rglob('*') if p.is_file())
with zipfile.ZipFile(full,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=4,allowZip64=True) as z:
 for p in tqdm(files,desc='UN ZIP · TODO original y 20k',unit='archivo'):z.write(p,Path(RUN.name)/p.relative_to(RUN))
with zipfile.ZipFile(full) as z:
 for member in tqdm(z.infolist(),desc='Verificando CRC · ZIP completo',unit='archivo'):
  if member.is_dir():continue
  with z.open(member) as stream:
   for _ in iter(lambda:stream.read(2**20),b''):pass
print('Descargar:',full,'| MiB:',round(full.stat().st_size/2**20,1));display(FileLink(str(full)))
''',
}

## Dispositivos y configuración heredada

In [ ]:

CONFIG=dict(PARENT_CONFIG)
CONFIG.update(VERSION=EXT['VERSION'],SALIDA=EXT['SALIDA'],RAIZ=EXT['RAIZ'],
 REANUDAR_DE=None,REANUDAR=True,LOG_CADA=100,EVAL_CADA=EXT['EVAL_CADA'],
 GUARDAR_CADA=EXT['GUARDAR_CADA'])
DEVICE=torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
DINO_DEVICE=torch.device('cuda:1' if torch.cuda.device_count()>1 else DEVICE)
AMP=DEVICE.type=='cuda'
if DEVICE.type=='cpu':torch.set_num_threads(min(4,os.cpu_count() or 1))
def seed_all(seed):
 random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
 if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
def amp_ctx():return torch.autocast('cuda',dtype=torch.float16) if AMP else nullcontext()
def sha_file(path):
 h=hashlib.sha256()
 with open(path,'rb') as f:
  for b in iter(lambda:f.read(1<<20),b''):h.update(b)
 return h.hexdigest()
def atomic_save(obj,path):
 path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
 temp=path.with_suffix(path.suffix+'.tmp');torch.save(obj,temp);temp.replace(path)
def load_module(name,path):
 spec=importlib.util.spec_from_file_location(name,path);mod=importlib.util.module_from_spec(spec)
 spec.loader.exec_module(mod);return mod
def cpu_weights(model):return {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
seed_all(CONFIG['SEMILLA']);torch.backends.cudnn.benchmark=False
print('Flow 2:',DEVICE,'| DINO:',DINO_DEVICE,'| AMP:',AMP)
print('Lote efectivo:',CONFIG['LOTE']*CONFIG['ACUMULAR'],'| resúmenes cada 100 pasos')
if CONFIG['SMOKE']:print('SMOKE: datos y pesos artificiales; no evalúa calidad.')



## Recuperar los codecs congelados

In [ ]:

def safe_extract(z,names,dest):
 dest=Path(dest);dest.mkdir(parents=True,exist_ok=True)
 for name in names:
  rel=Path(name)
  if rel.is_absolute() or '..' in rel.parts:raise ValueError('Ruta ZIP insegura: '+name)
  if name.endswith('/'):continue
  p=dest/rel;p.parent.mkdir(parents=True,exist_ok=True);p.write_bytes(z.read(name))
 return dest
def choose(paths,label):
 paths=sorted(set(Path(p) for p in paths if Path(p).is_file()),key=str)
 if not paths:raise FileNotFoundError(label)
 if len(paths)>1 and len({sha_file(p) for p in paths})>1:raise ValueError(f'{label}: varias versiones; fija CONFIG: {paths}')
 return paths[0]

def prepare_assets():
 existing_color=RUN/'assets'/'color';existing_flow=RUN/'assets'/'flow1'
 if (existing_color/'color_codec.py').is_file() and (existing_flow/'flow1_dito.pt').is_file() and (RUN/'cache'/'rgba.pt').is_file():
  return existing_color,existing_flow,[]
 root=Path(CONFIG['RAIZ']);files=[p for p in tqdm(root.rglob('*'),desc='Indexando inputs',unit='ruta') if p.is_file()]
 archives=[p for p in files if p.suffix.lower()=='.zip']
 def find_asset(name,explicit=None):
  if explicit:return choose([explicit],name)
  candidates=[p for p in files if p.name==name]
  if candidates:return choose(candidates,name)
  for archive in tqdm(archives,desc='Buscando '+name+' en ZIP',unit='zip',leave=False):
   with zipfile.ZipFile(archive) as z:
    members=[n for n in z.namelist() if Path(n).name==name]
    if not members:continue
    dest=RUN/'assets'/'imports'/sha_file(archive)[:12]
    selected=set(members)
    for member in members:
     parent=Path(member).parent
     if name=='color_codec.py':selected.update(n for n in z.namelist() if str(Path(n).parent).startswith(str(parent)))
     if name=='flow1_dito.pt':
      selected.update(n for n in z.namelist() if Path(n).parent==parent and Path(n).name in ['pares_auditados.csv','stats_flow.pt'])
    safe_extract(z,sorted(selected),dest)
    candidates.extend(dest/n for n in members)
  return choose(candidates,'Falta '+name)
 color_script=find_asset('color_codec.py',Path(CONFIG['COLOR_DIR'])/'color_codec.py' if CONFIG['COLOR_DIR'] else None)
 color_dir=RUN/'assets'/'color'
 shutil.copytree(color_script.parent,color_dir,dirs_exist_ok=True,ignore=shutil.ignore_patterns('__pycache__'))
 ck_flow=find_asset('flow1_dito.pt',CONFIG['CK_FLOW1'])
 flow_dest=RUN/'assets'/'flow1';flow_dest.mkdir(exist_ok=True)
 shutil.copyfile(ck_flow,flow_dest/'flow1_dito.pt')
 audit=find_asset('pares_auditados.csv',CONFIG['PARES_AUDITADOS'])
 shutil.copyfile(audit,flow_dest/'pares_auditados.csv')
 cp=torch.load(flow_dest/'flow1_dito.pt',map_location='cpu',weights_only=False)
 if cp.get('config',{}).get('SMOKE'):raise ValueError('El Flow 1 adjunto es una prueba SMOKE')
 expected=cp.get('tokenizer_sha')
 if not expected:raise ValueError('Flow 1 sin tokenizer_sha; se requiere el export verificado')
 candidates=[Path(CONFIG['CK_DITO'])] if CONFIG['CK_DITO'] else [p for p in files if p.name=='best.pt' and p.parent.name=='dito_base']
 matched=[p for p in candidates if p.is_file() and sha_file(p)==expected]
 if not matched:
  for archive in tqdm(archives,desc='Localizando el DiTo de máscaras exacto',unit='zip',leave=False):
   with zipfile.ZipFile(archive) as z:
    for member in z.namelist():
     p=Path(member)
     if p.name=='best.pt' and p.parent.name=='dito_base':
      b=z.read(member)
      if hashlib.sha256(b).hexdigest()==expected:
       target=flow_dest/'mask_dito.pt';target.write_bytes(b);matched.append(target)
 if not matched:raise FileNotFoundError('Falta el best.pt de dito_base que corresponde al Flow 1. Añade el output de geometría.')
 if matched[0]!=flow_dest/'mask_dito.pt':shutil.copyfile(matched[0],flow_dest/'mask_dito.pt')
 return color_dir,flow_dest,files

if CONFIG['SMOKE']:
 COLOR_DIR=RUN/'assets'/'color';F1_DIR=RUN/'assets'/'flow1';INPUT_FILES=[];FLOW1_CK=None
 color_module=load_module('flow2_smoke_color',COLOR_DIR/'color_codec.py')
 COLOR_CK=torch.load(COLOR_DIR/'vae_control.pt',map_location='cpu',weights_only=False)
 COLOR=color_module.VAEColor();COLOR.load_state_dict(COLOR_CK['ema']);COLOR.eval().requires_grad_(False)
else:
 COLOR_DIR,F1_DIR,INPUT_FILES=prepare_assets()
 selection=json.loads((COLOR_DIR/'seleccion.json').read_text())
 if selection.get('smoke'):raise ValueError('El paquete de color adjunto es SMOKE')
 color_module=load_module('flow2_color_codec',COLOR_DIR/'color_codec.py')
 COLOR,COLOR_CK=color_module.cargar_color(COLOR_DIR/selection['checkpoint'],DEVICE)
 if COLOR_CK['kind']!='vae_control':raise ValueError('Esta versión integra la VAE de color ganadora; se esperaba vae_control')
 FLOW1_CK=torch.load(F1_DIR/'flow1_dito.pt',map_location='cpu',weights_only=False)
 COLOR_STATS=COLOR_CK['stats']
 COLOR.requires_grad_(False).eval()
COLOR=COLOR.to(DEVICE).eval().requires_grad_(False)
COLOR_STATS=COLOR_CK['stats']
C_MEAN=COLOR_STATS['media'].to(DEVICE)[None,:,None,None]
C_STD=COLOR_STATS['desv'].to(DEVICE)[None,:,None,None].clamp_min(1e-6)
def active(m):return (F.adaptive_max_pool2d(m,16)>.5).float()
def to_tokens(z):return z.flatten(2).transpose(1,2)
def from_tokens(z):return z.transpose(1,2).reshape(len(z),8,16,16)
def standardize(z,m):return ((z-C_MEAN)/C_STD)*active(m)
def unstandardize(z,m):return (z*C_STD+C_MEAN)*active(m)
def color_decode(z,m):
 # Mantiene gradiente respecto a z; los parámetros del decoder están congelados.
 return COLOR.decodificar(unstandardize(from_tokens(z),m),m)
print('Codec de color congelado:',COLOR_CK['kind'])


## Restaurar las mismas particiones

In [ ]:

def canonical_rgba(path):
 with Image.open(path) as im:
  if 'A' not in im.getbands() and 'transparency' not in im.info:raise ValueError('Estampa sin alfa: '+str(path))
  a=np.asarray(im.convert('RGBA')).copy()
 h,w=a.shape[:2]
 if h>128 or w>128:raise ValueError(f'Tamaño inesperado {w}x{h}: {path}; no redimensionar')
 out=np.zeros((4,128,128),np.uint8);y=(128-h)//2;x=(128-w)//2;m=a[:,:,3]>=128
 out[:3,y:y+h,x:x+w]=(a[:,:,:3]*m[:,:,None]).transpose(2,0,1);out[3,y:y+h,x:x+w]=m*255
 if not 0<out[3].mean()<255:raise ValueError('Máscara vacía/completa: '+str(path))
 return out

def fixture_data():
 root=RUN/'fixture';root.mkdir(exist_ok=True);aliases=[];rows=[];arrays=[]
 for i in range(18):
  rgba=np.zeros((128,128,4),np.uint8);rgba[25:34,25:36,3]=255;rgba[25:34,85:96,3]=255
  rgba[65:70+i%3,45:80,3]=255;rgba[88+i%6,20+i,3]=255
  rgba[:,:,:3]=(rgba[:,:,3,None]>0)*np.array([25+i*5,15+i*2,30],np.uint8)
  p=root/f's{i:02d}.png';Image.fromarray(rgba).save(p);c=canonical_rgba(p);arrays.append(c)
  split='entrenamiento' if i<12 else ('validacion' if i<15 else 'prueba')
  cid=('real:' if i<9 else 'int:')+f's{i:02d}'
  aliases.append(dict(id=cid,color_i=i,particion=split,sha_color=hashlib.sha256(c.tobytes()).hexdigest(),ruta=str(p),tipo='real' if i<9 else 'intermedia'))
  for k in range(2):
   photo=root/f'photo_{i}_{k}.jpg';bg=np.array([220,200,175],np.uint8)
   a=(c[:3].transpose(1,2,0)*(c[3,:, :,None]>0)+bg*(c[3,:,:,None]==0)).astype(np.uint8)
   Image.fromarray(a).resize((256,256)).save(photo)
   rows.append(dict(id=f's{i:02d}_{k:02d}',pieza=f's{i:02d}',k=k,version='g1' if i<9 else 'gi',
      split_flow=split,archivo=str(photo),color_i=i,color_id=cid,sha_foto_efectivo=sha_file(photo)))
 return pd.DataFrame(rows),pd.DataFrame(aliases),torch.from_numpy(np.stack(arrays))

if (RUN/'cache'/'rgba.pt').is_file() and (RUN/'pares_flow2.csv').is_file() and (RUN/'manifest_color.csv').is_file():
 pairs=pd.read_csv(RUN/'pares_flow2.csv',keep_default_na=False,dtype={'id':str,'pieza':str,'version':str})
 aliases=pd.read_csv(RUN/'alias_color.csv',keep_default_na=False,dtype={'id':str})
 color_table=pd.read_csv(RUN/'manifest_color.csv',keep_default_na=False,dtype={'id':str})
 RGBA=torch.load(RUN/'cache'/'rgba.pt',map_location='cpu',weights_only=False)
elif CONFIG['SMOKE']:
 pairs,aliases,RGBA=fixture_data();color_table=aliases.copy()
else:
 aliases=pd.read_csv(COLOR_DIR/'manifest_color_aliases.csv',keep_default_na=False,dtype={'id':str})
 color_table=pd.read_csv(COLOR_DIR/'manifest_color.csv',keep_default_na=False,dtype={'id':str})
 assert aliases.id.is_unique and color_table.color_i.is_unique
 pairs=pd.read_csv(F1_DIR/'pares_auditados.csv',keep_default_na=False,dtype={'id':str,'pieza':str,'version':str})
 assert {'id','pieza','version','split_flow'}<=set(pairs.columns)
 pairs['color_id']=np.where(pairs.version.eq('gi'),'int:','real:')+pairs.pieza
 mapping=aliases.set_index('id').color_i.to_dict()
 missing=set(pairs.color_id)-set(mapping)
 if missing:raise ValueError(f'Estampas emparejadas ausentes del paquete de color: {sorted(missing)[:15]}')
 pairs['color_i']=pairs.color_id.map(mapping).astype(int)
 image_index={}
 for p in INPUT_FILES:
  if p.suffix.lower() in ['.png','.jpg','.jpeg']:image_index.setdefault(p.name,[]).append(p)
 def resolve_image(raw,expected_sha=None,is_rgba=False,expected_color=None):
  raw=Path(str(raw));cand=[raw] if raw.is_file() else []
  suffix='/'.join(raw.parts[-3:])
  if not cand:cand=[p for p in image_index.get(raw.name,[]) if str(p).endswith(suffix)]
  if not cand:cand=image_index.get(raw.name,[])
  if expected_sha:cand=[p for p in cand if sha_file(p)==expected_sha]
  if is_rgba and expected_color:
   cand=[p for p in cand if hashlib.sha256(canonical_rgba(p).tobytes()).hexdigest()==expected_color]
  return choose(cand,f'Imagen ausente/ambigua: {raw}')
 paths=[];arrays=[]
 for r in tqdm(color_table.itertuples(),total=len(color_table),desc='Verificando RGB LEGO',unit='estampa'):
  p=resolve_image(r.ruta,is_rgba=True,expected_color=r.sha_color);c=canonical_rgba(p)
  assert hashlib.sha256(c.tobytes()).hexdigest()==r.sha_color
  arrays.append(c);paths.append(str(p))
 assert np.array_equal(color_table.color_i.to_numpy(),np.arange(len(color_table)))
 RGBA=torch.from_numpy(np.stack(arrays));del arrays
 color_table['ruta_actual']=paths
 resolved=[]
 for r in tqdm(pairs.to_dict('records'),desc='Verificando pares y fotos',unit='foto'):
  expected=r.get('sha_foto_efectivo') or r.get('sha256_foto') or None
  raw=r.get('archivo') or r.get('ruta')
  if not raw:raise ValueError('Fila sin ruta de foto: '+r['id'])
  p=resolve_image(raw,expected_sha=expected)
  assert p.stem==r['id'],(p,r['id'])
  resolved.append(str(p))
 pairs['archivo']=resolved

color_split=color_table.set_index('color_i').particion.to_dict()
pairs['particion_color']=pairs.color_i.map(color_split)
pairs['split_flow2']=pairs.split_flow
# No usar para train del Flow 2 objetivos que el protocolo del codec reservó.
reserved=pairs.split_flow.eq('entrenamiento') & pairs.particion_color.ne('entrenamiento')
pairs.loc[reserved,'split_flow2']='reserva_codec'
pairs['codec_color_vio_objetivo']=pairs.particion_color.eq('entrenamiento')
TRAIN=np.flatnonzero(pairs.split_flow2.eq('entrenamiento'))
VAL=np.flatnonzero(pairs.split_flow2.eq('validacion'))
TEST=np.flatnonzero(pairs.split_flow2.eq('prueba'))
PERSON=np.flatnonzero(pairs.split_flow2.eq('val_personas'))
assert min(len(TRAIN),len(VAL),len(TEST))>0,'Train/val/test vacíos; revisar manifiestos'
assert not set(pairs.iloc[TRAIN].color_i)&set(pairs.iloc[np.r_[VAL,TEST]].color_i),'Una estampa cruza train y val/test del Flow 2'
def fixed_panel(indices,n):
 rng=np.random.default_rng(CONFIG['SEMILLA']);groups={int(k):list(g.index) for k,g in pairs.iloc[indices].groupby('color_i')}
 for g in groups.values():rng.shuffle(g)
 keys=list(groups);rng.shuffle(keys);result=[]
 while len(result)<min(n,len(indices)) and any(groups.values()):
  for k in keys:
   if groups[k] and len(result)<n:result.append(groups[k].pop())
 return np.array(result,dtype=int)
VAL_FIXED=fixed_panel(VAL,CONFIG['N_VAL']);TEST_FIXED=fixed_panel(TEST,CONFIG['N_TEST'])
TRAIN_COLORS=np.array(sorted(pairs.iloc[TRAIN].color_i.unique()))
ROWS_BY_COLOR={int(k):g.index.to_numpy() for k,g in pairs.iloc[TRAIN].groupby('color_i')}
display(pairs.groupby(['version','split_flow2']).size().rename('fotos').to_frame())
print('Objetivos train:',len(TRAIN_COLORS),'| val:',len(VAL_FIXED),'| test:',len(TEST_FIXED))
pairs.to_csv(RUN/'pares_flow2.csv',index=False);aliases.to_csv(RUN/'alias_color.csv',index=False)
color_table.to_csv(RUN/'manifest_color.csv',index=False)
def rgba_batch(ci):return RGBA[torch.as_tensor(ci,dtype=torch.long)].to(DEVICE).float()/255.


## Usar caches de fotos y códigos

In [ ]:

detector=cv2.CascadeClassifier(cv2.data.haarcascades+'haarcascade_frontalface_default.xml')
assert not detector.empty()
def crop_photo(row):
 b=Path(row.archivo).read_bytes()
 if CONFIG['CHECK_SHA_FOTOS'] and isinstance(row.get('sha256_foto'),str) and row.sha256_foto:
  assert hashlib.sha256(b).hexdigest()==row.sha256_foto,('SHA foto distinto',row.archivo)
 with Image.open(io.BytesIO(b)) as im:im=im.convert('RGB');im.load();a=np.asarray(im)
 h,w=a.shape[:2];found=detector.detectMultiScale(cv2.cvtColor(a,cv2.COLOR_RGB2GRAY),1.1,5,minSize=(max(16,w//6),)*2)
 if len(found):
  x,y,cw,ch=max(found,key=lambda c:c[2]*c[3]);cx,cy,s=x+cw/2,y+ch/2,max(cw,ch)*1.8
 else:cx,cy,s=w/2,h*.45,min(w,h)*.75
 x0,y0=max(0,int(cx-s/2)),max(0,int(cy-s/2));x1,y1=min(w,int(cx+s/2)),min(h,int(cy+s/2))
 return np.asarray(Image.fromarray(a[y0:y1,x0:x1]).resize((256,256),Image.Resampling.BICUBIC)),bool(len(found)),hashlib.sha256(b).hexdigest()

PHOTO_SIGNATURE=hashlib.sha256((pairs[['id','version','color_i','archivo']].to_csv(index=False)+str(CONFIG['DINO'])).encode()).hexdigest()
cache=RUN/'cache';cache.mkdir(exist_ok=True)
if CONFIG['SMOKE']:
 class ToyDino(nn.Module):
  def __init__(self):super().__init__();self.conv=nn.Conv2d(3,32,14,14)
  def forward(self,pixel_values):
   h=self.conv(pixel_values).flatten(2).transpose(1,2)
   return type('Output',(),{'last_hidden_state':torch.cat([h.mean(1,keepdim=True),h],1)})()
 DINO=ToyDino();DINO_DIM=32
 if (RUN/'assets'/'dino'/'toy_dino.pt').is_file():DINO.load_state_dict(torch.load(RUN/'assets'/'dino'/'toy_dino.pt',map_location='cpu',weights_only=False))
 IMNET_M=torch.tensor([.485,.456,.406])[None,:,None,None];IMNET_S=torch.tensor([.229,.224,.225])[None,:,None,None]
 (RUN/'assets'/'dino').mkdir(exist_ok=True)
 atomic_save(DINO.state_dict(),RUN/'assets'/'dino'/'toy_dino.pt')
else:
 from transformers import AutoModel,AutoImageProcessor
 local=RUN/'assets'/'dino'
 dino_source=CONFIG['DINO_LOCAL'] or (str(local) if (local/'config.json').is_file() else FLOW1_CK.get('dino',CONFIG['DINO']))
 processor=AutoImageProcessor.from_pretrained(dino_source)
 DINO=AutoModel.from_pretrained(dino_source)
 DINO_DIM=DINO.config.hidden_size
 IMNET_M=torch.tensor(processor.image_mean)[None,:,None,None];IMNET_S=torch.tensor(processor.image_std)[None,:,None,None]
 DINO.save_pretrained(RUN/'assets'/'dino');processor.save_pretrained(RUN/'assets'/'dino')
DINO=DINO.to(DINO_DEVICE).eval().requires_grad_(False)
IMNET_M,IMNET_S=IMNET_M.to(DINO_DEVICE),IMNET_S.to(DINO_DEVICE)

# Restaurar Flow 1 exactamente desde su export y el checkpoint DiTo verificado.
if not CONFIG['SMOKE']:
 f1_up=F1_DIR/'upstream';f1_up.mkdir(exist_ok=True)
 for name,content in FLOW1_CK['upstream'].items():(f1_up/name).write_text(content)
 (f1_up/'upstream_registry.py').write_text('def register(name):\n    return lambda cls: cls\n')
 sys.path.insert(0,str(f1_up))
 f1cfg=FLOW1_CK['config'];assert f1cfg['DINO']==FLOW1_CK['dino']
 if str(CONFIG['DINO'])!=str(FLOW1_CK['dino']):raise ValueError('DINO distinto al del Flow 1')
 f1ns=dict(torch=torch,nn=nn,F=F,math=math,CONFIG=f1cfg,DEVICE=DEVICE,amp_ctx=amp_ctx,DIM_DINO=DINO_DIM)
 f1ns['codec']=load_module('f2_mask_codec',f1_up/'codec.py')
 f1ns['renderer']=load_module('f2_mask_renderer',f1_up/'renderer.py')
 f1ns['fm']=load_module('f2_mask_fm',f1_up/'fm.py').FM(sigma_min=1e-5,timescale=1000.)
 f1ns['sampler']=load_module('f2_mask_sampler',f1_up/'samplers.py').FMEulerSampler(f1ns['fm'])
 exec(FLOW1_MODEL_SOURCE,f1ns);exec(MASK_CODEC_SOURCE,f1ns);exec(FLOW1_GENERATE_SOURCE,f1ns)
 F1=f1ns['DiT1'](dim_dino=DINO_DIM).to(DEVICE).eval().requires_grad_(False)
 F1.load_state_dict(FLOW1_CK['ema'],strict=True)
 MASK_CODEC=f1ns['FrozenDiTo']().to(DEVICE).eval().requires_grad_(False)
 ck=torch.load(F1_DIR/'mask_dito.pt',map_location='cpu',weights_only=False)
 assert ck.get('kind')=='dito_base';MASK_CODEC.load_state_dict(ck['ema'],strict=True)
 F1_MEAN=FLOW1_CK['stats']['media'].to(DEVICE);F1_STD=FLOW1_CK['stats']['desv'].to(DEVICE)
 del ck

@torch.no_grad()
def dino_features(rgb):
 x=F.interpolate(rgb.to(DINO_DEVICE),size=(224,224),mode='bilinear',align_corners=False,antialias=True)
 with amp_ctx():h=DINO(pixel_values=(x-IMNET_M)/IMNET_S).last_hidden_state
 return h.float().to(DEVICE)
def pool_dino(h):
 b,n,d=h.shape;s=int(math.sqrt(n-1));assert s*s==n-1
 p=F.adaptive_avg_pool2d(h[:,1:].transpose(1,2).reshape(b,d,s,s),CONFIG['DINO_POOL'])
 return torch.cat([h[:,:1],p.flatten(2).transpose(1,2)],1)
@torch.no_grad()
def predict_mask(h,seeds,rows=None):
 if CONFIG['SMOKE']:
  if rows is None:raise ValueError('Inferencia libre no disponible con modelos SMOKE')
  out=RGBA[torch.as_tensor(pairs.iloc[rows].color_i.to_numpy()),3:4].float().to(DEVICE)/255
  # Cambio local controlado para ejercitar pseudoetiquetas; no es una predicción real.
  out=out.clone();out[:,:,65,44]=1
  return out
 z=f1ns['generate'](F1,h,seeds,w=float(FLOW1_CK.get('cfg_w',1.)),steps=FLOW1_CK['config']['PASOS_FLOW'])
 raw=from_tokens(z)*F1_STD+F1_MEAN
 with amp_ctx():m=MASK_CODEC.decode(raw,seeds,steps=FLOW1_CK['config'].get('PASOS_DITO_FINAL',32))
 return (m>=.5).float()

if (cache/'photos.pt').is_file():
 cp=torch.load(cache/'photos.pt',map_location='cpu',weights_only=False)
 assert cp['signature']==PHOTO_SIGNATURE,'Cambiaron los pares/fotos; usar otra VERSION'
 FOTOS=cp['photos'];TOK=cp['tokens'];TOK_M=cp['tokens_mirror'];pairs['sha_foto_verificado']=cp['hashes'];del cp
else:
 photos=[];hashes=[]
 for _,r in tqdm(pairs.iterrows(),total=len(pairs),desc='Recortando y verificando fotos',unit='foto'):
  a,_,h=crop_photo(r);photos.append(a);hashes.append(h)
 FOTOS=torch.from_numpy(np.stack(photos)).permute(0,3,1,2).contiguous();del photos
 pairs['sha_foto_verificado']=hashes
 TOK=[];TOK_M=[]
 for j in tqdm(range(0,len(pairs),CONFIG['LOTE_PREPARACION']),desc='Cache DINO y espejos',unit='lote'):
  x=FOTOS[j:j+CONFIG['LOTE_PREPARACION']].to(DEVICE).float()/255
  TOK.append(pool_dino(dino_features(x)).half().cpu());TOK_M.append(pool_dino(dino_features(x.flip(-1))).half().cpu())
 TOK=torch.cat(TOK);TOK_M=torch.cat(TOK_M)
 atomic_save(dict(signature=PHOTO_SIGNATURE,photos=FOTOS,tokens=TOK,tokens_mirror=TOK_M,hashes=hashes),cache/'photos.pt')
assert not pairs.groupby('sha_foto_verificado').split_flow2.nunique().gt(1).any(),'Foto idéntica cruza splits'

@torch.no_grad()
def encode_color_arrays(c):
 with amp_ctx():z=COLOR.encode(c[:,:3],c[:,3:4]).float()
 return standardize(z,c[:,3:4])
if (cache/'color_bank.pt').is_file():
 bank=torch.load(cache/'color_bank.pt',map_location='cpu',weights_only=False)
 assert bank['rgba_sha']==hashlib.sha256(RGBA.numpy().tobytes()).hexdigest()
 Z=bank['z'];ZM=bank['z_mirror']
else:
 zs=[];zms=[]
 for j in tqdm(range(0,len(RGBA),CONFIG['LOTE_PREPARACION']),desc='Códigos de color y espejos',unit='lote'):
  c=rgba_batch(np.arange(j,min(j+CONFIG['LOTE_PREPARACION'],len(RGBA))))
  zs.append(encode_color_arrays(c).cpu());zms.append(encode_color_arrays(c.flip(-1)).cpu())
 Z=torch.cat(zs);ZM=torch.cat(zms)
 atomic_save(dict(z=Z,z_mirror=ZM,rgba_sha=hashlib.sha256(RGBA.numpy().tobytes()).hexdigest()),cache/'color_bank.pt')
assert Z.shape[1:]==(8,16,16) and torch.isfinite(Z).all()
atomic_save(RGBA,cache/'rgba.pt');pairs.to_csv(RUN/'pares_flow2.csv',index=False)
print('Fotos:',tuple(FOTOS.shape),'| tokens:',tuple(TOK.shape),'| códigos:',tuple(Z.shape))


## Conservar predicciones de máscaras

In [ ]:

def pseudo_color(c,mhat):
 # Solo extensiones locales de color uniforme. No se afirma correspondencia semántica global.
 rgb=c[:3].transpose(1,2,0).astype(np.float32)/255;gt=c[3]>127;mh=mhat.astype(bool)
 dice=2*(gt&mh).sum()/max(1,gt.sum()+mh.sum())
 if dice<CONFIG['PRED_DICE_MIN']:return None,dict(aceptado=False,motivo='dice',dice=float(dice))
 if not .85<=mh.sum()/max(1,gt.sum())<=1.15:return None,dict(aceptado=False,motivo='ocupacion',dice=float(dice))
 ng,labels_gt=cv2.connectedComponents(gt.astype(np.uint8),connectivity=8)
 npred,_=cv2.connectedComponents(mh.astype(np.uint8),connectivity=8)
 if ng!=npred:return None,dict(aceptado=False,motivo='componentes',dice=float(dice))
 for lab in range(1,ng):
  region=labels_gt==lab
  if (region&mh).sum()<.5*region.sum():return None,dict(aceptado=False,motivo='componente_perdido',dice=float(dice))
 dist,near=cv2.distanceTransformWithLabels((~gt).astype(np.uint8),cv2.DIST_L2,5,labelType=cv2.DIST_LABEL_PIXEL)
 added=mh&~gt
 if added.any() and dist[added].max()>CONFIG['PRED_DIST_MAX']:
  return None,dict(aceptado=False,motivo='extension_lejana',dice=float(dice))
 lookup=np.zeros((int(near.max())+1,3),np.float32);lookup[near[gt]]=rgb[gt]
 out=rgb.copy()
 for y,x in np.argwhere(added):
  ys=slice(max(0,y-2),min(128,y+3));xs=slice(max(0,x-2),min(128,x+3))
  colors=rgb[ys,xs][gt[ys,xs]]
  if not len(colors) or np.max(colors.max(0)-colors.min(0))>CONFIG['PRED_COLOR_SPREAD']:
   return None,dict(aceptado=False,motivo='color_ambiguo',dice=float(dice))
  out[y,x]=lookup[near[y,x]]
 result=np.zeros((4,128,128),np.uint8)
 result[:3]=(np.clip(out,0,1)*255).round().astype(np.uint8).transpose(2,0,1)*mh
 result[3]=mh.astype(np.uint8)*255
 return result,dict(aceptado=True,motivo='pseudo_extension_local',dice=float(dice),nuevos=int(added.sum()))

train_pred_rows=fixed_panel(TRAIN,CONFIG['N_PRED_TRAIN']) if CONFIG['PRED_ENABLE'] else np.array([],dtype=int)
PRED_ROWS=np.unique(np.r_[train_pred_rows,VAL_FIXED,TEST_FIXED]).astype(int)
pred_signature=hashlib.sha256((PHOTO_SIGNATURE+'|'+str(sha_file(F1_DIR/'flow1_dito.pt') if not CONFIG['SMOKE'] else 'smoke')+
   '|'+','.join(map(str,PRED_ROWS))+'|'+json.dumps({k:v for k,v in CONFIG.items() if k.startswith('PRED_')},sort_keys=True)).encode()).hexdigest()
if (cache/'predictions.pt').is_file():
 pred_cache=torch.load(cache/'predictions.pt',map_location='cpu',weights_only=False)
 assert pred_cache['signature']==pred_signature,'Cache de predicciones incompatible; usar otra VERSION'
 PRED_MASKS=pred_cache['masks'];PSEUDO=pred_cache['pseudo'];PRED_AUDIT=pd.DataFrame(pred_cache['audit'])
else:
 PRED_MASKS={};PSEUDO={};audit=[]
 for j in tqdm(range(0,len(PRED_ROWS),CONFIG['LOTE_PREPARACION']),desc='Flow 1 congelado · cache de máscaras',unit='lote'):
  rows=PRED_ROWS[j:j+CONFIG['LOTE_PREPARACION']]
  x=FOTOS[torch.as_tensor(rows)].to(DEVICE).float()/255
  h=dino_features(x);seeds=[CONFIG['SEMILLA']+int(i) for i in rows]
  m=predict_mask(h,seeds,rows).byte().cpu().numpy()
  for k,i in enumerate(rows):
   mh=m[k,0];PRED_MASKS[int(i)]=torch.from_numpy(mh[None].copy())
   c=RGBA[int(pairs.iloc[i].color_i)].numpy();pseudo,info=pseudo_color(c,mh)
   info.update(row=int(i),color_i=int(pairs.iloc[i].color_i),split=str(pairs.iloc[i].split_flow2))
   audit.append(info)
   if pseudo is not None:PSEUDO[int(i)]=torch.from_numpy(pseudo)
 PRED_AUDIT=pd.DataFrame(audit)
 atomic_save(dict(signature=pred_signature,masks=PRED_MASKS,pseudo=PSEUDO,audit=audit),cache/'predictions.pt')
PRED_AUDIT.to_csv(RUN/'auditoria_pseudoetiquetas.csv',index=False)
PRED_TRAIN=np.array([i for i in train_pred_rows if int(i) in PSEUDO],dtype=int)
PRED_VAL=np.array([i for i in VAL_FIXED if int(i) in PSEUDO],dtype=int)
accepted_colors=pairs.iloc[PRED_TRAIN].color_i.nunique() if len(PRED_TRAIN) else 0
CAN_PRED=accepted_colors>=CONFIG['MIN_PRED_COLORES']
display(PRED_AUDIT.groupby(['split','aceptado']).size().rename('mascaras').to_frame())
print('Colores train con pseudoetiqueta aceptada:',accepted_colors,'| adaptación con predicciones:',CAN_PRED)
if not CAN_PRED:print('Se conservarán las deformaciones alineadas; no se fuerzan etiquetas para máscaras diferentes.')
PZ={};PZM={}
for j in tqdm(range(0,len(PSEUDO),CONFIG['LOTE_PREPARACION']),desc='Codificando pseudoetiquetas aceptadas',unit='lote'):
 keys=list(PSEUDO)[j:j+CONFIG['LOTE_PREPARACION']]
 c=torch.stack([PSEUDO[i] for i in keys]).to(DEVICE).float()/255
 z=encode_color_arrays(c).cpu();zm=encode_color_arrays(c.flip(-1)).cpu()
 for k,i in enumerate(keys):PZ[i]=z[k];PZM[i]=zm[k]
atomic_save(dict(z=PZ,z_mirror=PZM),cache/'pseudo_bank.pt')
# La preparación no deja modelos generativos extra en la memoria de entrenamiento.
if not CONFIG['SMOKE']:
 F1.cpu();MASK_CODEC.cpu()
DINO.cpu()
if AMP:torch.cuda.empty_cache()


## Definición del ganador

In [ ]:
def rope_2d(dim_cabeza, lado=16):
    d = dim_cabeza // 4
    frec = 1.0 / (10000 ** (torch.arange(d) / d))
    y, x = torch.meshgrid(torch.arange(lado), torch.arange(lado), indexing="ij")
    ang = torch.cat([y.flatten()[:, None] * frec, x.flatten()[:, None] * frec], 1)   # 256 × dim/2
    return torch.cos(ang), torch.sin(ang)

def aplicar_rope(x, cos, sin):                              # x: B×H×N×D
    x1, x2 = x[..., 0::2], x[..., 1::2]
    return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], -1).flatten(-2)

class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.peso, self.eps = nn.Parameter(torch.ones(d)), eps

    def forward(self, x):
        return x * torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps).to(x.dtype) * self.peso.to(x.dtype)

def modular(x, shift, scale):
    return x * (1 + scale[:, None]) + shift[:, None]

def embedding_tiempo(t, dim):
    mitad = dim // 2
    frec = torch.exp(-math.log(10000) * torch.arange(mitad, device=t.device) / mitad)
    a = t[:, None] * 1000 * frec[None]
    return torch.cat([torch.cos(a), torch.sin(a)], 1)

class Attention2(nn.Module):
 def __init__(self,dim,heads,dropout):
  super().__init__();self.heads=heads
  self.q=nn.Linear(dim,dim);self.kv=nn.Linear(dim,2*dim)
  self.nq=RMSNorm(dim//heads);self.nk=RMSNorm(dim//heads)
  self.out=nn.Sequential(nn.Linear(dim,dim),nn.Dropout(dropout))
 def forward(self,x,context=None,rope=None):
  b,n,d=x.shape;c=x if context is None else context
  q=self.q(x).reshape(b,n,self.heads,-1).transpose(1,2)
  k,v=self.kv(c).reshape(b,len(c[0]),2,self.heads,-1).permute(2,0,3,1,4)
  q,k=self.nq(q),self.nk(k)
  if rope is not None:q,k=aplicar_rope(q,*rope),aplicar_rope(k,*rope)
  y=F.scaled_dot_product_attention(q,k,v)
  return self.out(y.transpose(1,2).reshape(b,n,d))
class CrossBlock(nn.Module):
 def __init__(self,d,heads,ratio,drop):
  super().__init__();self.norms=nn.ModuleList([nn.LayerNorm(d,elementwise_affine=False) for _ in range(3)])
  self.selfatt=Attention2(d,heads,drop);self.cross=Attention2(d,heads,drop)
  self.mlp=nn.Sequential(nn.Linear(d,int(d*ratio)),nn.GELU(approximate='tanh'),nn.Dropout(drop),nn.Linear(int(d*ratio),d))
  self.ada=nn.Sequential(nn.SiLU(),nn.Linear(d,9*d));nn.init.zeros_(self.ada[-1].weight);nn.init.zeros_(self.ada[-1].bias)
 def forward(self,x,time,photo,rope):
  s1,e1,g1,s2,e2,g2,s3,e3,g3=self.ada(time).chunk(9,1)
  x=x+g1[:,None]*self.selfatt(modular(self.norms[0](x),s1,e1),rope=rope)
  x=x+g2[:,None]*self.cross(modular(self.norms[1](x),s2,e2),context=photo)
  return x+g3[:,None]*self.mlp(modular(self.norms[2](x),s3,e3))
class SingleBlock(nn.Module):
 def __init__(self,d,heads,ratio,drop):
  super().__init__();self.n1=nn.LayerNorm(d,elementwise_affine=False);self.n2=nn.LayerNorm(d,elementwise_affine=False)
  self.att=Attention2(d,heads,drop)
  self.mlp=nn.Sequential(nn.Linear(d,int(d*ratio)),nn.GELU(approximate='tanh'),nn.Dropout(drop),nn.Linear(int(d*ratio),d))
  self.ada=nn.Sequential(nn.SiLU(),nn.Linear(d,6*d));nn.init.zeros_(self.ada[-1].weight);nn.init.zeros_(self.ada[-1].bias)
 def forward(self,x,time,rope):
  s1,e1,g1,s2,e2,g2=self.ada(time).chunk(6,1)
  x=x+g1[:,None]*self.att(modular(self.n1(x),s1,e1),rope=rope)
  return x+g2[:,None]*self.mlp(modular(self.n2(x),s2,e2))
class PhotoColor(nn.Module):
 def __init__(self,grid):
  super().__init__();self.grid=grid
  self.net=nn.Sequential(nn.Conv2d(3,32,5,2,2),nn.SiLU(),nn.Conv2d(32,64,3,2,1),nn.SiLU(),nn.Conv2d(64,96,3,2,1),nn.SiLU())
 def forward(self,rgb):
  x=F.interpolate(rgb,size=(128,128),mode='bilinear',align_corners=False)
  f=F.adaptive_avg_pool2d(self.net(x),self.grid)
  mean=F.adaptive_avg_pool2d(x,self.grid)
  std=(F.adaptive_avg_pool2d(x*x,self.grid)-mean.square()).clamp_min(1e-6).sqrt()
  h=torch.cat([f,mean,std],1).flatten(2).transpose(1,2)
  return torch.cat([h.mean(1,keepdim=True),h],1)
class Flow2DiT(nn.Module):
 def __init__(self,arch,cfg,dino_dim):
  super().__init__();self.arch=arch;self.cfg=dict(cfg);d=cfg['DIM'];self.dim=d
  self.mask_encoder=nn.Sequential(nn.Conv2d(1,32,3,2,1),nn.SiLU(),nn.Conv2d(32,32,3,2,1),nn.SiLU(),nn.Conv2d(32,32,3,2,1),nn.SiLU())
  self.color=PhotoColor(cfg['DINO_POOL']);self.dino_proj=nn.Sequential(nn.LayerNorm(dino_dim),nn.Linear(dino_dim,d))
  self.rgb_proj=nn.Linear(102,d);self.null=nn.Parameter(torch.zeros(1,1,d))
  self.time=nn.Sequential(nn.Linear(d,d),nn.SiLU(),nn.Linear(d,d))
  self.pos=nn.Parameter(torch.randn(1,256,d)*.02);self.types=nn.Parameter(torch.randn(3,d)*.02)
  self.input=nn.Linear(41 if arch=='cross' else 8,d)
  self.mask_proj=nn.Linear(33,d)
  if arch=='cross':self.blocks=nn.ModuleList([CrossBlock(d,cfg['CABEZAS'],cfg['MLP'],cfg['DROPOUT']) for _ in range(cfg['PROF_CROSS'])])
  elif arch=='single':self.blocks=nn.ModuleList([SingleBlock(d,cfg['CABEZAS'],cfg['MLP'],cfg['DROPOUT']) for _ in range(cfg['PROF_SINGLE'])])
  else:raise ValueError(arch)
  self.final_norm=nn.LayerNorm(d,elementwise_affine=False);self.final_ada=nn.Sequential(nn.SiLU(),nn.Linear(d,2*d));self.out=nn.Linear(d,8)
  for layer in [self.final_ada[-1],self.out]:nn.init.zeros_(layer.weight);nn.init.zeros_(layer.bias)
  cos,sin=rope_2d(d//cfg['CABEZAS']);self.register_buffer('cos',cos,persistent=False);self.register_buffer('sin',sin,persistent=False)
 def forward(self,xt,t,dino,rgb,mask,drop=None):
  occ=(F.adaptive_max_pool2d(mask,16)>.5).float().flatten(2).transpose(1,2)
  shape=self.mask_encoder(mask).flatten(2).transpose(1,2)
  shape=torch.cat([shape,occ],-1)
  photo=self.dino_proj(dino)+self.rgb_proj(self.color(rgb))
  if drop is not None:photo=torch.where(drop[:,None,None],self.null.expand_as(photo),photo)
  time=self.time(embedding_tiempo(t,self.dim))
  if self.arch=='cross':
   x=self.input(torch.cat([xt,shape],-1))+self.pos
   for block in self.blocks:x=block(x,time,photo,(self.cos,self.sin))
  else:
   noisy=self.input(xt)+self.pos+self.types[0]
   shapes=self.mask_proj(shape)+self.pos+self.types[1]
   x=torch.cat([noisy,shapes,photo+self.types[2]],1)
   n=photo.shape[1]
   cos=torch.cat([self.cos,self.cos,torch.ones(n,self.cos.shape[1],device=xt.device)],0)
   sin=torch.cat([self.sin,self.sin,torch.zeros(n,self.sin.shape[1],device=xt.device)],0)
   for block in self.blocks:x=block(x,time,(cos,sin))
   x=x[:,:256]
  shift,scale=self.final_ada(time).chunk(2,1)
  return self.out(modular(self.final_norm(x),shift,scale))*occ


## Evaluación con las mismas semillas

In [ ]:
def masked_mean(x,m):return (x*m).sum((1,2,3))/(x.shape[1]*m.sum((1,2,3)).clamp_min(1))
def mse_mask(x,y,m):return masked_mean((x-y).square(),m)
def rgb_to_lab(rgb):
    rgb=rgb.clamp(0,1)
    lin=torch.where(rgb<=0.04045,rgb/12.92,((rgb+0.055)/1.055).pow(2.4))
    mat=rgb.new_tensor([[.4124564,.3575761,.1804375],[.2126729,.7151522,.0721750],[.0193339,.1191920,.9503041]])
    xyz=torch.einsum('ij,bjhw->bihw',mat,lin)/rgb.new_tensor([.95047,1.,1.08883])[None,:,None,None]
    delta=6/29
    f=torch.where(xyz>delta**3,xyz.clamp_min(1e-12).pow(1/3),xyz/(3*delta**2)+4/29)
    return torch.stack([116*f[:,1]-16,500*(f[:,0]-f[:,1]),200*(f[:,1]-f[:,2])],1)
def ssim_common_background(x,y,m):
    # La ventana de SSIM ve EL MISMO fondo en objetivo y reconstrucción.
    x=x*m+0.5*(1-m);y=y*m+0.5*(1-m)
    g=torch.exp(-(torch.arange(11,device=x.device,dtype=x.dtype)-5).square()/(2*1.5**2));g=g/g.sum()
    w=(g[:,None]*g[None,:]).expand(3,1,11,11).contiguous()
    conv=lambda a:F.conv2d(a,w,padding=5,groups=3)
    mx,my=conv(x),conv(y);sx=(conv(x*x)-mx*mx).clamp_min(0);sy=(conv(y*y)-my*my).clamp_min(0)
    sxy=conv(x*y)-mx*my
    ss=((2*mx*my+.01**2)*(2*sxy+.03**2))/((mx*mx+my*my+.01**2)*(sx+sy+.03**2))
    return masked_mean(ss,m)
def metricas(x,y,m):
    x=x.float();y=y.float();m=m.float();mse=mse_mask(x,y,m)
    de=(rgb_to_lab(x)-rgb_to_lab(y)).square().sum(1,keepdim=True).sqrt()
    return dict(mse=mse,mae=masked_mean((x-y).abs(),m),
                psnr=-10*torch.log10(mse.clamp_min(1e-10)),
                ssim=ssim_common_background(x,y,m),deltaE76=masked_mean(de,m))



def conditions(rows,mirror=None,photo_override=None):
 source=rows if photo_override is None else photo_override
 source=np.asarray(source,dtype=int)
 if mirror is None:mirror=torch.zeros(len(rows),dtype=torch.bool,device=DEVICE)
 a=TOK[torch.as_tensor(source)].to(DEVICE).float();b=TOK_M[torch.as_tensor(source)].to(DEVICE).float()
 h=torch.where(mirror[:,None,None],b,a)
 rgb=FOTOS[torch.as_tensor(source)].to(DEVICE).float()/255
 rgb=torch.where(mirror[:,None,None,None],rgb.flip(-1),rgb)
 return h,rgb
@torch.no_grad()
def generate2(model,h,rgb,m,seeds,w=1.,steps=None,method='euler'):
 model.eval();steps=steps or CONFIG['PASOS_FLOW'];a=to_tokens(active(m))
 x=torch.stack([torch.randn((256,8),generator=torch.Generator().manual_seed(int(s))) for s in seeds]).to(DEVICE)*a
 def vel(x,t):
  tt=torch.full((len(x),),float(t),device=DEVICE)
  with amp_ctx():
   v=model(x,tt,h,rgb,m,torch.zeros(len(x),dtype=torch.bool,device=DEVICE)).float()
   if w!=1:
    u=model(x,tt,h,rgb,m,torch.ones(len(x),dtype=torch.bool,device=DEVICE)).float();v=u+w*(v-u)
  return v
 for j in range(steps):
  v=vel(x,j/steps)
  x=x+(v+vel(x+v/steps,(j+1)/steps))/(2*steps) if method=='heun' else x+v/steps
  x=x*a
 if not torch.isfinite(x).all():raise FloatingPointError('Latente generado no finito')
 return x

def macro(df,metric='mse'):
 return float(df.groupby('color_i')[metric].mean().mean())
@torch.no_grad()
def evaluate2(model,indices,mode='real',w=1.,shuffle=False,keep=False,progress=False):
 model.eval();rows=[];images=[]
 indices=np.asarray(indices,dtype=int)
 if mode=='pseudo':indices=np.array([i for i in indices if int(i) in PSEUDO],dtype=int)
 shuffled=np.roll(indices,1) if shuffle else indices
 for j in tqdm(range(0,len(indices),CONFIG['LOTE_EVAL']),desc=f'Evaluando {mode}',unit='lote',leave=False,disable=not progress):
  ii=indices[j:j+CONFIG['LOTE_EVAL']];ci=pairs.iloc[ii].color_i.to_numpy()
  c=rgba_batch(ci)
  if mode=='pseudo':c=torch.stack([PSEUDO[int(i)] for i in ii]).to(DEVICE).float()/255
  if mode=='generada':m=torch.stack([PRED_MASKS[int(i)] for i in ii]).to(DEVICE).float()
  else:m=c[:,3:4]
  h,photo=conditions(ii,photo_override=shuffled[j:j+len(ii)])
  seeds=[CONFIG['SEMILLA']+int(i) for i in ii]
  z=generate2(model,h,photo,m,seeds,w)
  with amp_ctx():y=color_decode(z,m).float()
  if mode!='generada':scores=metricas(c[:,:3],y,m)
  else:
   # Intersección espacial: NO representa una correspondencia semántica certificada.
   intersection=m*c[:,3:4]
   proxy=mse_mask(c[:,:3],y,intersection)
   proxy=torch.where(intersection.sum((1,2,3))>0,proxy,torch.full_like(proxy,float('nan')))
   scores={'mse_interseccion_proxy':proxy,
           'pixeles_interseccion':intersection.sum((1,2,3)),
           'mascara_vacia':(m.sum((1,2,3))==0).float(),
           'dice_mascara':2*(m*c[:,3:4]).sum((1,2,3))/(m.sum((1,2,3))+c[:,3:4].sum((1,2,3))).clamp_min(1)}
  for k,i in enumerate(ii):
   rows.append(dict(row=int(i),id=pairs.iloc[i].id,color_i=int(ci[k]),version=pairs.iloc[i].version,
    codec_color_vio_objetivo=bool(pairs.iloc[i].codec_color_vio_objetivo),
    **{name:float(v[k].cpu()) for name,v in scores.items()}))
  if keep:images.append(y.cpu())
 return pd.DataFrame(rows),torch.cat(images) if images else None

@torch.no_grad()
def reference_scores(indices):
 out=[]
 for j in range(0,len(indices),CONFIG['LOTE_EVAL']):
  ii=indices[j:j+CONFIG['LOTE_EVAL']];ci=pairs.iloc[ii].color_i.to_numpy();c=rgba_batch(ci);m=c[:,3:4]
  z=to_tokens(Z[torch.as_tensor(ci)].to(DEVICE));base=torch.zeros_like(z)
  with amp_ctx():oracle=color_decode(z,m).float();mean_color=color_decode(base,m).float()
  a=mse_mask(c[:,:3],oracle,m);b=mse_mask(c[:,:3],mean_color,m)
  for k,i in enumerate(ii):out.append(dict(row=int(i),color_i=int(ci[k]),codec_mse=float(a[k].cpu()),base_mse=float(b[k].cpu())))
 return pd.DataFrame(out)
REF_VAL=reference_scores(VAL_FIXED)
CODEC_MSE=macro(REF_VAL,'codec_mse');BASE_MSE=macro(REF_VAL,'base_mse')
print('Referencias en el mismo panel · codec:',CODEC_MSE,'| código medio:',BASE_MSE)


## Funciones de entrenamiento

In [ ]:

def warp_rgba(c):
 n=len(c);ang=(torch.rand(n,device=DEVICE)*2-1)*math.radians(3)
 scale=1+(torch.rand(n,device=DEVICE)*2-1)*.03
 xy=(torch.rand(n,2,device=DEVICE)*2-1)*(2/64)
 co,si=torch.cos(ang)/scale,torch.sin(ang)/scale
 theta=torch.stack([torch.stack([co,-si,xy[:,0]],1),torch.stack([si,co,xy[:,1]],1)],1)
 q=F.grid_sample(c,F.affine_grid(theta,c.shape,align_corners=False),align_corners=False,padding_mode='zeros')
 alpha=q[:,3:4];m=(alpha>.5).float()
 return torch.cat([(q[:,:3]/alpha.clamp_min(1e-3)).clamp(0,1)*m,m],1)

class RowSampler:
 def __init__(self):self.rng=np.random.default_rng(CONFIG['SEMILLA']+123)
 def get(self,n,p_pred=0.,p_aug=0.):
  rows=[];modes=[]
  for _ in range(n):
   if CAN_PRED and self.rng.random()<p_pred:
    rows.append(int(self.rng.choice(PRED_TRAIN)));modes.append('pred')
   else:
    ci=int(self.rng.choice(TRAIN_COLORS));rows.append(int(self.rng.choice(ROWS_BY_COLOR[ci])))
    modes.append('aug' if self.rng.random()<p_aug else 'clean')
  return np.array(rows,dtype=int),np.array(modes)
 def state_dict(self):return self.rng.bit_generator.state
 def load_state_dict(self,s):self.rng.bit_generator.state=s

def training_batch(sampler,p_pred,p_aug):
 rows,modes=sampler.get(CONFIG['LOTE'],p_pred,p_aug);ci=pairs.iloc[rows].color_i.to_numpy()
 c=rgba_batch(ci);flip=(torch.rand(len(rows),device=DEVICE)<.5) if CONFIG['ESPEJO'] else torch.zeros(len(rows),dtype=torch.bool,device=DEVICE)
 z=Z[torch.as_tensor(ci)].to(DEVICE);zm=ZM[torch.as_tensor(ci)].to(DEVICE)
 for j,i in enumerate(rows):
  if modes[j]=='pred':
   c[j]=PSEUDO[int(i)].to(DEVICE).float()/255
   z[j]=PZ[int(i)].to(DEVICE);zm[j]=PZM[int(i)].to(DEVICE)
 c=torch.where(flip[:,None,None,None],c.flip(-1),c)
 target=torch.where(flip[:,None,None,None],zm,z)
 aug=np.flatnonzero(modes=='aug')
 if len(aug):
  jj=torch.as_tensor(aug,device=DEVICE);cc=warp_rgba(c[jj]);c[jj]=cc
  target[jj]=encode_color_arrays(cc)
 h,photo=conditions(rows,flip)
 return rows,modes,c,h,photo,to_tokens(target)

def capture_rng():
 return dict(py=random.getstate(),np=np.random.get_state(),torch=torch.get_rng_state(),
             cuda=torch.cuda.get_rng_state_all() if AMP else None)
def restore_rng(s):
 random.setstate(s['py']);np.random.set_state(s['np']);torch.set_rng_state(s['torch'])
 if AMP and s['cuda'] is not None:
  for i,rng in enumerate(s['cuda'][:torch.cuda.device_count()]):torch.cuda.set_rng_state(rng,i)
@torch.no_grad()
def ema_update(ema,model,step):
 decay=min(CONFIG['EMA'],(1+step)/(10+step))
 for a,b in zip(ema.parameters(),model.parameters()):a.lerp_(b.detach(),1-decay)
 for a,b in zip(ema.buffers(),model.buffers()):a.copy_(b)
def lr_factor(step,total):
 if step<CONFIG['WARMUP']:return (step+1)/max(1,CONFIG['WARMUP'])
 t=(step-CONFIG['WARMUP'])/max(1,total-CONFIG['WARMUP'])
 return CONFIG['LR_FINAL']+(1-CONFIG['LR_FINAL'])*.5*(1+math.cos(math.pi*min(1.,t)))

@torch.no_grad()
def validation(model,curriculum=False):
 clean,_=evaluate2(model,VAL_FIXED)
 shuffle,_=evaluate2(model,VAL_FIXED,shuffle=True)
 mse=macro(clean);shuf=macro(shuffle)
 info=dict(mse=mse,psnr=macro(clean,'psnr'),deltaE76=macro(clean,'deltaE76'),
           mse_foto_barajada=shuf,ratio_codec=mse/max(1e-10,CODEC_MSE))
 if curriculum and len(PRED_VAL)>=CONFIG['MIN_VAL_PSEUDO']:
  pseudo,_=evaluate2(model,PRED_VAL,mode='pseudo');info['mse_pseudo']=macro(pseudo)
  info['score']=.7*mse+.3*info['mse_pseudo']
 else:info['mse_pseudo']=None;info['score']=mse
 return info

class Curriculum:
 def __init__(self,clean_hist=None):
  self.stage='A';self.entered=0;self.history=list(clean_hist or []);self.pred_disabled=False
 def ratios(self,step):
  if self.stage=='A':return 0.,0.
  if self.stage=='B':return 0.,CONFIG['P_AUG']
  p=CONFIG['PRED_MAX']*min(1.,(step-self.entered)/max(1,CONFIG['RAMPA_PRED']))
  # Se mantiene >=70% limpio; la corrupción alineada ocupa el resto del presupuesto.
  return p,max(0.,(CONFIG['P_AUG']-p)/max(1e-6,1-p))
 def update(self,step,info):
  self.history.append(info['mse'])
  if CONFIG['SMOKE']:
   if self.stage=='A' and step>=2:self.stage='B';self.entered=step
   elif self.stage=='B' and step>=4 and CAN_PRED:self.stage='C';self.entered=step
   return
  count=CONFIG['PLATEAU_EVALS']
  plateau=len(self.history)>=count and (max(self.history[-count:])-min(self.history[-count:]))/max(1e-9,min(self.history[-count:]))<CONFIG['PLATEAU_MEJORA']
  useful=info['mse']<(1-CONFIG['MEJORA_BASE'])*BASE_MSE
  photo=info['mse']<(1-CONFIG['MEJORA_FOTO'])*info['mse_foto_barajada']
  if self.stage=='A' and CONFIG['PASOS_LIMPIOS']+step>=CONFIG['MIN_PASOS_A'] and plateau and useful and (photo or not CONFIG['REQUERIR_USO_FOTO']):
   self.stage='B';self.entered=step
  elif self.stage=='B' and step-self.entered>=CONFIG['MIN_PASOS_B'] and CAN_PRED and not self.pred_disabled:
   self.stage='C';self.entered=step
  elif self.stage=='C' and info['mse']>min(self.history)*(1+CONFIG['DEGRADACION_MAX']):
   self.stage='B';self.entered=step;self.pred_disabled=True
 def state_dict(self):return dict(stage=self.stage,entered=self.entered,history=self.history,pred_disabled=self.pred_disabled)
 def load_state_dict(self,s):self.__dict__.update(s)

DATA_SHA=hashlib.sha256((pairs[['id','version','color_i','split_flow2','sha_foto_verificado']].to_csv(index=False)+
      hashlib.sha256(RGBA.numpy().tobytes()).hexdigest()).encode()).hexdigest()
CODE_SHA=hashlib.sha256(json.dumps(dict(sources=RUNTIME_SOURCES,flow1=FLOW1_MODEL_SOURCE,mask=MASK_CODEC_SOURCE),sort_keys=True).encode()).hexdigest()
def run_signature(arch,phase):
 # Ubicaciones físicas no forman parte de la firma: el ZIP completo puede moverse.
 skip={'RAIZ','SALIDA','REANUDAR_DE','COLOR_DIR','CK_FLOW1','CK_DITO','PARES_AUDITADOS','DINO_LOCAL','REANUDAR'}
 cfg={k:v for k,v in CONFIG.items() if k not in skip}
 return hashlib.sha256(json.dumps(dict(config=cfg,arch=arch,phase=phase,data=DATA_SHA,code=CODE_SHA),sort_keys=True).encode()).hexdigest()



## Verificar compatibilidad

In [ ]:

assert DATA_SHA==ORIGIN['data_sha']==PARENT_CK['data_sha'],'Los datos no coinciden con los pesos.'
assert CODE_SHA==PARENT_CK['code_sha'],'Fuentes de modelo distintas al checkpoint original.'
assert CONFIG['DIM']==PARENT_CK['config']['DIM'] and DINO_DIM==PARENT_CK['dino_dim']
REFERENCE=Flow2DiT(WINNER,CONFIG,DINO_DIM).eval().requires_grad_(False)
REFERENCE.load_state_dict(BASELINE_EXPORT['ema'],strict=True)
# El EMA del checkpoint best y el export seleccionado deben representar los mismos pesos.
assert all(torch.equal(PARENT_CK['ema'][k],BASELINE_EXPORT['ema'][k]) for k in BASELINE_EXPORT['ema'])
print('Preflight correcto: pesos originales, código, datos y normalización compatibles.')
print('Referencia anterior · CFG:',ORIGIN['cfg_original'],'| val y test conservados:',len(VAL_FIXED),len(TEST_FIXED))


## Continuar 20 000 pasos adicionales

In [ ]:

_ext_cfg={k:v for k,v in EXT.items() if k not in ['PAQUETE_ANTERIOR','RAIZ','SALIDA','REANUDAR']}
_cfg={k:v for k,v in CONFIG.items() if k not in ['RAIZ','SALIDA','REANUDAR_DE','COLOR_DIR','CK_FLOW1','CK_DITO','PARES_AUDITADOS','DINO_LOCAL','REANUDAR']}
EXT_CODE_SHA=hashlib.sha256(json.dumps(EXT_RUNTIME_SOURCES,sort_keys=True).encode()).hexdigest()
EXT_SIGNATURE=hashlib.sha256(json.dumps(dict(parent=ORIGIN['checkpoint_sha'],config=_cfg,
 extension=_ext_cfg,data=DATA_SHA,code=CODE_SHA,extension_code=EXT_CODE_SHA),sort_keys=True).encode()).hexdigest()
START_LR=float(PARENT_CK['opt']['param_groups'][0]['lr'])
def extension_lr(step,total):
 warm=min(EXT['WARMUP_EXTRA'],max(1,total-1));peak=float(EXT['LR_MAX'])
 if step<warm:return START_LR+(peak-START_LR)*(step+1)/warm
 u=(step-warm)/max(1,total-warm)
 return peak*(EXT['LR_FINAL_FRACCION']+(1-EXT['LR_FINAL_FRACCION'])*.5*(1+math.cos(math.pi*min(1.,u))))

def train_extension(arch):
 phase='extension_20k';initial=None;clean_hist=[]
 total=EXT['PASOS_EXTRA']
 folder=RUN/'runs'/f'{arch}_{phase}';folder.mkdir(parents=True,exist_ok=True)
 signature=EXT_SIGNATURE;seed_all(CONFIG['SEMILLA'])
 model=Flow2DiT(arch,CONFIG,DINO_DIM).to(DEVICE)
 if initial is not None:model.load_state_dict(initial,strict=True)
 ema=copy.deepcopy(model).eval().requires_grad_(False)
 opt=torch.optim.AdamW(model.parameters(),lr=EXT['LR_MAX'],weight_decay=CONFIG['WD'])
 scaler=torch.amp.GradScaler('cuda',enabled=AMP);sampler=RowSampler();controller=Curriculum(clean_hist)
 step=0;best=float('inf');history=[];vals=[];previous=0.;skipped=0
 def state():
  return dict(arch=arch,phase=phase,step=step,best=best,ema=cpu_weights(ema),model=cpu_weights(model),
   opt=opt.state_dict(),scaler=scaler.state_dict(),rng=capture_rng(),sampler=sampler.state_dict(),
   controller=controller.state_dict(),history=history,validations=vals,seconds=previous+time.monotonic()-start,
   signature=signature,config=CONFIG,data_sha=DATA_SHA,code_sha=CODE_SHA,skipped=skipped,dino_dim=DINO_DIM,
   artifact_role='flow2_training',smoke=CONFIG['SMOKE'],parent_sha=ORIGIN['checkpoint_sha'],extra_steps=step,extension=EXT)
 last=folder/'last.pt'
 if CONFIG['REANUDAR'] and last.is_file():
  ck=torch.load(last,map_location='cpu',weights_only=False)
  assert ck['signature']==signature,'Código, datos o plan de entrenamiento distintos. Usar otra VERSION.'
  model.load_state_dict(ck['model']);ema.load_state_dict(ck['ema']);opt.load_state_dict(ck['opt']);scaler.load_state_dict(ck['scaler'])
  sampler.load_state_dict(ck['sampler']);controller.load_state_dict(ck['controller']);restore_rng(ck['rng'])
  step=ck['step'];best=ck['best'];history=ck['history'];vals=ck['validations'];previous=ck['seconds'];skipped=ck['skipped'];del ck
  assert (folder/'best.pt').is_file(),'Falta best asociado al last'
  print(arch,phase,': reanudando',step,'de',total)
 else:
  model.load_state_dict(PARENT_CK['model'],strict=True);ema.load_state_dict(PARENT_CK['ema'],strict=True)
  opt.load_state_dict(PARENT_CK['opt']);scaler.load_state_dict(PARENT_CK['scaler'])
  sampler.load_state_dict(PARENT_CK['sampler']);restore_rng(PARENT_CK['rng'])
  controller.load_state_dict(copy.deepcopy(PARENT_CK['controller']))
  if not EXT['CONTINUAR_CURRICULUM']:controller.stage='A'
 start=time.monotonic()
 # La fase de adaptación también compite contra el modelo de entrada sin modificar.
 if not (folder/'best.pt').is_file():
  info=validation(ema,curriculum=False);best=info['score'];vals.append(dict(step=0,stage='A',**info))
  atomic_save(state(),folder/'best.pt');atomic_save(state(),last)
 params=sum(p.numel() for p in model.parameters());print(arch,phase,':',round(params/1e6,2),'M parámetros')
 bar=tqdm(total=total,initial=step,desc=f'Flow 2 · {arch} · {phase}',unit='step',mininterval=1.)
 sums={};window=0;overflows=0
 try:
  while step<total:
   model.train();opt.zero_grad(set_to_none=True)
   lr=extension_lr(step,total)
   for g in opt.param_groups:g['lr']=lr
   pp,pa=controller.ratios(PARENT_CK['step']+step) if EXT['CONTINUAR_CURRICULUM'] else (0.,0.)
   stats={}
   for micro in range(CONFIG['ACUMULAR']):
    rows,modes,c,h,photo,target=training_batch(sampler,pp,pa);m=c[:,3:4];a=to_tokens(active(m))
    noise=torch.randn_like(target)*a;t=torch.randn(len(rows),device=DEVICE).sigmoid()
    xt=(1-t[:,None,None])*noise+t[:,None,None]*target
    drop=torch.rand(len(rows),device=DEVICE)<CONFIG['P_SIN_FOTO']
    with amp_ctx():pred=model(xt,t,h,photo,m,drop)
    diff=(pred.float()-(target-noise)).square()*a
    fm=(diff.sum((1,2))/(8*a.sum((1,2))).clamp_min(1)).mean()
    aux=torch.zeros((),device=DEVICE)
    selected=torch.nonzero((t>=.5)&~drop).flatten()[:CONFIG['AUX_MAX']]
    if CONFIG['AUX_RGB'] and step%CONFIG['AUX_CADA']==0 and len(selected):
     estimate=xt[selected]+(1-t[selected,None,None])*pred[selected].float()
     with amp_ctx():out=color_decode(estimate,m[selected]).float()
     aux=mse_mask(c[selected,:3],out,m[selected]).mean()
    loss=fm+CONFIG['AUX_RGB']*aux
    if not torch.isfinite(loss):raise FloatingPointError('Pérdida no finita; last conserva el estado anterior')
    scaler.scale(loss/CONFIG['ACUMULAR']).backward()
    for k,v in dict(loss=loss.detach(),fm=fm.detach(),rgb=aux.detach(),pred_fraction=float(np.mean(modes=='pred'))).items():
     stats[k]=stats.get(k,0)+float(v)/CONFIG['ACUMULAR']
   scaler.unscale_(opt);gnorm=torch.nn.utils.clip_grad_norm_(model.parameters(),CONFIG['CLIP'])
   if not torch.isfinite(gnorm):
    if not AMP:raise FloatingPointError('Gradiente no finito')
    scaler.step(opt);scaler.update();skipped+=1;overflows+=1
    if overflows>=50:raise FloatingPointError('50 overflows AMP consecutivos')
    continue
   overflows=0;scaler.step(opt);scaler.update();ema_update(ema,model,PARENT_CK['step']+step);step+=1;bar.update(1);window+=1
   for k,v in stats.items():sums[k]=sums.get(k,0)+v
   if step%CONFIG['LOG_CADA']==0 or step==total:
    row=dict(step=step,stage='A' if phase=='clean' else controller.stage,**{k:v/window for k,v in sums.items()},
      lr=lr,grad_norm=float(gnorm.cpu()),amp_skips=skipped)
    history.append(row);bar.set_postfix(loss=f'{row["loss"]:.4f}',etapa=row['stage'],pred=f'{row["pred_fraction"]:.0%}')
    if step%CONFIG['LOG_CADA']==0:
     tqdm.write(f'{arch} {phase} · step {step}: loss={row["loss"]:.5f} · FM={row["fm"]:.5f} · RGB={row["rgb"]:.5f} · etapa={row["stage"]} · pred={row["pred_fraction"]:.1%}')
    sums={};window=0
   if step%CONFIG['EVAL_CADA']==0 or step==total:
    info=validation(ema,curriculum=False);oldstage=controller.stage
    if EXT['CONTINUAR_CURRICULUM']:controller.update(PARENT_CK['step']+step,info)
    vals.append(dict(step=step,stage=controller.stage,**info));improved=info['score']<best
    if improved:best=info['score']
    ck=state()
    if improved:atomic_save(ck,folder/'best.pt')
    atomic_save(ck,last)
    pd.DataFrame(history).to_csv(folder/'history.csv',index=False);pd.DataFrame(vals).to_csv(folder/'validation.csv',index=False)
    tqdm.write(f'{arch} {phase} · {step}: MSE={info["mse"]:.5f} · R codec={info["ratio_codec"]:.2f} · etapa={controller.stage}')
    if oldstage!=controller.stage:tqdm.write(f'Curriculum: {oldstage} → {controller.stage}')
   elif step%CONFIG['GUARDAR_CADA']==0:atomic_save(state(),last)
 finally:bar.close()
 ck=torch.load(folder/'best.pt',map_location='cpu',weights_only=False)
 ema.load_state_dict(ck['ema']);ema.cpu();del model,opt,scaler,ck;gc.collect()
 if AMP:torch.cuda.empty_cache()
 return ema,pd.DataFrame(history),pd.DataFrame(vals),dict(arch=arch,phase=phase,best_score=best,params=params)

FINAL,HISTORY,VALS,TRAIN_INFO=train_extension(WINNER)


## Elegir pesos y CFG con validación

In [ ]:

FINAL.to(DEVICE);REFERENCE.to(DEVICE)
# La referencia permanece fija, con sus pesos y CFG originales.
BEFORE_VAL,_=evaluate2(REFERENCE,VAL_FIXED,w=ORIGIN['cfg_original'])
BEFORE_VAL.to_csv(RUN/'val_antes_20k.csv',index=False)
grid=CONFIG['CFG_GRID'] if EXT['RESELECCIONAR_CFG'] else [ORIGIN['cfg_original']]
cfg_results=[]
for w in tqdm(grid,desc='CFG · SOLO validación',unit='valor'):
 scores,_=evaluate2(FINAL,VAL_FIXED,w=w);cfg_results.append(dict(w=w,mse=macro(scores)))
pd.DataFrame(cfg_results).to_csv(RUN/'cfg_validacion_20k.csv',index=False)
CFG_BEST=float(min(cfg_results,key=lambda a:a['mse'])['w'])
AFTER_VAL,_=evaluate2(FINAL,VAL_FIXED,w=CFG_BEST)
AFTER_VAL.to_csv(RUN/'val_despues_20k.csv',index=False)
BEFORE_MSE=macro(BEFORE_VAL);AFTER_MSE=macro(AFTER_VAL)
USE_EXTENSION=AFTER_MSE<BEFORE_MSE
# Puede conservarse el anterior si ni el best extendido lo supera con su CFG elegido.
DEPLOY=FINAL if USE_EXTENSION else REFERENCE
DEPLOY_CFG=CFG_BEST if USE_EXTENSION else ORIGIN['cfg_original']
print('Validación fija: antes=',round(BEFORE_MSE,6),'| best extendido=',round(AFTER_MSE,6))
print('Export seleccionado SOLO con validación:', 'extensión' if USE_EXTENSION else 'anterior', '| CFG:',DEPLOY_CFG)
print('El test que sigue no cambia la selección.')
FINAL.cpu();REFERENCE.cpu()
fig,ax=plt.subplots(1,2,figsize=(12,4))
if len(HISTORY):ax[0].plot(HISTORY.step,HISTORY.loss,label='Loss');ax[0].plot(HISTORY.step,HISTORY.fm,label='FM')
ax[0].set_title('Entrenamiento adicional');ax[0].legend()
ax[1].plot(VALS.step,VALS.mse,label='Validación · CFG 1')
ax[1].axhline(float(VALS.iloc[0].mse),ls='--',color='gray',label='Entrada · CFG 1')
ax[1].set_title('Validación fija durante 20k');ax[1].legend()
for a in ax:a.set_xlabel('Pasos adicionales');a.grid(alpha=.2)
fig.tight_layout();fig.savefig(RUN/'curvas_20k.png',dpi=140);plt.show();plt.close(fig)


## TEST · antes y después en pantalla

In [ ]:

# Son los mismos 100 ejemplos y las mismas semillas del experimento anterior.
# Este panel ya fue inspeccionado: comparar resultados es diagnóstico; no es un nuevo test independiente.
T_SCORES={};T_IMAGES={};G_SCORES={};G_IMAGES={}
for label,model,w in [('antes',REFERENCE,ORIGIN['cfg_original']),('despues_best',FINAL,CFG_BEST)]:
 model.to(DEVICE)
 d,im=evaluate2(model,TEST_FIXED,w=w,keep=True,progress=True)
 g,gi=evaluate2(model,TEST_FIXED,mode='generada',w=w,keep=True,progress=True)
 # Diagnóstico de detalles finos: error en una banda interior de dos píxeles del contorno GT.
 # Es una región geométrica reproducible, no una anotación semántica de arrugas o lentes.
 ci=pairs.iloc[TEST_FIXED].color_i.to_numpy();gt=RGBA[torch.as_tensor(ci)].float()/255
 mm=gt[:,3:4];eroded=1-F.max_pool2d(F.pad(1-mm,(2,2,2,2),value=1),5,stride=1)
 edge=mm*(1-eroded);edge_mse=mse_mask(gt[:,:3],im,edge)
 edge_mse=torch.where(edge.sum((1,2,3))>0,edge_mse,torch.full_like(edge_mse,float('nan')))
 d['mse_borde_2px']=edge_mse.numpy()
 T_SCORES[label]=d;T_IMAGES[label]=im;G_SCORES[label]=g;G_IMAGES[label]=gi
 d.to_csv(RUN/f'test_20k_mascara_real_{label}.csv',index=False)
 g.to_csv(RUN/f'test_20k_mascara_generada_{label}_PROXY.csv',index=False)
 model.cpu()
comparison=pd.DataFrame([dict(modelo=k,mse=macro(d),mse_borde_2px=macro(d,'mse_borde_2px'),psnr=macro(d,'psnr'),deltaE76=macro(d,'deltaE76')) for k,d in T_SCORES.items()])
display(comparison);comparison.to_csv(RUN/'comparacion_test_20k.csv',index=False)
paired=T_SCORES['antes'][['row','id','color_i','mse','mse_borde_2px','deltaE76']].merge(
 T_SCORES['despues_best'][['row','mse','mse_borde_2px','deltaE76']],on='row',suffixes=('_antes','_despues'),validate='one_to_one')
paired['delta_mse']=paired.mse_despues-paired.mse_antes
paired['delta_mse_borde_2px']=paired.mse_borde_2px_despues-paired.mse_borde_2px_antes
paired['delta_deltaE76']=paired.deltaE76_despues-paired.deltaE76_antes
paired.to_csv(RUN/'test_20k_por_foto_antes_despues.csv',index=False)
for label,d in T_SCORES.items():
 d.groupby(['codec_color_vio_objetivo','color_i'])[['mse','deltaE76']].mean().groupby('codec_color_vio_objetivo').mean().to_csv(RUN/f'test_20k_codec_visto_{label}.csv')
print('Fotos con menor MSE después:',round(float((paired.delta_mse<0).mean())*100,1),'%')
print('Con máscaras generadas se reportan proxies espaciales; no GT cromático alineado.')

def composite(rgb,m):
 rgb=np.asarray(rgb);m=np.asarray(m)
 return np.clip((rgb*m+np.array([.98,.79,.16])[:,None,None]*(1-m)).transpose(1,2,0),0,1)
figdir=RUN/'test_imagenes_20k';figdir.mkdir(exist_ok=True)
for start in tqdm(range(0,len(TEST_FIXED),CONFIG['POR_PAGINA']),desc='MOSTRANDO antes/después · test fijo',unit='página'):
 end=min(len(TEST_FIXED),start+CONFIG['POR_PAGINA']);ii=TEST_FIXED[start:end];ci=pairs.iloc[ii].color_i.to_numpy()
 c=rgba_batch(ci);m=c[:,3:4]
 with torch.no_grad(),amp_ctx():oracle=color_decode(to_tokens(Z[torch.as_tensor(ci)].to(DEVICE)),m).cpu()
 c=c.cpu();fig,axes=plt.subplots(len(ii),8,figsize=(19,2.5*len(ii)),squeeze=False,facecolor='#f4f6fb')
 titles=['Foto','LEGO objetivo','VAE referencia','Antes · GT','Después best · GT','Flow 1 · máscara','Antes · generada','Después best · generada']
 for row,idx in enumerate(ii):
  j=start+row;mask=c[row,3:4].numpy();mh=PRED_MASKS[int(idx)].numpy()
  panels=[FOTOS[idx].permute(1,2,0).numpy(),composite(c[row,:3],mask),composite(oracle[row],mask),
          composite(T_IMAGES['antes'][j],mask),composite(T_IMAGES['despues_best'][j],mask),mh[0],
          composite(G_IMAGES['antes'][j],mh),composite(G_IMAGES['despues_best'][j],mh)]
  for col,panel in enumerate(panels):
   axes[row,col].imshow(panel,cmap='gray' if panel.ndim==2 else None,vmin=0,vmax=1 if panel.ndim==2 else None)
   axes[row,col].axis('off')
   if row==0:axes[row,col].set_title(titles[col],fontsize=9)
  axes[row,0].text(0,-.03,str(pairs.iloc[idx].id),transform=axes[row,0].transAxes,fontsize=7)
  for col,label in [(3,'antes'),(4,'despues_best')]:
   value=float(T_SCORES[label].iloc[j].mse)
   axes[row,col].text(.5,-.035,f'MSE {value:.4f}',ha='center',transform=axes[row,col].transAxes,fontsize=7)
 fig.suptitle(f'20k adicionales · {start+1}–{end} de {len(TEST_FIXED)} · CFG antes {ORIGIN["cfg_original"]:g} / después {CFG_BEST:g}',fontsize=14,fontweight='bold')
 fig.tight_layout(rect=[0,0,1,.97]);fig.savefig(figdir/f'test_20k_{start+1:03d}_{end:03d}.png',dpi=120)
 plt.show();plt.close(fig)


## Export independiente y UN ZIP con TODO

In [ ]:

EXPORT=RUN/'inferencia';EXPORT.mkdir(exist_ok=True)
# Original intacto en anterior/inferencia. Nuevo export: ganador entre anterior y extensión, por validación.
atomic_save(dict(ema=cpu_weights(DEPLOY),arch=WINNER,config=CONFIG,dino_dim=DINO_DIM,cfg_w=DEPLOY_CFG,
 data_sha=DATA_SHA,code_sha=CODE_SHA,artifact_role='flow2_color',smoke=CONFIG['SMOKE']),EXPORT/'flow2_color.pt')
# Guardar también el candidato extendido aunque validación prefiera el anterior.
atomic_save(dict(ema=cpu_weights(FINAL),arch=WINNER,config=CONFIG,dino_dim=DINO_DIM,cfg_w=CFG_BEST,
 data_sha=DATA_SHA,code_sha=CODE_SHA,artifact_role='flow2_color',smoke=CONFIG['SMOKE']),EXPORT/'flow2_color_candidato_20k.pt')
(EXPORT/'model.py').write_text('import math\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n'+MODEL_SOURCE)
(EXPORT/'runtime_constants.json').write_text(json.dumps(dict(flow1_model=FLOW1_MODEL_SOURCE,mask_codec=MASK_CODEC_SOURCE,
 flow1_generate=FLOW1_GENERATE_SOURCE,smoke_vae=SMOKE_VAE_SOURCE,crop=CROP_SOURCE,generate2=GENERATE2_SOURCE)))
(EXPORT/'pipeline.py').write_text(INFERENCE_SOURCE)
manifest=dict(ganador_arquitectura=WINNER,cfg_w=DEPLOY_CFG,smoke=CONFIG['SMOKE'],data_sha=DATA_SHA,
 n_train=len(TRAIN),n_val=len(VAL_FIXED),n_test=len(TEST_FIXED),pasos_extra=EXT['PASOS_EXTRA'],
 parent_checkpoint_sha=ORIGIN['checkpoint_sha'],extension_signature=EXT_SIGNATURE,
 validacion_antes=BEFORE_MSE,validacion_best_20k=AFTER_MSE,seleccion='extension' if USE_EXTENSION else 'anterior',
 pred_train_aceptadas=len(PRED_TRAIN),colores_pred_aceptados=int(accepted_colors),
 curriculum_llego_a_C=bool((VALS.stage=='C').any()),continuar_curriculum=EXT['CONTINUAR_CURRICULUM'],
 eleccion='SOLO validacion; el test ya fue inspeccionado y se usa como comparacion diagnostica')
(RUN/'resumen.json').write_text(json.dumps(manifest,indent=2));(RUN/'config.json').write_text(json.dumps(CONFIG,indent=2))
(RUN/'config_extension_20k.json').write_text(json.dumps(EXT,indent=2))
(RUN/'LEEME_20K.txt').write_text('Continuación de 20000 pasos ADICIONALES solo del ganador.\n'
 'El modelo inicial y sus resultados se conservan en anterior/. Todos los runs y assets originales permanecen.\n'
 'runs/*_extension_20k/best.pt y last.pt guardan optimizador, EMA, RNG, sampler y configuración para continuar.\n'
 'El export inferencia/flow2_color.pt se elige con validación; inferencia/flow2_color_candidato_20k.pt conserva el candidato.\n'
 'Por defecto se entrena con máscaras reales para aislar el efecto de más pasos; Flow 1 y VAE siguen congelados.\n'
 'test_imagenes_20k contiene las 100 comparaciones mostradas en pantalla.\n'
 'Para reanudar una interrupción: adjuntar ESTE ZIP completo y mantener el mismo plan de 20000 pasos.\n')

def render_literal(value):
 if isinstance(value,str):
  for q in [chr(39)*3,chr(34)*3]:
   if q not in value and not value.endswith(('\\',q[0])):return 'r'+q+value+q
  return repr(value)
 if isinstance(value,dict):return '{\n'+''.join(repr(k)+': '+render_literal(v)+',\n' for k,v in value.items())+'}'
 return repr(value)
constants=dict(FLOW1_MODEL_SOURCE=FLOW1_MODEL_SOURCE,MASK_CODEC_SOURCE=MASK_CODEC_SOURCE,
 FLOW1_GENERATE_SOURCE=FLOW1_GENERATE_SOURCE,SMOKE_VAE_SOURCE=SMOKE_VAE_SOURCE,CROP_SOURCE=CROP_SOURCE,
 MODEL_SOURCE=MODEL_SOURCE,GENERATE2_SOURCE=GENERATE2_SOURCE,INFERENCE_SOURCE=INFERENCE_SOURCE,
 RESEARCH_TEXT=RESEARCH_TEXT,RUNTIME_SOURCES=RUNTIME_SOURCES,EXT_RUNTIME_SOURCES=EXT_RUNTIME_SOURCES)
bootstrap='\n\n'.join(k+' = '+render_literal(v) for k,v in constants.items())
# El intake debe preceder al setup; el bootstrap de fuentes puede ir primero en la versión reproducible.
nb=dict(nbformat=4,nbformat_minor=5,metadata=dict(kernelspec=dict(name='python3',display_name='Python 3',language='python')),
 cells=[dict(cell_type='code',id='fuentes',metadata={'jupyter':{'source_hidden':True}},execution_count=None,outputs=[],source=bootstrap)]+
 [dict(cell_type='code',id=f'ext{i}',metadata={},execution_count=None,outputs=[],source=src) for i,src in enumerate(EXT_RUNTIME_SOURCES.values())])
(RUN/'flow2_continuacion_20k.ipynb').write_text(json.dumps(nb,ensure_ascii=False,indent=1))
(RUN/'entrenar_continuacion_20k.py').write_text(bootstrap+'\n\n'+'\n\n'.join(EXT_RUNTIME_SOURCES.values()))

# Comprobar el export fuera de los objetos del notebook.
pipeline_mod=load_module('flow2_ext_export',EXPORT/'pipeline.py')
pipeline=pipeline_mod.cargar_pipeline(RUN,device=str(DEVICE))
idx=int(TEST_FIXED[0]);mh=PRED_MASKS[idx].numpy()[0]
if not 0<float(mh.mean())<1:
 mh=RGBA[int(pairs.iloc[idx].color_i),3].numpy()/255
answer=pipeline.inferir_array(FOTOS[idx].permute(1,2,0).numpy(),mascara=mh,seed=CONFIG['SEMILLA']+idx,mostrar=False)
assert answer['rgba'].shape==(128,128,4) and np.array_equal(answer['rgba'][:,:,3],mh.astype(np.uint8)*255)
assert np.isfinite(answer['rgb']).all()
del pipeline
print('Export independiente comprobado.')

# UN ZIP contiene TODO: original, continuación, modelos, optimizador, caches, resultados e imágenes.
full=RUN.parent/(RUN.name+'_TODO.zip');files=sorted(p for p in RUN.rglob('*') if p.is_file())
with zipfile.ZipFile(full,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=4,allowZip64=True) as z:
 for p in tqdm(files,desc='UN ZIP · TODO original y 20k',unit='archivo'):z.write(p,Path(RUN.name)/p.relative_to(RUN))
with zipfile.ZipFile(full) as z:
 for member in tqdm(z.infolist(),desc='Verificando CRC · ZIP completo',unit='archivo'):
  if member.is_dir():continue
  with z.open(member) as stream:
   for _ in iter(lambda:stream.read(2**20),b''):pass
print('Descargar:',full,'| MiB:',round(full.stat().st_size/2**20,1));display(FileLink(str(full)))
